# 🚀 FINE-TUNING TỐI ƯU HÓA TOÀN DIỆN QWEN2.5-VL-3B
- **System Prompt Optimization:** Ép câu trả lời súc tích tuyệt đối.
- **Data Scale:** 2,400 mẫu cân bằng trên 15 template.
- **PEFT LoRA:** r=16, alpha=32, Gradient Checkpointing (VRAM < 6GB).

In [ ]:
# 1. Cài đặt môi trường chuẩn xác
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers>=4.49.0" "peft>=0.13.2" "accelerate>=0.34.2" pillow torchvision

import sys
for mod in list(sys.modules.keys()):
    if any(mod.startswith(k) for k in ["transformers", "peft", "accelerate", "torchao", "qwen_vl_utils"]):
        del sys.modules[mod]

import time
import json
import re
import zipfile
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType
from qwen_vl_utils import process_vision_info

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
print(f"🧠 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Giải nén và lập chỉ mục ảnh
extract_dir = "/kaggle/working/extracted_images"
os.makedirs(extract_dir, exist_ok=True)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "images.zip":
            print(f"📦 Đang giải nén {f}...")
            with zipfile.ZipFile(os.path.join(root, f), 'r') as zf:
                zf.extractall(extract_dir)

image_map = {}
for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            image_map[f] = os.path.join(root, f)

print(f"📸 Tổng số ảnh đã lập chỉ mục trên Kaggle: {len(image_map)}")


In [ ]:
# 3. Khởi tạo Qwen2.5-VL-3B với Gradient Checkpointing
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"⏳ Đang nạp Base Model: {model_name} (Native FP16)... ")
processor = AutoProcessor.from_pretrained(model_name, min_pixels=256*28*28, max_pixels=512*28*28)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 4. Dataset với Strict System Prompt & Target-Only Masking
raw_train_data = [
  {
    "image_name": "einvoice_viettel_train_068.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Bảo trì Hệ thống mạng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,500,000"
  },
  {
    "image_name": "supermarket_winmart_train_221.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_train_171.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_189.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-ELEVEN BÙI VIỆN\", \"timestamp\": \"18/06/2026 17:18\", \"total_cost\": \"184,800\", \"items\": [{\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"80,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"30,000\"}, {\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"20,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"38,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_123.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 356 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"05/06/2026 15:55\", \"total_cost\": \"13,338,000đ\", \"items\": [{\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"2\", \"amount\": \"2,500,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"4\", \"amount\": \"6,000,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_215.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "722,700"
  },
  {
    "image_name": "convenience_gs25_train_214.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"GS25 NGUYỄN HUỆ\", \"address\": \"Số 239 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"23/05/2026 19:59\", \"total_cost\": \"69,120đ\", \"items\": [{\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"3\", \"amount\": \"36,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"2\", \"amount\": \"28,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_val_065.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "image_name": "minimart_anan_train_143.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_142.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH NGUYỄN TRÃI"
  },
  {
    "image_name": "convenience_circlek_train_061.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 444 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "einvoice_viettel_val_011.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "restaurant_jollibee_val_046.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Khoai Môn, Khoai Tây Lắc Phô Mai, Mì Ý Sốt Bò Bằm, Nước ngọt 7Up L, Gà Giòn Vui Vẻ 1 miếng"
  },
  {
    "image_name": "convenience_circlek_train_170.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Hai Bà Trưng\", \"address\": \"Số 9 Lý Thường Kiệt, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"27/05/2026 07:20\", \"total_cost\": \"46,200đ\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24,000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"2\", \"amount\": \"18,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_213.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"HIGHLANDS COFFEE LANDMARK\", \"box\": [97, 16, 282, 37]}"
  },
  {
    "image_name": "cafe_highlands_train_094.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "restaurant_kfc_train_115.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 19:43"
  },
  {
    "image_name": "einvoice_viettel_train_235.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_circlek_train_025.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"Circle K Lê Lợi\", \"box\": [154, 21, 250, 44]}"
  },
  {
    "image_name": "receipt_c45_bb_train_065.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "einvoice_viettel_train_038.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Giấy Double A A4 70gsm, Bút bi Thiên Long FO-03, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "restaurant_kfc_val_066.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Chiên L trên hóa đơn là bao nhiêu?",
    "ground_truth": "28,000"
  },
  {
    "image_name": "einvoice_vnpt_train_239.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 6 Lê Lợi, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_202.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Saigon Trade Center\", \"box\": [92, 20, 285, 42]}"
  },
  {
    "image_name": "convenience_7eleven_train_262.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000"
  },
  {
    "image_name": "supermarket_winmart_train_101.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "image_name": "einvoice_vnpt_val_034.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"address\": \"Số 258 Lê Lợi, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"03/06/2026 20:47\", \"total_cost\": \"7,257,600đ\", \"items\": [{\"name\": \"Chuột không dây Logitech\", \"qty\": \"3\", \"amount\": \"870,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"1\", \"amount\": \"850,000\"}, {\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"2\", \"amount\": \"5,000,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_218.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chuột không dây Logitech trên hóa đơn là bao nhiêu?",
    "ground_truth": "290,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_176.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_082.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bột giặt Ariel 3.2kg trên hóa đơn là bao nhiêu?",
    "ground_truth": "175.000"
  },
  {
    "image_name": "restaurant_jollibee_train_240.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "70,000"
  },
  {
    "image_name": "einvoice_viettel_val_035.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_highlands_train_057.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "163,296"
  },
  {
    "image_name": "convenience_gs25_train_169.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cơm nắm cá hồi sốt Mayo, Trà sữa Kirin Latte 345ml, Sữa chua uống Proby"
  },
  {
    "image_name": "supermarket_winmart_val_020.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "76.000"
  },
  {
    "image_name": "supermarket_lotte_train_065.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hộp dâu tây Đà Lạt 250g trên hóa đơn là bao nhiêu?",
    "ground_truth": "65,000.00"
  },
  {
    "image_name": "convenience_circlek_train_218.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Lê Lợi\", \"address\": \"Số 277 Bùi Thị Xuân, Quận 1, Đồng Nai\", \"timestamp\": \"29/05/2026 17:47\", \"total_cost\": \"192,500đ\", \"items\": [{\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24,000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"2\", \"amount\": \"32,000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"4\", \"amount\": \"88,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"2\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_039.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "convenience_gs25_train_056.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Sen Vàng Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "minimart_anan_val_041.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 09:22"
  },
  {
    "image_name": "receipt_c45_bb_train_155.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1.815.000 VND"
  },
  {
    "image_name": "cafe_phuclong_train_235.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà sữa Phúc Long L, Bánh Croissant Bơ Pháp, Trà Đào Phúc Long L, Hồng Trà Sữa M, Cà phê Latte"
  },
  {
    "image_name": "restaurant_jollibee_train_223.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "10/06/2026 08:36"
  },
  {
    "image_name": "restaurant_jollibee_train_231.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_vnpt_train_180.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"address\": \"Số 225 Hai Bà Trưng, Quận 1, Hải Phòng\", \"timestamp\": \"18/06/2026 21:48\", \"total_cost\": \"16,577,000đ\", \"items\": [{\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"3\", \"amount\": \"7,500,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"3\", \"amount\": \"2,550,000\"}, {\"name\": \"Chuột không dây Logitech\", \"qty\": \"3\", \"amount\": \"870,000\"}, {\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"2\", \"amount\": \"3,700,000\"}, {\"name\": \"Bàn phím cơ Dareu EK87\", \"qty\": \"1\", \"amount\": \"450,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_val_040.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Caramel Macchiato L, Caffe Americano L, Cold Brew Coffee M"
  },
  {
    "image_name": "supermarket_winmart_train_007.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "VM+QNH CẨM PHẢ"
  },
  {
    "image_name": "receipt_c45_bb_train_190.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 303 Hai Bà Trưng, Quận 1, Cần Thơ\", \"timestamp\": \"Ngày 31 tháng 05 năm 2026\", \"total_cost\": \"1.815.000 VND\"}"
  },
  {
    "image_name": "cafe_starbucks_train_112.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"561,000\", \"box\": [307, 456, 359, 477]}"
  },
  {
    "image_name": "restaurant_kfc_train_140.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bắp Cải Trộn L trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "convenience_circlek_train_080.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "einvoice_vnpt_train_083.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "17/06/2026 19:15"
  },
  {
    "image_name": "cafe_phuclong_train_060.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LANDMARK\", \"address\": \"Số 246 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"07/06/2026 10:26\", \"total_cost\": \"594,000\", \"items\": [{\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"1\", \"amount\": \"165,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"2\", \"amount\": \"45,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"110,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"150,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_val_005.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"box\": [79, 16, 300, 40]}"
  },
  {
    "image_name": "convenience_7eleven_train_075.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước khoáng Dasani 500ml là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "supermarket_winmart_val_015.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Custas Orion 6P trên hóa đơn là bao nhiêu?",
    "ground_truth": "34.000"
  },
  {
    "image_name": "restaurant_jollibee_train_175.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "35,000"
  },
  {
    "image_name": "einvoice_viettel_train_153.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mực in Canon Cartridge trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,250,000"
  },
  {
    "image_name": "supermarket_winmart_val_011.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "484.920"
  },
  {
    "image_name": "convenience_circlek_train_212.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Hai Bà Trưng\", \"address\": \"Số 203 Nguyễn Trãi, Quận 1, Cần Thơ\", \"timestamp\": \"24/05/2026 13:04\", \"total_cost\": \"59,400đ\", \"items\": [{\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_157.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "einvoice_vnpt_train_111.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "01/06/2026 14:27"
  },
  {
    "image_name": "restaurant_kfc_train_115.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "11/06/2026 19:43"
  },
  {
    "image_name": "cafe_highlands_train_194.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 08:22"
  },
  {
    "image_name": "supermarket_winmart_train_127.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước rửa chén Sunlight 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "87.000"
  },
  {
    "image_name": "convenience_circlek_train_058.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_160.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_162.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước uống Aquafina 500ml, Bánh bao trứng muối"
  },
  {
    "image_name": "cafe_phuclong_train_215.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Đào Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "55,000"
  },
  {
    "image_name": "convenience_gs25_train_199.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "63,800đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_242.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH NGUYỄN TRÃI"
  },
  {
    "image_name": "einvoice_vnpt_train_044.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bàn phím cơ Dareu EK87, Bộ phát Wifi TP-Link Archer, Cáp mạng Cat6 UTP 305m, Dịch vụ Lắp đặt camera giám sát, Chuột không dây Logitech"
  },
  {
    "image_name": "restaurant_kfc_train_072.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Nguyễn Thái Học"
  },
  {
    "image_name": "supermarket_winmart_train_249.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn giấy Pulppy 100 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "18.000"
  },
  {
    "image_name": "supermarket_lotte_train_133.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"363,960.00\", \"box\": [318, 484, 388, 506]}"
  },
  {
    "image_name": "minimart_anan_train_205.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"106.920\", \"box\": [340, 381, 391, 403]}"
  },
  {
    "image_name": "einvoice_viettel_train_046.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "3,850,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_264.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 450 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "einvoice_viettel_train_069.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_7eleven_train_118.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-Eleven Saigon Trade Center\", \"timestamp\": \"28/05/2026 09:37\", \"total_cost\": \"105,840\", \"items\": [{\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"38,000\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_134.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 443 Nguyễn Huệ, Quận 1, Cần Thơ"
  },
  {
    "image_name": "cafe_highlands_train_234.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "117,000"
  },
  {
    "image_name": "cafe_highlands_train_264.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE"
  },
  {
    "image_name": "einvoice_viettel_train_033.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "27/05/2026 20:07"
  },
  {
    "image_name": "cafe_phuclong_train_094.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_jollibee_train_212.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "103,400đ"
  },
  {
    "image_name": "convenience_gs25_train_236.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "29/05/2026 10:46"
  },
  {
    "image_name": "restaurant_jollibee_train_085.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "JOLLIBEE PASTEUR"
  },
  {
    "image_name": "convenience_gs25_train_161.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh giò thịt băm trứng cút là bao nhiêu?",
    "ground_truth": "42,000"
  },
  {
    "image_name": "einvoice_viettel_val_021.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 69 Nguyễn Huệ, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_087.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Nguyễn Du\", \"box\": [122, 22, 247, 49]}"
  },
  {
    "image_name": "minimart_anan_train_261.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "image_name": "restaurant_jollibee_train_140.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_val_040.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC VIỆT NAM\", \"address\": \"Số 312 Bùi Thị Xuân, Quận Hai Bà Trưng, Hà Nội\", \"timestamp\": \"29/05/2026 10:24\", \"total_cost\": \"375,100\", \"items\": [{\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"4\", \"amount\": \"72,000\"}, {\"name\": \"Nước ngọt Pepsi L\", \"qty\": \"3\", \"amount\": \"57,000\"}, {\"name\": \"Bắp Cải Trộn L\", \"qty\": \"1\", \"amount\": \"22,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"2\", \"amount\": \"78,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"4\", \"amount\": \"112,000\"}]}"
  },
  {
    "image_name": "minimart_anan_val_048.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "receipt_c45_bb_train_037.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 154 Lý Thường Kiệt, Quận 1, Cần Thơ"
  },
  {
    "image_name": "cafe_starbucks_train_028.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_vnpt_train_138.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 83 Nguyễn Huệ, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "convenience_gs25_train_070.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 LANDMARK 81\", \"address\": \"Số 118 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"28/05/2026 10:55\", \"total_cost\": \"146,880đ\", \"items\": [{\"name\": \"Cafe sữa đá GS25\", \"qty\": \"4\", \"amount\": \"88,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"4\", \"amount\": \"48,000\"}]}"
  },
  {
    "image_name": "minimart_anan_val_036.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"MINIMART ANAN\", \"box\": [156, 23, 251, 45]}"
  },
  {
    "image_name": "einvoice_viettel_train_026.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"20,788,900đ\", \"box\": [659, 839, 740, 861]}"
  },
  {
    "image_name": "cafe_phuclong_train_064.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LÊ LAI"
  },
  {
    "image_name": "minimart_anan_train_141.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "10.000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_212.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hành lá 100g là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_train_194.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 221 Hai Bà Trưng, Quận 1, Đồng Nai"
  },
  {
    "image_name": "supermarket_lotte_train_203.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bia Heineken Lon 330ml là bao nhiêu?",
    "ground_truth": "19,500.00"
  },
  {
    "image_name": "einvoice_vnpt_train_085.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "13/06/2026 19:05"
  },
  {
    "image_name": "convenience_gs25_train_152.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"77,000đ\", \"box\": [337, 419, 390, 438]}"
  },
  {
    "image_name": "supermarket_lotte_train_056.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"box\": [99, 17, 306, 40]}"
  },
  {
    "image_name": "receipt_c45_bb_train_143.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2.052.000 VND"
  },
  {
    "image_name": "supermarket_winmart_train_261.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH CẨM PHẢ\", \"address\": \"Số 395 Lý Thường Kiệt, Quận 1, Đồng Nai\", \"timestamp\": \"29/05/2026 07:29\", \"total_cost\": \"386.650\", \"items\": [{\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"1\", \"amount\": \"10.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"4\", \"amount\": \"152.000\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"1\", \"amount\": \"18.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"1\", \"amount\": \"3.500\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"3\", \"amount\": \"168.000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 187 Nguyễn Trãi, Quận Tân Bình, TP. Hồ Chí Minh\", \"timestamp\": \"30/05/2026 14:20\", \"total_cost\": \"210,100\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"1\", \"amount\": \"156,000\"}, {\"name\": \"Bánh Tiramisu\", \"qty\": \"1\", \"amount\": \"35,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_075.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "31/05/2026 20:51"
  },
  {
    "image_name": "cafe_highlands_train_032.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "cafe_starbucks_train_226.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 183 Hai Bà Trưng, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"25/05/2026\", \"total_cost\": \"311,850\", \"items\": [{\"name\": \"Chocolate Muffin\", \"qty\": \"2\", \"amount\": \"90,000\"}, {\"name\": \"Green Tea Latte M\", \"qty\": \"3\", \"amount\": \"225,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_155.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LÊ LAI\", \"address\": \"Số 227 Lý Thường Kiệt, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"22/05/2026 13:30\", \"total_cost\": \"449,350\", \"items\": [{\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"2\", \"amount\": \"50,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"1\", \"amount\": \"90,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"55,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"4\", \"amount\": \"55,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"1\", \"amount\": \"180,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_247.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"8,756,000đ\", \"box\": [675, 713, 747, 734]}"
  },
  {
    "image_name": "convenience_circlek_train_223.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 13:45"
  },
  {
    "image_name": "convenience_7eleven_train_057.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Saigon Trade Center\", \"box\": [96, 15, 288, 36]}"
  },
  {
    "image_name": "einvoice_viettel_train_155.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "supermarket_lotte_train_086.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "convenience_gs25_train_031.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cafe sữa đá GS25, Sữa chua uống Proby, Bánh giò thịt băm trứng cút, Trà sữa Kirin Latte 345ml"
  },
  {
    "image_name": "convenience_gs25_train_159.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích Đức ăn liền trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "restaurant_jollibee_train_145.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 09:04"
  },
  {
    "image_name": "convenience_circlek_train_054.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CIRCLE K"
  },
  {
    "image_name": "cafe_starbucks_val_062.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_kfc_train_076.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 114 Bùi Thị Xuân, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "convenience_gs25_train_094.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"56,160đ\", \"box\": [339, 360, 391, 382]}"
  },
  {
    "image_name": "cafe_phuclong_val_051.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 321 Cách Mạng Tháng Tám, Quận 10, TP. Hồ Chí Minh"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_020.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Đường cát trắng Biên Hòa 1kg, Rau muống nước 500g"
  },
  {
    "image_name": "cafe_starbucks_train_256.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_jollibee_val_060.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "257,040đ"
  },
  {
    "image_name": "supermarket_winmart_train_199.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"419.650\", \"box\": [338, 530, 388, 552]}"
  },
  {
    "image_name": "convenience_circlek_train_105.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kẹo cao su Cool Air hũ, Bánh bao trứng muối"
  },
  {
    "image_name": "restaurant_jollibee_train_082.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "377,300đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_011.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "restaurant_kfc_train_013.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước ngọt Pepsi L, Gà Giòn Cay 1 miếng, Burger Tôm, Khoai Tây Chiên L, Bắp Cải Trộn L, Bánh Trứng Egg Tart"
  },
  {
    "image_name": "cafe_phuclong_train_034.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_127.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"address\": \"Số 256 Lê Lợi, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"05/06/2026 22:03:00\", \"total_cost\": \"542,300.00\", \"items\": [{\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"3\", \"amount\": \"81,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"4\", \"amount\": \"88,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"2\", \"amount\": \"64,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"4\", \"amount\": \"260,000.00\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_108.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "image_name": "supermarket_lotte_train_235.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"SAIGON CO.OP\", \"address\": \"Số 448 Trần Hưng Đạo, Quận 1, Cần Thơ\", \"timestamp\": \"24/05/2026 11:49:00\", \"total_cost\": \"1,057,320.00\", \"items\": [{\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"2\", \"amount\": \"56,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"2\", \"amount\": \"130,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"3\", \"amount\": \"66,000.00\"}, {\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"1\", \"amount\": \"27,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"4\", \"amount\": \"700,000.00\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Tiramisu trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "restaurant_kfc_train_204.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "cafe_starbucks_train_266.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Cold Brew Coffee M, Chocolate Muffin, Caramel Macchiato L"
  },
  {
    "image_name": "supermarket_lotte_train_253.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 152 Lý Thường Kiệt, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "supermarket_lotte_val_039.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "27/05/2026 16:17:00"
  },
  {
    "image_name": "supermarket_winmart_train_036.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "28/05/2026 18:17"
  },
  {
    "image_name": "restaurant_jollibee_train_082.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Mì Ý Sốt Bò Bằm, Nước ngọt 7Up L, Gà Giòn Vui Vẻ 1 miếng, Khoai Tây Lắc Phô Mai, Bánh Khoai Môn"
  },
  {
    "image_name": "minimart_anan_train_262.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "178.200"
  },
  {
    "image_name": "restaurant_jollibee_train_051.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Khoai Môn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "cafe_phuclong_train_090.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"588,060\", \"box\": [313, 398, 365, 419]}"
  },
  {
    "image_name": "minimart_anan_train_106.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_kfc_train_222.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "526,900"
  },
  {
    "image_name": "convenience_circlek_train_040.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "image_name": "einvoice_viettel_val_014.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 70 Nguyễn Huệ, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_train_166.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 413 Hai Bà Trưng, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_lotte_val_052.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "213,300.00"
  },
  {
    "image_name": "cafe_starbucks_train_238.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_198.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC VIỆT NAM"
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "39,000"
  },
  {
    "image_name": "restaurant_jollibee_val_037.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "35,000"
  },
  {
    "image_name": "supermarket_winmart_train_140.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì tôm Hảo Hảo chua cay trên hóa đơn là bao nhiêu?",
    "ground_truth": "14.000"
  },
  {
    "image_name": "convenience_gs25_val_003.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "cafe_highlands_val_044.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "302,500"
  },
  {
    "image_name": "convenience_7eleven_train_089.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Saigon Trade Center\", \"box\": [92, 15, 288, 36]}"
  },
  {
    "image_name": "restaurant_jollibee_val_011.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Khoai Môn, Gà Giòn Vui Vẻ 1 miếng, Khoai Tây Lắc Phô Mai, Nước ngọt 7Up L, Mì Ý Sốt Bò Bằm"
  },
  {
    "image_name": "restaurant_jollibee_train_150.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 337 Lê Lợi, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "supermarket_winmart_train_257.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_val_055.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "140,000"
  },
  {
    "image_name": "cafe_highlands_val_010.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "convenience_7eleven_train_131.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "24/05/2026 16:32"
  },
  {
    "image_name": "restaurant_kfc_train_077.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_jollibee_train_144.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_040.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 16 Nguyễn Trãi, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"28/05/2026 11:59\", \"total_cost\": \"10,047,240đ\", \"items\": [{\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"2\", \"amount\": \"8,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"3\", \"amount\": \"3,750,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"1\", \"amount\": \"1,500,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_095.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "253.800"
  },
  {
    "image_name": "convenience_gs25_train_211.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "cafe_highlands_val_009.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE LÊ LỢI"
  },
  {
    "image_name": "supermarket_lotte_train_167.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"SAIGON CO.OP\", \"address\": \"Số 43 Lý Thường Kiệt, Quận 1, Bình Dương\", \"timestamp\": \"20/06/2026 14:09:00\", \"total_cost\": \"470,340.00\", \"items\": [{\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"3\", \"amount\": \"58,500.00\"}, {\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"2\", \"amount\": \"54,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"3\", \"amount\": \"84,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"3\", \"amount\": \"195,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"2\", \"amount\": \"44,000.00\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_061.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_val_066.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "14/06/2026"
  },
  {
    "image_name": "einvoice_viettel_val_031.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "convenience_7eleven_train_131.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Sandwich xá xíu phô mai, Snack khoai tây Lays 95g, Mì xào tương đen"
  },
  {
    "image_name": "convenience_circlek_train_129.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "supermarket_winmart_train_188.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 203 Lê Lợi, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "cafe_highlands_train_013.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Thạch Đào Size L, Phin Sữa Đá Size L, Trà Sen Vàng Size M, Freeze Trà Xanh Size M, Cà Phê Đen Đá Size M, Bánh Tiramisu"
  },
  {
    "image_name": "restaurant_jollibee_train_019.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "178,200đ"
  },
  {
    "image_name": "restaurant_kfc_val_061.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_vnpt_train_167.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_033.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Bà Hom\", \"box\": [163, 15, 248, 36]}"
  },
  {
    "image_name": "restaurant_kfc_train_093.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 145 Cách Mạng Tháng Tám, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_063.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "image_name": "convenience_gs25_train_256.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 50 Hai Bà Trưng, Quận 1, Đồng Nai"
  },
  {
    "image_name": "convenience_7eleven_train_166.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000"
  },
  {
    "image_name": "convenience_circlek_val_030.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"Circle K Lê Lợi\", \"box\": [157, 26, 251, 47]}"
  },
  {
    "image_name": "convenience_circlek_train_181.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "64,000"
  },
  {
    "image_name": "minimart_anan_train_036.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì Ly ăn liền Modern, Xúc xích tiệt trùng Ponnie"
  },
  {
    "image_name": "receipt_c45_bb_train_199.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2.090.000 VND"
  },
  {
    "image_name": "cafe_starbucks_train_139.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "160,000"
  },
  {
    "image_name": "convenience_7eleven_val_052.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sandwich xá xíu phô mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "image_name": "supermarket_lotte_train_230.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "866,250.00"
  },
  {
    "image_name": "restaurant_jollibee_val_043.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "17,000"
  },
  {
    "image_name": "einvoice_viettel_train_247.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "20/06/2026 12:51"
  },
  {
    "image_name": "restaurant_jollibee_train_045.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt 7Up L trên hóa đơn là bao nhiêu?",
    "ground_truth": "17,000"
  },
  {
    "image_name": "restaurant_kfc_val_011.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "cafe_starbucks_val_065.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "einvoice_vnpt_train_047.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH\", \"box\": [47, 86, 372, 115]}"
  },
  {
    "image_name": "cafe_highlands_train_186.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE LÊ LỢI"
  },
  {
    "image_name": "convenience_7eleven_train_157.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "168,480"
  },
  {
    "image_name": "convenience_gs25_train_206.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "63,800đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_063.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Thịt ba rọi heo 500g, Trứng gà tươi hộp 10 quả, Hành lá 100g, Rau muống nước 500g, Nước mắm Nam Ngư 750ml, Đường cát trắng Biên Hòa 1kg"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_196.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "424,600"
  },
  {
    "image_name": "restaurant_kfc_val_026.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Khoai Tây Chiên L, Bánh Trứng Egg Tart, Bắp Cải Trộn L, Burger Tôm, Nước ngọt Pepsi L"
  },
  {
    "image_name": "restaurant_kfc_train_106.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "supermarket_winmart_val_006.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "354.200"
  },
  {
    "image_name": "supermarket_winmart_train_212.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 07:47"
  },
  {
    "image_name": "convenience_gs25_train_158.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "einvoice_viettel_train_259.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "65,000"
  },
  {
    "image_name": "receipt_c45_bb_train_097.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "270.000 VND"
  },
  {
    "image_name": "einvoice_viettel_val_047.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"box\": [49, 68, 350, 94]}"
  },
  {
    "image_name": "convenience_7eleven_train_175.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì xào tương đen trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "cafe_starbucks_train_092.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "130,000"
  },
  {
    "image_name": "supermarket_lotte_val_027.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hộp dâu tây Đà Lạt 250g trên hóa đơn là bao nhiêu?",
    "ground_truth": "260,000.00"
  },
  {
    "image_name": "minimart_anan_val_048.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ly ăn liền Modern trên hóa đơn là bao nhiêu?",
    "ground_truth": "27.000"
  },
  {
    "image_name": "minimart_anan_train_007.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "8.000"
  },
  {
    "image_name": "convenience_circlek_train_178.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "28/05/2026 07:35"
  },
  {
    "image_name": "receipt_c45_bb_train_082.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 07 tháng 06 năm 2026"
  },
  {
    "image_name": "cafe_starbucks_train_051.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "869,940"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_218.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 32 Trần Hưng Đạo, Quận 1, Đồng Nai"
  },
  {
    "image_name": "supermarket_lotte_train_070.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "1,003,320.00"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_162.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "339,900"
  },
  {
    "image_name": "restaurant_jollibee_val_015.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "91,800đ"
  },
  {
    "image_name": "convenience_circlek_train_105.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Kẹo cao su Cool Air hũ, Bánh bao trứng muối"
  },
  {
    "image_name": "convenience_circlek_train_193.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CIRCLE K"
  },
  {
    "image_name": "supermarket_lotte_train_081.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bột giặt Ariel 3.2kg trên hóa đơn là bao nhiêu?",
    "ground_truth": "175,000.00"
  },
  {
    "image_name": "restaurant_jollibee_train_202.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "convenience_gs25_train_085.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 NGUYỄN HUỆ\", \"address\": \"Số 11 Lê Lợi, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"03/06/2026 15:05\", \"total_cost\": \"118,800đ\", \"items\": [{\"name\": \"Cafe sữa đá GS25\", \"qty\": \"1\", \"amount\": \"22,000\"}, {\"name\": \"Sữa chua uống Proby\", \"qty\": \"2\", \"amount\": \"16,000\"}, {\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"3\", \"amount\": \"48,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"1\", \"amount\": \"12,000\"}, {\"name\": \"Xúc xích Đức ăn liền\", \"qty\": \"1\", \"amount\": \"10,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_val_021.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "70,000"
  },
  {
    "image_name": "supermarket_lotte_val_051.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "15/06/2026 17:55:00"
  },
  {
    "image_name": "convenience_circlek_train_065.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "29/05/2026 13:46"
  },
  {
    "image_name": "convenience_gs25_train_006.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "GS25 THỦ ĐỨC"
  },
  {
    "image_name": "convenience_7eleven_train_006.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"93,500\", \"box\": [320, 176, 365, 200]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_023.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Rau muống nước 500g, Thịt đùi heo 500g, Thịt ba rọi heo 500g, Đường cát trắng Biên Hòa 1kg, Trứng gà tươi hộp 10 quả"
  },
  {
    "image_name": "restaurant_kfc_val_025.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_lotte_train_243.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "718,300.00"
  },
  {
    "image_name": "cafe_highlands_train_148.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 335 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_phuclong_train_113.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Đào Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "einvoice_vnpt_train_132.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "450,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_058.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt đùi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "68,000"
  },
  {
    "image_name": "convenience_7eleven_train_232.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-Eleven Nguyễn Du\", \"timestamp\": \"17/06/2026 12:06\", \"total_cost\": \"225,720\", \"items\": [{\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"100,000\"}, {\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"40,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"54,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_174.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_190.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "48.000"
  },
  {
    "image_name": "convenience_7eleven_train_052.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "receipt_c45_bb_train_199.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 24 tháng 05 năm 2026"
  },
  {
    "image_name": "convenience_gs25_train_044.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "241,920đ"
  },
  {
    "image_name": "supermarket_winmart_train_259.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 286 Trần Hưng Đạo, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "convenience_gs25_val_044.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Kirin Latte 345ml là bao nhiêu?",
    "ground_truth": "64,000"
  },
  {
    "image_name": "cafe_highlands_val_028.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE\", \"address\": \"Số 33 Nguyễn Trãi, Quận 7, TP. Hồ Chí Minh\", \"timestamp\": \"14/06/2026 20:40\", \"total_cost\": \"322,300\", \"items\": [{\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"1\", \"amount\": \"58,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"4\", \"amount\": \"55,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"1\", \"amount\": \"180,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_113.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 96 Trần Hưng Đạo, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "einvoice_viettel_train_142.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 22:07"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_266.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "supermarket_lotte_train_136.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 81 Lê Lợi, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"25/05/2026 12:28:00\", \"total_cost\": \"907,200.00\", \"items\": [{\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"3\", \"amount\": \"84,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"4\", \"amount\": \"78,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"4\", \"amount\": \"88,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"1\", \"amount\": \"65,000.00\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_214.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước uống Aquafina 500ml, Mì Ly ăn liền Modern, Trà xanh Không Độ 500ml, Xúc xích tiệt trùng Ponnie"
  },
  {
    "image_name": "convenience_gs25_train_145.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,840đ"
  },
  {
    "image_name": "minimart_anan_val_066.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 435 Lê Lợi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_lotte_train_041.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART LÝ THƯỜNG KIỆT\", \"address\": \"Số 258 Trần Hưng Đạo, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"30/05/2026 18:26:00\", \"total_cost\": \"243,000.00\", \"items\": [{\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"2\", \"amount\": \"130,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"2\", \"amount\": \"39,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"2\", \"amount\": \"56,000.00\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_141.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "image_name": "supermarket_winmart_train_209.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "7.000"
  },
  {
    "image_name": "supermarket_lotte_train_029.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,031,800.00"
  },
  {
    "image_name": "supermarket_winmart_train_079.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "3.500"
  },
  {
    "image_name": "cafe_phuclong_train_251.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG NGUYỄN VĂN CỪ\", \"address\": \"Số 412 Cách Mạng Tháng Tám, Quận 1, Cần Thơ\", \"timestamp\": \"27/05/2026 08:37\", \"total_cost\": \"433,675\", \"items\": [{\"name\": \"Cà phê Latte\", \"qty\": \"3\", \"amount\": \"180,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"2\", \"amount\": \"135,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"100,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Freeze Trà Xanh Size M là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_027.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Đường cát trắng Biên Hòa 1kg, Hành lá 100g, Nước mắm Nam Ngư 750ml, Thịt đùi heo 500g, Rau muống nước 500g, Thịt ba rọi heo 500g"
  },
  {
    "image_name": "convenience_7eleven_train_205.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "minimart_anan_train_115.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 393 Nguyễn Huệ, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "einvoice_vnpt_train_097.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "1,850,000"
  },
  {
    "image_name": "einvoice_vnpt_train_022.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "11,781,000đ"
  },
  {
    "image_name": "supermarket_winmart_train_066.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_circlek_train_183.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "192,500đ"
  },
  {
    "image_name": "supermarket_lotte_val_019.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước cam ép Twister 1L, Bột giặt Ariel 3.2kg, Bia Heineken Lon 330ml, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "supermarket_winmart_val_026.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 102 Trần Hưng Đạo, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "convenience_gs25_train_157.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 145 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "image_name": "convenience_7eleven_val_035.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Nguyễn Du\", \"box\": [133, 23, 256, 49]}"
  },
  {
    "image_name": "minimart_anan_val_031.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 12 Nguyễn Trãi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_7eleven_train_175.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 22:16"
  },
  {
    "image_name": "restaurant_kfc_train_054.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Nguyễn Ảnh Thủ"
  },
  {
    "image_name": "convenience_circlek_train_244.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "cafe_highlands_train_144.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_train_001.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_winmart_train_140.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dầu ăn Simply 1L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_021.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_circlek_val_047.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 397 Nguyễn Huệ, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"25/05/2026 14:02\", \"total_cost\": \"92,400đ\", \"items\": [{\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_258.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,600đ"
  },
  {
    "image_name": "convenience_gs25_train_207.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "44,000"
  },
  {
    "image_name": "convenience_gs25_train_156.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_starbucks_val_033.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "supermarket_lotte_val_046.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"597,240.00\", \"box\": [315, 425, 389, 446]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_158.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trứng gà tươi hộp 10 quả trên hóa đơn là bao nhiêu?",
    "ground_truth": "96,000"
  },
  {
    "image_name": "einvoice_vnpt_train_245.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"19,148,400đ\", \"box\": [652, 839, 730, 860]}"
  },
  {
    "image_name": "minimart_anan_train_094.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_070.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OPMART LÝ THƯỜNG KIỆT\", \"address\": \"Số 360 Nguyễn Trãi, Quận Hai Bà Trưng, Hà Nội\", \"timestamp\": \"22/05/2026 18:34:00\", \"total_cost\": \"1,003,320.00\", \"items\": [{\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"4\", \"amount\": \"700,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"2\", \"amount\": \"130,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"2\", \"amount\": \"39,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"1\", \"amount\": \"28,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"1\", \"amount\": \"32,000.00\"}]}"
  },
  {
    "image_name": "minimart_anan_train_264.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "32.000"
  },
  {
    "image_name": "cafe_starbucks_train_219.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_162.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Chiên L trên hóa đơn là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "cafe_starbucks_train_167.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026"
  },
  {
    "image_name": "cafe_starbucks_train_016.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Green Tea Latte M trên hóa đơn là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "cafe_starbucks_train_195.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "supermarket_lotte_train_130.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "700,000.00"
  },
  {
    "image_name": "supermarket_winmart_train_089.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VM+QNH HẠ LONG\", \"box\": [153, 22, 258, 43]}"
  },
  {
    "image_name": "cafe_starbucks_train_005.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "convenience_circlek_val_047.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Băng cá nhân Urgo hộp 10 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "convenience_circlek_train_026.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Băng cá nhân Urgo hộp 10 miếng là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_jollibee_val_020.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "366,120đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_133.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH LÊ ĐỨC THỌ"
  },
  {
    "image_name": "restaurant_jollibee_train_096.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "restaurant_kfc_train_229.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Burger Tôm là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "minimart_anan_train_079.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 08:46"
  },
  {
    "image_name": "convenience_gs25_train_140.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 THỦ ĐỨC\", \"box\": [160, 26, 248, 46]}"
  },
  {
    "image_name": "convenience_circlek_train_103.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 109 Lý Thường Kiệt, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_140.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "cafe_starbucks_train_013.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "255,000"
  },
  {
    "image_name": "cafe_highlands_train_123.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà Phê Đen Đá Size M là bao nhiêu?",
    "ground_truth": "58,000"
  },
  {
    "image_name": "supermarket_winmart_val_067.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "3.500"
  },
  {
    "image_name": "cafe_phuclong_val_027.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 395 Trần Hưng Đạo, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_7eleven_train_059.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Snack khoai tây Lays 95g là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "einvoice_viettel_val_012.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "restaurant_jollibee_train_049.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_7eleven_val_049.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "convenience_gs25_train_251.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 365 Lý Thường Kiệt, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "minimart_anan_val_035.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "16/06/2026 21:57"
  },
  {
    "image_name": "cafe_highlands_val_005.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà Phê Đen Đá Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "29,000"
  },
  {
    "image_name": "restaurant_kfc_train_214.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Chiên L trên hóa đơn là bao nhiêu?",
    "ground_truth": "56,000"
  },
  {
    "image_name": "cafe_highlands_train_253.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE LANDMARK"
  },
  {
    "image_name": "minimart_anan_train_168.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "image_name": "einvoice_vnpt_train_058.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "06/06/2026 15:30"
  },
  {
    "image_name": "receipt_c45_bb_train_021.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1.890.000 VND"
  },
  {
    "image_name": "convenience_circlek_val_055.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "image_name": "restaurant_kfc_val_026.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_winmart_train_215.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dầu ăn Simply 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "168.000"
  },
  {
    "image_name": "receipt_c45_bb_train_229.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 346 Trần Hưng Đạo, Quận 1, Đồng Nai\", \"timestamp\": \"Ngày 07 tháng 06 năm 2026\", \"total_cost\": \"2.090.000 VND\"}"
  },
  {
    "image_name": "restaurant_kfc_val_047.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "restaurant_kfc_train_089.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Trứng Egg Tart, Nước ngọt Pepsi L, Bắp Cải Trộn L, Burger Tôm, Khoai Tây Chiên L, Gà Giòn Cay 1 miếng"
  },
  {
    "image_name": "einvoice_vnpt_train_066.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 91 Trần Hưng Đạo, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "convenience_7eleven_train_056.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "113,400"
  },
  {
    "image_name": "cafe_phuclong_train_077.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "receipt_c45_bb_train_142.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 142 Trần Hưng Đạo, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 19 tháng 06 năm 2026\", \"total_cost\": \"2.052.000 VND\"}"
  },
  {
    "image_name": "cafe_starbucks_train_183.png",
    "template": "cafe_starbucks",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 384 Nguyễn Huệ, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "cafe_phuclong_train_139.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "restaurant_kfc_val_016.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "170,500"
  },
  {
    "image_name": "convenience_gs25_val_041.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_156.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_031.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_lotte_train_218.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Bột giặt Ariel 3.2kg"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_202.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 327 Hai Bà Trưng, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "convenience_circlek_val_045.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước uống Aquafina 500ml, Bánh bao trứng muối, Băng cá nhân Urgo hộp 10 miếng, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "supermarket_winmart_train_153.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "76.000"
  },
  {
    "image_name": "minimart_anan_train_170.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 279 Nguyễn Huệ, Quận 1, Đồng Nai"
  },
  {
    "image_name": "einvoice_viettel_train_266.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_183.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 247 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "restaurant_jollibee_train_102.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "restaurant_kfc_val_041.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"KFC Bà Hom\", \"address\": \"Số 276 Lê Lợi, Quận Hai Bà Trưng, Hà Nội\", \"timestamp\": \"12/06/2026 12:03\", \"total_cost\": \"479,600\", \"items\": [{\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"1\", \"amount\": \"18,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"3\", \"amount\": \"84,000\"}, {\"name\": \"Burger Tôm\", \"qty\": \"4\", \"amount\": \"168,000\"}, {\"name\": \"Bắp Cải Trộn L\", \"qty\": \"4\", \"amount\": \"88,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"2\", \"amount\": \"78,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_080.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Cáp mạng Cat6 UTP 305m, Dịch vụ Lắp đặt camera giám sát, Bàn phím cơ Dareu EK87, Chuột không dây Logitech"
  },
  {
    "image_name": "cafe_phuclong_train_022.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG LÊ LAI"
  },
  {
    "image_name": "einvoice_vnpt_train_036.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"box\": [37, 78, 360, 103]}"
  },
  {
    "image_name": "minimart_anan_train_152.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_042.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước mắm Nam Ngư 750ml là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "convenience_circlek_val_011.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "minimart_anan_train_168.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kem đánh răng Colgate trên hóa đơn là bao nhiêu?",
    "ground_truth": "96.000"
  },
  {
    "image_name": "cafe_highlands_train_191.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "533,628"
  },
  {
    "image_name": "supermarket_winmart_train_046.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH HẠ LONG\", \"address\": \"Số 298 Bùi Thị Xuân, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"05/06/2026 20:29\", \"total_cost\": \"297.000\", \"items\": [{\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"3\", \"amount\": \"87.000\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"2\", \"amount\": \"36.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"4\", \"amount\": \"152.000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_val_002.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "einvoice_viettel_val_061.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "3,000,000"
  },
  {
    "image_name": "restaurant_kfc_train_222.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Chiên L trên hóa đơn là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "convenience_circlek_train_008.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "convenience_7eleven_train_012.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "cafe_highlands_train_170.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "117,000"
  },
  {
    "image_name": "receipt_c45_bb_train_041.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "einvoice_vnpt_val_053.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,550,000"
  },
  {
    "image_name": "convenience_circlek_val_043.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "36,300đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_241.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 258 Nguyễn Trãi, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "cafe_starbucks_train_013.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "convenience_7eleven_train_070.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-Eleven Nguyễn Du"
  },
  {
    "image_name": "einvoice_viettel_train_251.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"box\": [44, 54, 373, 103]}"
  },
  {
    "image_name": "cafe_starbucks_val_048.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_7eleven_train_176.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "180,400"
  },
  {
    "image_name": "restaurant_kfc_train_083.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "16/06/2026 11:56"
  },
  {
    "image_name": "cafe_starbucks_train_034.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cold Brew Coffee M, Chocolate Muffin"
  },
  {
    "image_name": "einvoice_viettel_train_222.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "7,103,160đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_182.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "image_name": "convenience_gs25_train_049.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "22/05/2026 22:38"
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE"
  },
  {
    "image_name": "einvoice_vnpt_train_077.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"box\": [41, 66, 362, 92]}"
  },
  {
    "image_name": "convenience_gs25_train_104.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích Đức ăn liền trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "einvoice_vnpt_train_056.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "07/06/2026 09:48"
  },
  {
    "image_name": "minimart_anan_train_177.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_circlek_val_010.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Băng cá nhân Urgo hộp 10 miếng, Kẹo cao su Cool Air hũ, Xúc xích tiệt trùng Ponnie, Nước uống Aquafina 500ml, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "cafe_phuclong_train_152.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "495,000"
  },
  {
    "image_name": "cafe_phuclong_train_117.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"PHÚC LONG TEA & COFFEE\", \"address\": \"Số 354 Nguyễn Huệ, Quận 1, Bình Dương\", \"timestamp\": \"26/05/2026 16:53\", \"total_cost\": \"369,360\", \"items\": [{\"name\": \"Cà phê Latte\", \"qty\": \"1\", \"amount\": \"180,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"1\", \"amount\": \"55,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"2\", \"amount\": \"45,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"1\", \"amount\": \"50,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"50,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_val_040.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_starbucks_val_035.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "minimart_anan_val_026.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_kfc_train_088.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "minimart_anan_val_007.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "MINIMART ANAN"
  },
  {
    "image_name": "restaurant_kfc_train_221.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 267 Lê Lợi, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_winmart_train_012.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "116.000"
  },
  {
    "image_name": "einvoice_vnpt_train_009.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bàn phím cơ Dareu EK87 trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,800,000"
  },
  {
    "image_name": "supermarket_winmart_train_263.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_263.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "660,744"
  },
  {
    "image_name": "convenience_gs25_val_047.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 326 Cách Mạng Tháng Tám, Quận 1, Bình Dương"
  },
  {
    "image_name": "einvoice_viettel_train_053.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "image_name": "cafe_phuclong_train_035.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hồng Trà Sữa M trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "restaurant_kfc_val_006.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Gà Giòn Cay 1 miếng, Burger Tôm, Bắp Cải Trộn L, Khoai Tây Chiên L, Bánh Trứng Egg Tart, Nước ngọt Pepsi L"
  },
  {
    "image_name": "cafe_starbucks_val_064.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caramel Macchiato L trên hóa đơn là bao nhiêu?",
    "ground_truth": "85,000"
  },
  {
    "image_name": "restaurant_jollibee_train_020.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"JOLLIBEE PASTEUR\", \"address\": \"Số 282 Nguyễn Trãi, Quận 1, Hải Phòng\", \"timestamp\": \"08/06/2026 10:06\", \"total_cost\": \"258,500đ\", \"items\": [{\"name\": \"Nước ngọt 7Up L\", \"qty\": \"3\", \"amount\": \"51,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"2\", \"amount\": \"70,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"3\", \"amount\": \"114,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "cafe_phuclong_train_176.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_139.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_087.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "cafe_starbucks_train_116.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_136.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "200,000"
  },
  {
    "image_name": "cafe_starbucks_train_166.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "590,425"
  },
  {
    "image_name": "convenience_7eleven_train_125.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 09:01"
  },
  {
    "image_name": "restaurant_jollibee_train_148.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "114,000"
  },
  {
    "image_name": "supermarket_winmart_train_197.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VM+QNH CẨM PHẢ\", \"box\": [149, 24, 253, 45]}"
  },
  {
    "image_name": "restaurant_jollibee_train_265.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_train_232.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_val_031.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "minimart_anan_train_203.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "28/05/2026 08:47"
  },
  {
    "image_name": "supermarket_winmart_val_033.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dầu ăn Simply 1L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_021.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 41 Cách Mạng Tháng Tám, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_194.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Rau muống nước 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "restaurant_jollibee_train_181.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"207,900đ\", \"box\": [333, 387, 391, 409]}"
  },
  {
    "image_name": "convenience_7eleven_train_244.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "cafe_highlands_train_258.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 19:34"
  },
  {
    "image_name": "supermarket_winmart_val_050.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dầu ăn Simply 1L, Mì tôm Hảo Hảo chua cay, Sữa tươi TH True Milk 1L, Khăn giấy Pulppy 100 tờ, Bánh Custas Orion 6P"
  },
  {
    "image_name": "cafe_starbucks_train_129.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Green Tea Latte M trên hóa đơn là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "einvoice_viettel_train_187.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 426 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_starbucks_train_020.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_092.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"149,600đ\", \"box\": [335, 472, 395, 493]}"
  },
  {
    "image_name": "einvoice_viettel_val_063.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_highlands_train_101.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Freeze Trà Xanh Size M là bao nhiêu?",
    "ground_truth": "110,000"
  },
  {
    "image_name": "minimart_anan_train_251.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 299 Nguyễn Trãi, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_circlek_train_049.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "cafe_starbucks_train_251.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026"
  },
  {
    "image_name": "minimart_anan_train_184.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CỬA HÀNG TIỆN LỢI AN AN\", \"address\": \"Số 237 Bùi Thị Xuân, Quận 1, Cần Thơ\", \"timestamp\": \"11/06/2026 07:40\", \"total_cost\": \"92.880\", \"items\": [{\"name\": \"Bánh bao trứng muối\", \"qty\": \"2\", \"amount\": \"32.000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"4\", \"amount\": \"36.000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"3\", \"amount\": \"18.000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_036.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE\", \"address\": \"Số 371 Nguyễn Huệ, Quận Ba Đình, Hà Nội\", \"timestamp\": \"12/06/2026 13:23\", \"total_cost\": \"911,520\", \"items\": [{\"name\": \"Bánh Tiramisu\", \"qty\": \"4\", \"amount\": \"105,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"4\", \"amount\": \"220,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"3\", \"amount\": \"180,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"3\", \"amount\": \"57,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"3\", \"amount\": \"165,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_051.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trứng gà tươi hộp 10 quả trên hóa đơn là bao nhiêu?",
    "ground_truth": "96,000"
  },
  {
    "image_name": "cafe_phuclong_train_205.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "einvoice_vnpt_train_172.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "17/06/2026 17:26"
  },
  {
    "image_name": "convenience_gs25_train_148.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "supermarket_winmart_val_018.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VM+HN TRẦN DUY HƯNG\", \"box\": [133, 35, 271, 58]}"
  },
  {
    "image_name": "convenience_7eleven_train_076.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì xào tương đen trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "einvoice_viettel_train_172.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_172.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART LÝ THƯỜNG KIỆT"
  },
  {
    "image_name": "convenience_7eleven_val_033.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "72,000"
  },
  {
    "image_name": "restaurant_jollibee_train_178.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "70,000"
  },
  {
    "image_name": "einvoice_vnpt_train_143.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chuột không dây Logitech trên hóa đơn là bao nhiêu?",
    "ground_truth": "290,000"
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "supermarket_lotte_train_182.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 434 Lý Thường Kiệt, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"11/06/2026 15:46:00\", \"total_cost\": \"146,340.00\", \"items\": [{\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"4\", \"amount\": \"88,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"1\", \"amount\": \"19,500.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"1\", \"amount\": \"28,000.00\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_073.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "60,500đ"
  },
  {
    "image_name": "receipt_c45_bb_val_038.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 31 tháng 05 năm 2026"
  },
  {
    "image_name": "supermarket_winmart_train_226.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Khăn giấy Pulppy 100 tờ, Nước rửa chén Sunlight 750ml"
  },
  {
    "image_name": "convenience_circlek_train_160.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 196 Trần Hưng Đạo, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_phuclong_train_066.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Lài Đác Thơm L, Trà Đào Phúc Long L, Bánh Croissant Bơ Pháp"
  },
  {
    "image_name": "supermarket_winmart_train_164.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 325 Nguyễn Huệ, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_starbucks_train_034.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "convenience_7eleven_train_229.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "restaurant_jollibee_train_231.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_156.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Phin Sữa Đá Size L, Trà Thạch Đào Size L"
  },
  {
    "image_name": "einvoice_viettel_train_042.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mực in Canon Cartridge, Bút bi Thiên Long FO-03, Dịch vụ Bảo trì Hệ thống mạng"
  },
  {
    "image_name": "restaurant_jollibee_val_061.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 62 Bùi Thị Xuân, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "einvoice_vnpt_train_110.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,850,400đ"
  },
  {
    "image_name": "einvoice_viettel_train_089.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 12:14"
  },
  {
    "image_name": "supermarket_lotte_train_105.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "15/06/2026 12:29:00"
  },
  {
    "image_name": "supermarket_lotte_train_243.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 134 Hai Bà Trưng, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_train_080.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "5,000,000"
  },
  {
    "image_name": "einvoice_viettel_train_256.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "3,750,000"
  },
  {
    "image_name": "einvoice_vnpt_train_237.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bộ phát Wifi TP-Link Archer, Cáp mạng Cat6 UTP 305m, Dịch vụ Lắp đặt camera giám sát"
  },
  {
    "image_name": "cafe_starbucks_train_150.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS REX HOTEL\", \"address\": \"Số 246 Cách Mạng Tháng Tám, Quận Đống Đa, Hà Nội\", \"timestamp\": \"24/05/2026\", \"total_cost\": \"1,023,000\", \"items\": [{\"name\": \"Green Tea Latte M\", \"qty\": \"2\", \"amount\": \"150,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"4\", \"amount\": \"260,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"3\", \"amount\": \"180,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"3\", \"amount\": \"120,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_088.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "supermarket_winmart_train_066.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Custas Orion 6P trên hóa đơn là bao nhiêu?",
    "ground_truth": "136.000"
  },
  {
    "image_name": "einvoice_viettel_train_124.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "130,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_051.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "96,000"
  },
  {
    "image_name": "convenience_gs25_train_100.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "GS25 LANDMARK 81"
  },
  {
    "image_name": "cafe_phuclong_train_090.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "18/06/2026 14:22"
  },
  {
    "image_name": "restaurant_kfc_train_049.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 371 Bùi Thị Xuân, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "restaurant_jollibee_train_037.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Gà Giòn Vui Vẻ 1 miếng, Mì Ý Sốt Bò Bằm, Khoai Tây Lắc Phô Mai, Bánh Khoai Môn, Nước ngọt 7Up L"
  },
  {
    "image_name": "restaurant_kfc_train_078.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Khoai Tây Chiên L, Gà Giòn Cay 1 miếng, Burger Tôm, Bắp Cải Trộn L, Nước ngọt Pepsi L"
  },
  {
    "image_name": "minimart_anan_train_201.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "564.300"
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà Phê Đen Đá Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "87,000"
  },
  {
    "image_name": "cafe_phuclong_train_063.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Đào Phúc Long L, Trà sữa Phúc Long L, Cà phê Latte"
  },
  {
    "image_name": "convenience_gs25_train_109.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "71,280đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_057.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"770,000\", \"box\": [313, 580, 365, 601]}"
  },
  {
    "image_name": "convenience_gs25_train_221.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "132,840đ"
  },
  {
    "image_name": "restaurant_jollibee_train_134.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "224,640đ"
  },
  {
    "image_name": "cafe_highlands_val_021.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Thạch Đào Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "cafe_starbucks_train_216.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"STARBUCKS REX HOTEL\", \"box\": [114, 84, 272, 109]}"
  },
  {
    "image_name": "convenience_7eleven_train_072.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sandwich xá xíu phô mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "einvoice_vnpt_train_254.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "10,098,000đ"
  },
  {
    "image_name": "restaurant_jollibee_train_212.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "27/05/2026 10:58"
  },
  {
    "image_name": "supermarket_winmart_train_249.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_vnpt_train_128.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "22/05/2026 18:50"
  },
  {
    "image_name": "minimart_anan_train_079.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Xúc xích tiệt trùng Ponnie, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "cafe_phuclong_val_045.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 212 Trần Hưng Đạo, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "minimart_anan_train_029.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_lotte_train_111.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"590,760.00\", \"box\": [314, 596, 387, 616]}"
  },
  {
    "image_name": "restaurant_kfc_val_051.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"176,000\", \"box\": [324, 385, 382, 408]}"
  },
  {
    "image_name": "supermarket_winmart_train_227.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 420 Lê Lợi, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "convenience_gs25_val_043.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 363 Nguyễn Trãi, Quận Thanh Xuân, Hà Nội"
  },
  {
    "image_name": "cafe_highlands_train_084.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "421,300"
  },
  {
    "image_name": "cafe_phuclong_train_163.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LÊ LAI\", \"address\": \"Số 90 Lý Thường Kiệt, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"31/05/2026 19:45\", \"total_cost\": \"550,000\", \"items\": [{\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"3\", \"amount\": \"150,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"2\", \"amount\": \"75,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"3\", \"amount\": \"110,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"1\", \"amount\": \"165,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_230.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "cafe_phuclong_train_195.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"696,300\", \"box\": [305, 428, 357, 449]}"
  },
  {
    "image_name": "restaurant_jollibee_train_025.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "supermarket_lotte_train_005.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hộp dâu tây Đà Lạt 250g là bao nhiêu?",
    "ground_truth": "130,000.00"
  },
  {
    "image_name": "cafe_phuclong_train_002.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 123 Nguyễn Trãi, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "cafe_starbucks_train_140.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_243.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "128,000"
  },
  {
    "image_name": "restaurant_kfc_train_025.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_120.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 389 Hai Bà Trưng, Quận 1, Bình Dương"
  },
  {
    "image_name": "convenience_7eleven_train_163.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "supermarket_winmart_train_182.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước rửa chén Sunlight 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "116.000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_069.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"110,000\", \"box\": [311, 418, 364, 441]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_174.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trứng gà tươi hộp 10 quả trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "convenience_7eleven_train_190.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "11/06/2026 19:31"
  },
  {
    "image_name": "supermarket_lotte_train_188.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Bột giặt Ariel 3.2kg, Bia Heineken Lon 330ml, Kem đánh răng Colgate"
  },
  {
    "image_name": "receipt_c45_bb_train_094.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"216.000 VND\", \"box\": [128, 255, 226, 271]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_163.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "supermarket_lotte_train_037.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_jollibee_train_189.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE VINCOM\", \"address\": \"Số 374 Nguyễn Huệ, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"04/06/2026 18:17\", \"total_cost\": \"349,800đ\", \"items\": [{\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"2\", \"amount\": \"50,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"4\", \"amount\": \"140,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"4\", \"amount\": \"68,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_val_009.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VINCOMMERCE\", \"box\": [166, 30, 245, 52]}"
  },
  {
    "image_name": "supermarket_lotte_train_140.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_circlek_train_029.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_train_018.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_kfc_val_052.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "einvoice_vnpt_val_054.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_val_033.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_train_188.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_156.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 336 Hai Bà Trưng, Quận 1, Đồng Nai"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_193.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Đường cát trắng Biên Hòa 1kg trên hóa đơn là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "restaurant_kfc_train_098.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "restaurant_kfc_train_008.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "261,360"
  },
  {
    "image_name": "cafe_starbucks_train_101.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "cafe_highlands_train_266.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 61 Lê Lợi, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"13/06/2026 18:44\", \"total_cost\": \"396,000\", \"items\": [{\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"4\", \"amount\": \"180,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"1\", \"amount\": \"220,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_val_055.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Nguyễn Ảnh Thủ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_021.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH\", \"address\": \"Số 105 Bùi Thị Xuân, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"09/06/2026 12:20\", \"total_cost\": \"306,720\", \"items\": [{\"name\": \"Hành lá 100g\", \"qty\": \"4\", \"amount\": \"12,000\"}, {\"name\": \"Thịt đùi heo 500g\", \"qty\": \"4\", \"amount\": \"272,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_027.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"address\": \"Số 300 Hai Bà Trưng, Quận 1, Đồng Nai\", \"timestamp\": \"23/05/2026 22:16\", \"total_cost\": \"12,091,200đ\", \"items\": [{\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"3\", \"amount\": \"12,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"3\", \"amount\": \"4,500,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"2\", \"amount\": \"2,500,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"2\", \"amount\": \"130,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_val_065.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS REX HOTEL\", \"address\": \"Số 433 Nguyễn Huệ, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"23/05/2026\", \"total_cost\": \"1,034,550\", \"items\": [{\"name\": \"Butter Croissant\", \"qty\": \"3\", \"amount\": \"120,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"2\", \"amount\": \"120,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"4\", \"amount\": \"260,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"4\", \"amount\": \"180,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_165.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"MINIMART ANAN\", \"address\": \"Số 109 Cách Mạng Tháng Tám, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"04/06/2026 19:38\", \"total_cost\": \"622.600\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24.000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"1\", \"amount\": \"22.000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"2\", \"amount\": \"18.000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"4\", \"amount\": \"128.000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_050.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 342 Hai Bà Trưng, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"14/06/2026 17:48:00\", \"total_cost\": \"699,600.00\", \"items\": [{\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"3\", \"amount\": \"84,000.00\"}, {\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"1\", \"amount\": \"27,000.00\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_057.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_gs25_train_189.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"GS25 NGUYỄN HUỆ\", \"address\": \"Số 132 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"15/06/2026 17:33\", \"total_cost\": \"151,200đ\", \"items\": [{\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"3\", \"amount\": \"48,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"4\", \"amount\": \"48,000\"}, {\"name\": \"Cafe sữa đá GS25\", \"qty\": \"2\", \"amount\": \"44,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_072.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì tôm Hảo Hảo chua cay trên hóa đơn là bao nhiêu?",
    "ground_truth": "10.500"
  },
  {
    "image_name": "supermarket_lotte_train_161.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "175,000.00"
  },
  {
    "image_name": "cafe_phuclong_train_207.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "image_name": "cafe_highlands_val_064.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "574,200"
  },
  {
    "image_name": "cafe_starbucks_train_174.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caffe Americano L là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "cafe_highlands_train_002.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "supermarket_lotte_train_127.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"address\": \"Số 256 Lê Lợi, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"05/06/2026 22:03:00\", \"total_cost\": \"542,300.00\", \"items\": [{\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"3\", \"amount\": \"81,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"4\", \"amount\": \"88,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"2\", \"amount\": \"64,000.00\"}, {\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"4\", \"amount\": \"260,000.00\"}]}"
  },
  {
    "image_name": "restaurant_kfc_val_056.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "402,840"
  },
  {
    "image_name": "cafe_highlands_train_255.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE LÊ LỢI"
  },
  {
    "image_name": "supermarket_winmart_train_138.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Custas Orion 6P trên hóa đơn là bao nhiêu?",
    "ground_truth": "102.000"
  },
  {
    "image_name": "receipt_c45_bb_train_090.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 226 Hai Bà Trưng, Quận Thanh Xuân, Hà Nội"
  },
  {
    "image_name": "einvoice_viettel_train_167.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 11:42"
  },
  {
    "image_name": "einvoice_vnpt_val_038.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"address\": \"Số 300 Trần Hưng Đạo, Quận 1, Hải Phòng\", \"timestamp\": \"26/05/2026 16:08\", \"total_cost\": \"16,426,800đ\", \"items\": [{\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"2\", \"amount\": \"1,700,000\"}, {\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"3\", \"amount\": \"5,550,000\"}, {\"name\": \"Chuột không dây Logitech\", \"qty\": \"4\", \"amount\": \"1,160,000\"}, {\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"2\", \"amount\": \"5,000,000\"}, {\"name\": \"Bàn phím cơ Dareu EK87\", \"qty\": \"4\", \"amount\": \"1,800,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_val_012.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 THỦ ĐỨC"
  },
  {
    "image_name": "minimart_anan_train_172.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "08/06/2026 14:06"
  },
  {
    "image_name": "einvoice_viettel_train_111.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "9,200,400đ"
  },
  {
    "image_name": "restaurant_jollibee_train_060.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt 7Up L trên hóa đơn là bao nhiêu?",
    "ground_truth": "51,000"
  },
  {
    "image_name": "supermarket_lotte_val_016.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_train_010.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "915,750"
  },
  {
    "image_name": "einvoice_vnpt_train_158.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "7,400,000"
  },
  {
    "image_name": "receipt_c45_bb_train_187.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "supermarket_winmart_train_219.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"VM+QNH HẠ LONG\", \"address\": \"Số 216 Lê Lợi, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"03/06/2026 12:45\", \"total_cost\": \"336.600\", \"items\": [{\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"4\", \"amount\": \"72.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"1\", \"amount\": \"38.000\"}, {\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"1\", \"amount\": \"10.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"4\", \"amount\": \"14.000\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"4\", \"amount\": \"116.000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_136.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "restaurant_kfc_train_088.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_viettel_val_008.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "einvoice_viettel_train_017.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "13,424,400đ"
  },
  {
    "image_name": "einvoice_viettel_train_038.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_141.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "54.000"
  },
  {
    "image_name": "convenience_circlek_train_043.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "10/06/2026 07:20"
  },
  {
    "image_name": "convenience_gs25_train_074.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Kirin Latte 345ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "64,000"
  },
  {
    "image_name": "receipt_c45_bb_train_100.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 138 Nguyễn Trãi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_circlek_val_004.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "supermarket_lotte_train_200.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hộp dâu tây Đà Lạt 250g là bao nhiêu?",
    "ground_truth": "130,000.00"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_049.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hành lá 100g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_7eleven_train_231.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước khoáng Dasani 500ml là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "restaurant_kfc_train_163.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 363 Cách Mạng Tháng Tám, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_7eleven_train_208.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Snack khoai tây Lays 95g là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "receipt_c45_bb_train_198.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 07 tháng 06 năm 2026"
  },
  {
    "image_name": "convenience_gs25_train_179.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_kfc_train_121.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt Pepsi L trên hóa đơn là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "cafe_highlands_train_102.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 179 Lý Thường Kiệt, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "supermarket_winmart_val_059.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước rửa chén Sunlight 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "29.000"
  },
  {
    "image_name": "cafe_phuclong_train_262.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà sữa Phúc Long L, Trà Đào Phúc Long L, Hồng Trà Sữa M, Cà phê Latte"
  },
  {
    "image_name": "restaurant_jollibee_train_113.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_046.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hành lá 100g trên hóa đơn là bao nhiêu?",
    "ground_truth": "3,000"
  },
  {
    "image_name": "convenience_circlek_val_013.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "203,040đ"
  },
  {
    "image_name": "cafe_phuclong_train_051.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_079.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 48 Lê Lợi, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_winmart_train_061.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+HN TRẦN DUY HƯNG\", \"address\": \"Số 420 Nguyễn Huệ, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"01/06/2026 09:04\", \"total_cost\": \"22.140\", \"items\": [{\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"1\", \"amount\": \"10.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"3\", \"amount\": \"10.500\"}]}"
  },
  {
    "image_name": "minimart_anan_train_014.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "05/06/2026 09:57"
  },
  {
    "image_name": "einvoice_vnpt_train_110.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "5,550,000"
  },
  {
    "image_name": "convenience_gs25_val_022.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "supermarket_lotte_val_001.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "image_name": "einvoice_viettel_train_087.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Máy in HP LaserJet Pro trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,400,000"
  },
  {
    "image_name": "supermarket_winmart_train_019.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dầu ăn Simply 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "224.000"
  },
  {
    "image_name": "supermarket_lotte_train_030.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "261,800.00"
  },
  {
    "image_name": "convenience_gs25_train_069.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 TÔN THẤT THUYẾT\", \"box\": [132, 20, 280, 44]}"
  },
  {
    "image_name": "einvoice_viettel_train_053.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 37 Nguyễn Trãi, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"10/06/2026 11:25\", \"total_cost\": \"18,954,000đ\", \"items\": [{\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"4\", \"amount\": \"6,000,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"3\", \"amount\": \"11,550,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_176.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 08:34"
  },
  {
    "image_name": "einvoice_viettel_train_236.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"address\": \"Số 177 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"09/06/2026 17:32\", \"total_cost\": \"20,788,900đ\", \"items\": [{\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"2\", \"amount\": \"7,700,000\"}, {\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"1\", \"amount\": \"4,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"4\", \"amount\": \"5,000,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"4\", \"amount\": \"6,000,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"3\", \"amount\": \"195,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_val_046.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Mì trộn Indomie đặc biệt, Xúc xích Đức ăn liền, Cơm nắm cá hồi sốt Mayo, Cafe sữa đá GS25, Trà sữa Kirin Latte 345ml, Bánh giò thịt băm trứng cút"
  },
  {
    "image_name": "cafe_starbucks_train_250.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "353,970"
  },
  {
    "image_name": "cafe_highlands_val_058.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Mì Que Gà Xé trên hóa đơn là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "convenience_circlek_train_240.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "image_name": "convenience_circlek_train_227.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "image_name": "supermarket_lotte_train_057.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh quy Cosy Kinh Đô trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000.00"
  },
  {
    "image_name": "convenience_circlek_val_027.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"233,200đ\", \"box\": [326, 496, 383, 518]}"
  },
  {
    "image_name": "einvoice_viettel_train_014.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "receipt_c45_bb_train_242.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 19 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "einvoice_viettel_train_054.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "image_name": "supermarket_winmart_train_026.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì tôm Hảo Hảo chua cay trên hóa đơn là bao nhiêu?",
    "ground_truth": "3.500"
  },
  {
    "image_name": "cafe_highlands_train_179.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"743,600\", \"box\": [307, 442, 360, 464]}"
  },
  {
    "image_name": "convenience_7eleven_train_067.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Snack khoai tây Lays 95g là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "minimart_anan_train_104.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_starbucks_train_162.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 90 Nguyễn Huệ, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"18/06/2026\", \"total_cost\": \"231,000\", \"items\": [{\"name\": \"Chocolate Muffin\", \"qty\": \"3\", \"amount\": \"135,000\"}, {\"name\": \"Green Tea Latte M\", \"qty\": \"1\", \"amount\": \"75,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_106.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "415,800đ"
  },
  {
    "image_name": "supermarket_lotte_train_151.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hộp dâu tây Đà Lạt 250g trên hóa đơn là bao nhiêu?",
    "ground_truth": "130,000.00"
  },
  {
    "image_name": "convenience_gs25_train_095.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_gs25_train_092.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "minimart_anan_train_016.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ANAN CONVENIENCE STORE\", \"address\": \"Số 181 Trần Hưng Đạo, Quận 1, Cần Thơ\", \"timestamp\": \"01/06/2026 14:40\", \"total_cost\": \"193.600\", \"items\": [{\"name\": \"Bánh bao trứng muối\", \"qty\": \"3\", \"amount\": \"48.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"4\", \"amount\": \"128.000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_157.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "cafe_highlands_train_227.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 440 Trần Hưng Đạo, Quận 1, Hải Phòng\", \"timestamp\": \"12/06/2026 08:26\", \"total_cost\": \"811,800\", \"items\": [{\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"3\", \"amount\": \"116,000\"}, {\"name\": \"Bánh Tiramisu\", \"qty\": \"4\", \"amount\": \"105,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"2\", \"amount\": \"220,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"4\", \"amount\": \"38,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"1\", \"amount\": \"220,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_val_021.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 14:00"
  },
  {
    "image_name": "convenience_circlek_train_166.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CIRCLE K\", \"address\": \"Số 386 Nguyễn Trãi, Quận Cầu Giấy, Hà Nội\", \"timestamp\": \"30/05/2026 07:41\", \"total_cost\": \"28,080đ\", \"items\": [{\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"1\", \"amount\": \"10,000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"1\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_208.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "27/05/2026 12:07"
  },
  {
    "image_name": "supermarket_winmart_val_003.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"VINCOMMERCE\", \"address\": \"Số 308 Nguyễn Huệ, Quận 1, Bình Dương\", \"timestamp\": \"05/06/2026 09:14\", \"total_cost\": \"229.500\", \"items\": [{\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"3\", \"amount\": \"30.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"1\", \"amount\": \"3.500\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"1\", \"amount\": \"29.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"3\", \"amount\": \"114.000\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"2\", \"amount\": \"36.000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_232.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cà phê Latte, Hồng Trà Sữa M, Trà Lài Đác Thơm L, Trà sữa Phúc Long L, Trà Đào Phúc Long L, Bánh Croissant Bơ Pháp"
  },
  {
    "image_name": "supermarket_winmart_val_039.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "114.000"
  },
  {
    "image_name": "supermarket_winmart_train_171.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH CẨM PHẢ\", \"address\": \"Số 261 Hai Bà Trưng, Quận 1, Cần Thơ\", \"timestamp\": \"28/05/2026 19:53\", \"total_cost\": \"241.920\", \"items\": [{\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"2\", \"amount\": \"112.000\"}, {\"name\": \"Bánh Custas Orion 6P\", \"qty\": \"3\", \"amount\": \"102.000\"}, {\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"1\", \"amount\": \"10.000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_074.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "114.400"
  },
  {
    "image_name": "restaurant_jollibee_train_233.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE COOPMART\", \"box\": [141, 15, 269, 36]}"
  },
  {
    "image_name": "restaurant_kfc_train_046.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Nguyễn Ảnh Thủ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_024.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 40 Trần Hưng Đạo, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "supermarket_winmart_train_101.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 209 Hai Bà Trưng, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_winmart_val_044.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dầu ăn Simply 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "168.000"
  },
  {
    "image_name": "cafe_starbucks_train_186.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "cafe_phuclong_train_093.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Lài Đác Thơm L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_072.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "18/06/2026 14:49"
  },
  {
    "image_name": "restaurant_kfc_val_040.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bắp Cải Trộn L trên hóa đơn là bao nhiêu?",
    "ground_truth": "22,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_165.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH QUẬN 12"
  },
  {
    "image_name": "supermarket_winmart_train_095.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn giấy Pulppy 100 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "54.000"
  },
  {
    "image_name": "restaurant_jollibee_train_114.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "restaurant_kfc_val_008.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Cay 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "156,000"
  },
  {
    "image_name": "einvoice_viettel_train_260.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "19,080,600đ"
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_173.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "14/06/2026 11:37"
  },
  {
    "image_name": "minimart_anan_train_180.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_phuclong_train_121.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG TEA & COFFEE\", \"address\": \"Số 369 Cách Mạng Tháng Tám, Quận Cầu Giấy, Hà Nội\", \"timestamp\": \"23/05/2026 13:41\", \"total_cost\": \"346,500\", \"items\": [{\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"4\", \"amount\": \"55,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"1\", \"amount\": \"180,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"55,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"1\", \"amount\": \"25,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_035.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "einvoice_vnpt_val_064.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cáp mạng Cat6 UTP 305m, Bộ phát Wifi TP-Link Archer"
  },
  {
    "image_name": "supermarket_winmart_val_042.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VINCOMMERCE\", \"box\": [164, 22, 246, 43]}"
  },
  {
    "image_name": "convenience_circlek_train_109.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "restaurant_kfc_train_210.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "supermarket_winmart_train_188.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn giấy Pulppy 100 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "36.000"
  },
  {
    "image_name": "cafe_phuclong_val_049.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG LÊ LAI"
  },
  {
    "image_name": "convenience_circlek_train_127.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo cao su Cool Air hũ trên hóa đơn là bao nhiêu?",
    "ground_truth": "44,000"
  },
  {
    "image_name": "convenience_gs25_val_008.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_225.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "439,560"
  },
  {
    "image_name": "convenience_circlek_train_192.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_125.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Rau muống nước 500g, Thịt đùi heo 500g, Nước mắm Nam Ngư 750ml, Đường cát trắng Biên Hòa 1kg, Trứng gà tươi hộp 10 quả"
  },
  {
    "image_name": "cafe_phuclong_train_051.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hồng Trà Sữa M trên hóa đơn là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "restaurant_jollibee_train_172.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"JOLLIBEE PASTEUR\", \"address\": \"Số 445 Nguyễn Huệ, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"18/06/2026 14:49\", \"total_cost\": \"364,100đ\", \"items\": [{\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"3\", \"amount\": \"105,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"1\", \"amount\": \"17,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"3\", \"amount\": \"114,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"2\", \"amount\": \"50,000\"}]}"
  },
  {
    "image_name": "receipt_c45_bb_train_080.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "restaurant_jollibee_train_066.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE VINCOM\", \"address\": \"Số 213 Hai Bà Trưng, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"24/05/2026 17:45\", \"total_cost\": \"470,800đ\", \"items\": [{\"name\": \"Bánh Khoai Môn\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"4\", \"amount\": \"152,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"3\", \"amount\": \"75,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"3\", \"amount\": \"51,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"3\", \"amount\": \"105,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_168.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC Nguyễn Thái Học\", \"address\": \"Số 320 Nguyễn Huệ, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"04/06/2026 08:53\", \"total_cost\": \"336,600\", \"items\": [{\"name\": \"Burger Tôm\", \"qty\": \"4\", \"amount\": \"168,000\"}, {\"name\": \"Bắp Cải Trộn L\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"4\", \"amount\": \"72,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_val_002.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước cam ép Twister 1L là bao nhiêu?",
    "ground_truth": "84,000.00"
  },
  {
    "image_name": "supermarket_winmart_train_167.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 238 Trần Hưng Đạo, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_val_041.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "850,000"
  },
  {
    "image_name": "einvoice_vnpt_val_002.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "image_name": "cafe_highlands_train_208.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "788,975"
  },
  {
    "image_name": "convenience_circlek_train_253.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"Circle K Lê Lợi\", \"address\": \"Số 406 Lê Lợi, Quận Cầu Giấy, Hà Nội\", \"timestamp\": \"13/06/2026 13:52\", \"total_cost\": \"154,440đ\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"3\", \"amount\": \"30,000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"1\", \"amount\": \"15,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_049.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE PASTEUR\", \"box\": [145, 20, 265, 41]}"
  },
  {
    "image_name": "convenience_gs25_train_225.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "14/06/2026 15:58"
  },
  {
    "image_name": "convenience_gs25_val_007.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 THỦ ĐỨC\", \"address\": \"Số 60 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"03/06/2026 19:39\", \"total_cost\": \"134,200đ\", \"items\": [{\"name\": \"Cafe sữa đá GS25\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Sữa chua uống Proby\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"1\", \"amount\": \"12,000\"}, {\"name\": \"Xúc xích Đức ăn liền\", \"qty\": \"2\", \"amount\": \"20,000\"}, {\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"1\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_222.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "receipt_c45_bb_train_106.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 413 Hai Bà Trưng, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_044.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_195.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "15/06/2026 13:23"
  },
  {
    "image_name": "restaurant_jollibee_train_225.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "image_name": "convenience_7eleven_train_114.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì xào tương đen trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "supermarket_lotte_train_067.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_gs25_train_245.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "supermarket_winmart_train_176.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "241.450"
  },
  {
    "image_name": "einvoice_viettel_train_162.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_7eleven_val_022.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "convenience_circlek_train_243.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo cao su Cool Air hũ trên hóa đơn là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "restaurant_kfc_val_028.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Cay 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "39,000"
  },
  {
    "image_name": "supermarket_lotte_val_007.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hộp dâu tây Đà Lạt 250g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_vnpt_train_209.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Lắp đặt camera giám sát trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000,000"
  },
  {
    "image_name": "cafe_highlands_train_205.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Thạch Đào Size L, Bánh Tiramisu, Bánh Mì Que Gà Xé, Freeze Trà Xanh Size M, Trà Sen Vàng Size M, Cà Phê Đen Đá Size M"
  },
  {
    "image_name": "cafe_starbucks_val_048.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Caffe Americano L, Chocolate Muffin, Butter Croissant, Caramel Macchiato L, Green Tea Latte M"
  },
  {
    "image_name": "convenience_gs25_train_191.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh giò thịt băm trứng cút, Cafe sữa đá GS25, Cơm nắm cá hồi sốt Mayo, Sữa chua uống Proby"
  },
  {
    "image_name": "supermarket_lotte_train_165.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE LÊ LỢI"
  },
  {
    "image_name": "convenience_gs25_train_007.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sữa chua uống Proby trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "einvoice_viettel_train_127.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "4,594,320đ"
  },
  {
    "image_name": "restaurant_jollibee_train_130.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_gs25_val_034.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "41,800đ"
  },
  {
    "image_name": "restaurant_jollibee_train_182.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Mì Ý Sốt Bò Bằm, Bánh Khoai Môn"
  },
  {
    "image_name": "cafe_starbucks_train_242.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_040.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 167 Lê Lợi, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "image_name": "convenience_circlek_train_235.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_087.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà đào sả Slurpee, Mì xào tương đen, Sandwich xá xíu phô mai, Snack khoai tây Lays 95g, Kẹo dẻo Haribo Goldbears"
  },
  {
    "image_name": "convenience_7eleven_train_075.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước khoáng Dasani 500ml, Kẹo dẻo Haribo Goldbears"
  },
  {
    "image_name": "cafe_starbucks_train_020.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Green Tea Latte M, Caramel Macchiato L, Butter Croissant, Cold Brew Coffee M, Chocolate Muffin, Caffe Americano L"
  },
  {
    "image_name": "cafe_phuclong_val_015.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG NGUYỄN VĂN CỪ"
  },
  {
    "image_name": "restaurant_jollibee_val_043.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_starbucks_train_202.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "supermarket_winmart_train_039.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "292.680"
  },
  {
    "image_name": "cafe_phuclong_train_200.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_165.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "171.720"
  },
  {
    "image_name": "restaurant_jollibee_train_259.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "convenience_gs25_train_071.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Kirin Latte 345ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_7eleven_train_263.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "convenience_gs25_train_069.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "restaurant_kfc_train_056.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_139.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Nguyễn Thái Học\", \"box\": [124, 15, 286, 36]}"
  },
  {
    "image_name": "einvoice_vnpt_val_031.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_214.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "restaurant_kfc_train_094.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Trứng Egg Tart trên hóa đơn là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "einvoice_vnpt_train_136.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH\", \"box\": [33, 54, 371, 75]}"
  },
  {
    "image_name": "einvoice_vnpt_train_097.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Dịch vụ Lắp đặt camera giám sát, Cáp mạng Cat6 UTP 305m, Chuột không dây Logitech, Bàn phím cơ Dareu EK87"
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Tiramisu trên hóa đơn là bao nhiêu?",
    "ground_truth": "140,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_179.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH QUẬN 12\", \"box\": [114, 26, 266, 48]}"
  },
  {
    "image_name": "convenience_7eleven_train_215.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sandwich xá xíu phô mai là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "supermarket_winmart_val_013.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sữa tươi TH True Milk 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "152.000"
  },
  {
    "image_name": "convenience_7eleven_train_159.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"298,100\", \"box\": [310, 297, 361, 322]}"
  },
  {
    "image_name": "einvoice_vnpt_train_171.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"address\": \"Số 266 Nguyễn Trãi, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"23/05/2026 18:19\", \"total_cost\": \"20,098,800đ\", \"items\": [{\"name\": \"Chuột không dây Logitech\", \"qty\": \"4\", \"amount\": \"1,160,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"3\", \"amount\": \"2,550,000\"}, {\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"3\", \"amount\": \"7,500,000\"}, {\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"4\", \"amount\": \"7,400,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_224.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Croissant Bơ Pháp trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE"
  },
  {
    "image_name": "convenience_circlek_train_148.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "convenience_7eleven_val_052.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo dẻo Haribo Goldbears trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "supermarket_lotte_train_263.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "08/06/2026 13:22:00"
  },
  {
    "image_name": "receipt_c45_bb_train_143.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 392 Hai Bà Trưng, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_starbucks_train_059.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 184 Trần Hưng Đạo, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"21/05/2026\", \"total_cost\": \"787,050\", \"items\": [{\"name\": \"Cold Brew Coffee M\", \"qty\": \"4\", \"amount\": \"260,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"3\", \"amount\": \"120,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_025.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH HẠ LONG\", \"address\": \"Số 227 Lê Lợi, Quận 1, Đồng Nai\", \"timestamp\": \"14/06/2026 20:54\", \"total_cost\": \"384.450\", \"items\": [{\"name\": \"Bánh Custas Orion 6P\", \"qty\": \"3\", \"amount\": \"102.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"3\", \"amount\": \"168.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"1\", \"amount\": \"3.500\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"1\", \"amount\": \"18.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"1\", \"amount\": \"38.000\"}]}"
  },
  {
    "image_name": "minimart_anan_val_008.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "convenience_7eleven_train_074.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà đào sả Slurpee, Snack khoai tây Lays 95g"
  },
  {
    "image_name": "cafe_phuclong_val_014.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LANDMARK\", \"address\": \"Số 129 Nguyễn Huệ, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"12/06/2026 09:17\", \"total_cost\": \"688,050\", \"items\": [{\"name\": \"Cà phê Latte\", \"qty\": \"4\", \"amount\": \"90,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"3\", \"amount\": \"220,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"2\", \"amount\": \"75,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"100,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"3\", \"amount\": \"45,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_248.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "supermarket_lotte_train_094.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 345 Cách Mạng Tháng Tám, Thành phố Thủ Đức, TP. Hồ Chí Minh"
  },
  {
    "image_name": "supermarket_winmart_val_040.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 12:00"
  },
  {
    "image_name": "convenience_gs25_train_182.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "174,960đ"
  },
  {
    "image_name": "convenience_7eleven_train_084.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Snack khoai tây Lays 95g là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_075.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước mắm Nam Ngư 750ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_vnpt_train_212.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dịch vụ Lắp đặt camera giám sát, Bàn phím cơ Dareu EK87"
  },
  {
    "image_name": "receipt_c45_bb_train_043.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 343 Cách Mạng Tháng Tám, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"Ngày 20 tháng 06 năm 2026\", \"total_cost\": \"1.620.000 VND\"}"
  },
  {
    "image_name": "cafe_highlands_train_111.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "689,040"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Thịt ba rọi heo 500g, Rau muống nước 500g"
  },
  {
    "image_name": "convenience_7eleven_train_096.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Kẹo dẻo Haribo Goldbears, Sandwich xá xíu phô mai, Trà đào sả Slurpee, Nước khoáng Dasani 500ml, Mì xào tương đen, Snack khoai tây Lays 95g"
  },
  {
    "image_name": "convenience_gs25_val_022.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì trộn Indomie đặc biệt trên hóa đơn là bao nhiêu?",
    "ground_truth": "48,000"
  },
  {
    "image_name": "restaurant_jollibee_train_093.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "111,100đ"
  },
  {
    "image_name": "restaurant_kfc_train_239.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "10/06/2026 08:55"
  },
  {
    "image_name": "convenience_gs25_train_223.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 THỦ ĐỨC\", \"address\": \"Số 59 Lý Thường Kiệt, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"15/06/2026 12:34\", \"total_cost\": \"85,800đ\", \"items\": [{\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"3\", \"amount\": \"36,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"3\", \"amount\": \"42,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_128.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"116,600đ\", \"box\": [329, 418, 385, 440]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_050.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH QUẬN 12\", \"address\": \"Số 147 Bùi Thị Xuân, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"31/05/2026 15:43\", \"total_cost\": \"716,100\", \"items\": [{\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"4\", \"amount\": \"128,000\"}, {\"name\": \"Thịt đùi heo 500g\", \"qty\": \"4\", \"amount\": \"272,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}, {\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"3\", \"amount\": \"225,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_165.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 21:41"
  },
  {
    "image_name": "minimart_anan_train_040.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"MINIMART ANAN\", \"box\": [156, 15, 254, 36]}"
  },
  {
    "image_name": "einvoice_viettel_train_180.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY CỔ PHẦN CÔNG NGHỆ SAO NAM"
  },
  {
    "image_name": "cafe_highlands_train_008.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_057.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH QUẬN 12"
  },
  {
    "image_name": "minimart_anan_train_244.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "59.400"
  },
  {
    "image_name": "einvoice_viettel_train_197.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Bảo trì Hệ thống mạng trên hóa đơn là bao nhiêu?",
    "ground_truth": "3,000,000"
  },
  {
    "image_name": "supermarket_lotte_val_012.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Khăn ướt Bobby 80 tờ, Bột giặt Ariel 3.2kg, Bia Heineken Lon 330ml"
  },
  {
    "image_name": "convenience_7eleven_val_049.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "image_name": "convenience_gs25_train_017.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Kirin Latte 345ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "48,000"
  },
  {
    "image_name": "convenience_gs25_train_155.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cơm nắm cá hồi sốt Mayo, Cafe sữa đá GS25, Mì trộn Indomie đặc biệt, Trà sữa Kirin Latte 345ml, Xúc xích Đức ăn liền"
  },
  {
    "image_name": "convenience_7eleven_train_165.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "image_name": "einvoice_vnpt_train_041.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "14,740,000đ"
  },
  {
    "image_name": "cafe_starbucks_val_049.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "17/06/2026"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_119.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Thịt ba rọi heo 500g, Đường cát trắng Biên Hòa 1kg, Thịt đùi heo 500g"
  },
  {
    "image_name": "minimart_anan_train_084.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kem đánh răng Colgate, Nước uống Aquafina 500ml, Bột giặt Ariel 3.2kg, Trà xanh Không Độ 500ml, Xúc xích tiệt trùng Ponnie, Mì Ly ăn liền Modern"
  },
  {
    "image_name": "cafe_highlands_train_140.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Tiramisu, Trà Sen Vàng Size M, Bánh Mì Que Gà Xé"
  },
  {
    "image_name": "convenience_7eleven_train_176.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "image_name": "receipt_c45_bb_train_099.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 03 tháng 06 năm 2026"
  },
  {
    "image_name": "convenience_7eleven_train_071.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "11/06/2026 15:18"
  },
  {
    "image_name": "einvoice_vnpt_train_240.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chuột không dây Logitech trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,160,000"
  },
  {
    "image_name": "einvoice_vnpt_train_017.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Cáp mạng Cat6 UTP 305m, Bàn phím cơ Dareu EK87"
  },
  {
    "image_name": "cafe_phuclong_train_036.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "minimart_anan_train_195.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "minimart_anan_train_252.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "supermarket_winmart_train_142.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "29.000"
  },
  {
    "image_name": "cafe_phuclong_train_107.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LANDMARK\", \"address\": \"Số 280 Lê Lợi, Quận 10, TP. Hồ Chí Minh\", \"timestamp\": \"18/06/2026 22:52\", \"total_cost\": \"306,900\", \"items\": [{\"name\": \"Hồng Trà Sữa M\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"4\", \"amount\": \"165,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"1\", \"amount\": \"100,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_099.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "286,000"
  },
  {
    "image_name": "restaurant_kfc_train_212.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_train_152.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "423,500đ"
  },
  {
    "image_name": "cafe_starbucks_train_216.png",
    "template": "cafe_starbucks",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 161 Nguyễn Huệ, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_7eleven_val_005.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "12/06/2026 12:32"
  },
  {
    "image_name": "cafe_starbucks_train_146.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_198.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Tiramisu trên hóa đơn là bao nhiêu?",
    "ground_truth": "35,000"
  },
  {
    "image_name": "convenience_circlek_val_006.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "96,800đ"
  },
  {
    "image_name": "supermarket_winmart_train_080.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_211.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Thạch Đào Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "minimart_anan_val_015.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_train_065.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Giấy Double A A4 70gsm, Mực in Canon Cartridge, Máy in HP LaserJet Pro, Dịch vụ Bảo trì Hệ thống mạng"
  },
  {
    "image_name": "einvoice_vnpt_train_067.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bộ phát Wifi TP-Link Archer trên hóa đơn là bao nhiêu?",
    "ground_truth": "850,000"
  },
  {
    "image_name": "supermarket_winmart_train_227.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_129.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "supermarket_winmart_val_014.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_251.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Freeze Trà Xanh Size M là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_049.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "27/05/2026 16:43"
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Thạch Đào Size L, Phin Sữa Đá Size L, Freeze Trà Xanh Size M"
  },
  {
    "image_name": "convenience_gs25_train_120.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "restaurant_kfc_val_011.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 21:35"
  },
  {
    "image_name": "convenience_circlek_train_204.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "convenience_gs25_train_162.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_017.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"address\": \"Số 185 Lê Lợi, Quận 1, Cần Thơ\", \"timestamp\": \"30/05/2026 21:21\", \"total_cost\": \"360,800\", \"items\": [{\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"4\", \"amount\": \"300,000\"}, {\"name\": \"Hành lá 100g\", \"qty\": \"4\", \"amount\": \"12,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"2\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_val_064.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "199,800"
  },
  {
    "image_name": "cafe_starbucks_train_068.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_107.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 13:51"
  },
  {
    "image_name": "cafe_starbucks_train_249.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "receipt_c45_bb_train_022.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [82, 36, 352, 54]}"
  },
  {
    "image_name": "restaurant_kfc_train_054.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "cafe_highlands_train_049.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE\", \"address\": \"Số 250 Cách Mạng Tháng Tám, Quận 1, Cần Thơ\", \"timestamp\": \"24/05/2026 12:36\", \"total_cost\": \"472,230\", \"items\": [{\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"3\", \"amount\": \"55,000\"}, {\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"1\", \"amount\": \"117,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"3\", \"amount\": \"55,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"3\", \"amount\": \"135,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"2\", \"amount\": \"57,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_098.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "70,000"
  },
  {
    "image_name": "restaurant_jollibee_train_112.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "supermarket_winmart_train_082.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "68.000"
  },
  {
    "image_name": "cafe_highlands_val_021.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "minimart_anan_train_038.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "96.000"
  },
  {
    "image_name": "supermarket_lotte_train_043.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "10/06/2026 22:06:00"
  },
  {
    "image_name": "einvoice_vnpt_train_081.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026 19:07"
  },
  {
    "image_name": "supermarket_winmart_train_253.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "38.000"
  },
  {
    "image_name": "einvoice_vnpt_train_105.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "900,000"
  },
  {
    "image_name": "convenience_gs25_train_065.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cafe sữa đá GS25 trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "restaurant_jollibee_train_053.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_225.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Coca Cola Lon 320ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30.000"
  },
  {
    "image_name": "convenience_7eleven_val_001.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "image_name": "cafe_highlands_train_014.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE LÊ LỢI\", \"address\": \"Số 135 Cách Mạng Tháng Tám, Quận 1, Cần Thơ\", \"timestamp\": \"06/06/2026 18:14\", \"total_cost\": \"249,480\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"2\", \"amount\": \"156,000\"}, {\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"2\", \"amount\": \"58,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"1\", \"amount\": \"38,000\"}]}"
  },
  {
    "image_name": "convenience_circlek_val_056.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "restaurant_kfc_train_110.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Trứng Egg Tart, Nước ngọt Pepsi L, Gà Giòn Cay 1 miếng, Burger Tôm"
  },
  {
    "image_name": "minimart_anan_train_261.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kem đánh răng Colgate, Xúc xích tiệt trùng Ponnie"
  },
  {
    "image_name": "supermarket_lotte_val_025.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước cam ép Twister 1L, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "einvoice_viettel_train_238.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 155 Lý Thường Kiệt, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"02/06/2026 17:27\", \"total_cost\": \"22,046,200đ\", \"items\": [{\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"2\", \"amount\": \"130,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"3\", \"amount\": \"4,500,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"4\", \"amount\": \"15,400,000\"}, {\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"3\", \"amount\": \"12,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_019.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_098.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Bà Hom\", \"box\": [162, 17, 247, 40]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_135.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt đùi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "136,000"
  },
  {
    "image_name": "convenience_gs25_val_006.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "cafe_phuclong_train_209.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Hồng Trà Sữa M, Bánh Croissant Bơ Pháp, Trà Lài Đác Thơm L"
  },
  {
    "image_name": "convenience_circlek_train_058.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_circlek_train_154.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Băng cá nhân Urgo hộp 10 miếng, Kẹo cao su Cool Air hũ, Mì Ly ăn liền Modern"
  },
  {
    "image_name": "restaurant_kfc_train_140.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"373,680\", \"box\": [318, 508, 378, 529]}"
  },
  {
    "image_name": "cafe_starbucks_train_033.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "80,000"
  },
  {
    "image_name": "convenience_7eleven_val_009.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "54,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_012.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hành lá 100g là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "supermarket_lotte_train_032.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "698,760.00"
  },
  {
    "image_name": "cafe_starbucks_train_127.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"370,975\", \"box\": [313, 418, 365, 439]}"
  },
  {
    "image_name": "supermarket_lotte_train_159.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hộp dâu tây Đà Lạt 250g là bao nhiêu?",
    "ground_truth": "130,000.00"
  },
  {
    "image_name": "cafe_phuclong_train_117.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 16:53"
  },
  {
    "image_name": "restaurant_kfc_train_205.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_train_160.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "04/06/2026 13:38"
  },
  {
    "image_name": "restaurant_kfc_train_077.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Trứng Egg Tart trên hóa đơn là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "restaurant_jollibee_val_020.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"366,120đ\", \"box\": [331, 462, 389, 482]}"
  },
  {
    "image_name": "convenience_gs25_train_122.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cafe sữa đá GS25 trên hóa đơn là bao nhiêu?",
    "ground_truth": "44,000"
  },
  {
    "image_name": "convenience_circlek_train_157.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ly ăn liền Modern trên hóa đơn là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "convenience_circlek_val_059.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_train_125.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cà phê Latte là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_val_031.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "162,250.00"
  },
  {
    "image_name": "cafe_starbucks_train_016.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "cafe_phuclong_train_110.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_gs25_val_061.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh giò thịt băm trứng cút, Xúc xích Đức ăn liền, Trà sữa Kirin Latte 345ml, Cơm nắm cá hồi sốt Mayo, Mì trộn Indomie đặc biệt, Cafe sữa đá GS25"
  },
  {
    "image_name": "restaurant_kfc_train_231.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 116 Lý Thường Kiệt, Quận 1, Bình Dương"
  },
  {
    "image_name": "convenience_gs25_train_128.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_113.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 296 Trần Hưng Đạo, Quận 1, Cần Thơ\", \"timestamp\": \"11/06/2026 07:07\", \"total_cost\": \"9,666,000đ\", \"items\": [{\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"2\", \"amount\": \"7,700,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"1\", \"amount\": \"1,250,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_val_061.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Khoai Môn trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "cafe_starbucks_train_010.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "receipt_c45_bb_train_260.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 387 Nguyễn Huệ, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "cafe_starbucks_train_040.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "cafe_phuclong_train_127.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 247 Cách Mạng Tháng Tám, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_lotte_train_197.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "181,440.00"
  },
  {
    "image_name": "einvoice_vnpt_train_124.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bàn phím cơ Dareu EK87, Bộ phát Wifi TP-Link Archer, Chuột không dây Logitech, Cáp mạng Cat6 UTP 305m, Dịch vụ Lắp đặt camera giám sát"
  },
  {
    "image_name": "einvoice_vnpt_train_061.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Cáp mạng Cat6 UTP 305m, Chuột không dây Logitech, Bộ phát Wifi TP-Link Archer, Bàn phím cơ Dareu EK87, Dịch vụ Lắp đặt camera giám sát"
  },
  {
    "image_name": "convenience_7eleven_train_195.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sandwich xá xíu phô mai là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "supermarket_lotte_val_006.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_train_214.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Butter Croissant, Chocolate Muffin, Green Tea Latte M, Caffe Americano L"
  },
  {
    "image_name": "supermarket_winmart_val_059.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"VM+QNH HẠ LONG\", \"address\": \"Số 309 Trần Hưng Đạo, Quận 1, Hải Phòng\", \"timestamp\": \"12/06/2026 20:02\", \"total_cost\": \"391.600\", \"items\": [{\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"2\", \"amount\": \"7.000\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"1\", \"amount\": \"29.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"4\", \"amount\": \"152.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"3\", \"amount\": \"168.000\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_034.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "26/05/2026 20:54"
  },
  {
    "image_name": "einvoice_viettel_val_054.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "minimart_anan_train_176.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"MINIMART ANAN\", \"address\": \"Số 334 Bùi Thị Xuân, Quận Ba Đình, Hà Nội\", \"timestamp\": \"01/06/2026 15:54\", \"total_cost\": \"554.040\", \"items\": [{\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"3\", \"amount\": \"27.000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"1\", \"amount\": \"8.000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"4\", \"amount\": \"64.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"2\", \"amount\": \"64.000\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"2\", \"amount\": \"350.000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_209.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "19/06/2026 18:30"
  },
  {
    "image_name": "einvoice_viettel_train_247.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "20/06/2026 12:51"
  },
  {
    "image_name": "cafe_starbucks_train_232.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Thạch Đào Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "einvoice_vnpt_train_115.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,092,000đ"
  },
  {
    "image_name": "convenience_circlek_train_058.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "cafe_highlands_train_010.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 327 Nguyễn Huệ, Quận 1, Cần Thơ\", \"timestamp\": \"18/06/2026 17:32\", \"total_cost\": \"416,880\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"3\", \"amount\": \"78,000\"}, {\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"1\", \"amount\": \"87,000\"}, {\"name\": \"Bánh Tiramisu\", \"qty\": \"2\", \"amount\": \"35,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"4\", \"amount\": \"110,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"1\", \"amount\": \"76,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_val_067.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"434.700\", \"box\": [335, 482, 386, 504]}"
  },
  {
    "image_name": "cafe_highlands_train_215.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"574,200\", \"box\": [308, 435, 360, 456]}"
  },
  {
    "image_name": "convenience_7eleven_train_109.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-Eleven Saigon Trade Center\", \"timestamp\": \"17/06/2026 19:58\", \"total_cost\": \"311,040\", \"items\": [{\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"57,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"100,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_173.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"address\": \"Số 354 Lê Lợi, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"29/05/2026 10:46\", \"total_cost\": \"3,256,000đ\", \"items\": [{\"name\": \"Chuột không dây Logitech\", \"qty\": \"4\", \"amount\": \"1,160,000\"}, {\"name\": \"Bàn phím cơ Dareu EK87\", \"qty\": \"4\", \"amount\": \"1,800,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_145.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"address\": \"Số 127 Nguyễn Trãi, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"29/05/2026 08:02\", \"total_cost\": \"568,080\", \"items\": [{\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"3\", \"amount\": \"225,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}, {\"name\": \"Nước mắm Nam Ngư 750ml\", \"qty\": \"4\", \"amount\": \"168,000\"}, {\"name\": \"Hành lá 100g\", \"qty\": \"1\", \"amount\": \"3,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_012.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "156,000"
  },
  {
    "image_name": "minimart_anan_train_141.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "239.800"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_035.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 378 Lê Lợi, Quận Bình Thạnh, TP. Hồ Chí Minh"
  },
  {
    "image_name": "restaurant_jollibee_val_003.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_044.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Bà Hom\", \"box\": [163, 18, 248, 40]}"
  },
  {
    "image_name": "supermarket_winmart_val_014.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"VM+HN TRẦN DUY HƯNG\", \"address\": \"Số 16 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"17/06/2026 16:16\", \"total_cost\": \"355.850\", \"items\": [{\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"4\", \"amount\": \"152.000\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"1\", \"amount\": \"29.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"2\", \"amount\": \"112.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"3\", \"amount\": \"10.500\"}, {\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"2\", \"amount\": \"20.000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_028.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 335 Trần Hưng Đạo, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"29/05/2026 15:57\", \"total_cost\": \"7,020,000đ\", \"items\": [{\"name\": \"Mực in Canon Cartridge\", \"qty\": \"4\", \"amount\": \"5,000,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"1\", \"amount\": \"1,500,000\"}]}"
  },
  {
    "image_name": "receipt_c45_bb_train_031.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 368 Lý Thường Kiệt, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"Ngày 08 tháng 06 năm 2026\", \"total_cost\": \"1.650.000 VND\"}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_115.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_087.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_jollibee_train_252.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "276,100đ"
  },
  {
    "image_name": "restaurant_kfc_train_231.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt Pepsi L trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "minimart_anan_train_181.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 273 Hai Bà Trưng, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "einvoice_viettel_val_065.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 39 Bùi Thị Xuân, Quận Tân Bình, TP. Hồ Chí Minh\", \"timestamp\": \"27/05/2026 22:16\", \"total_cost\": \"10,986,800đ\", \"items\": [{\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"2\", \"amount\": \"8,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"2\", \"amount\": \"130,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"4\", \"amount\": \"6,000,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_028.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE COOPMART\", \"address\": \"Số 196 Trần Hưng Đạo, Quận 1, Cần Thơ\", \"timestamp\": \"15/06/2026 21:02\", \"total_cost\": \"347,600đ\", \"items\": [{\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"4\", \"amount\": \"100,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"4\", \"amount\": \"140,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"2\", \"amount\": \"76,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_106.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH\", \"box\": [34, 64, 369, 85]}"
  },
  {
    "image_name": "minimart_anan_val_047.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "48.000"
  },
  {
    "image_name": "minimart_anan_train_191.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 16:18"
  },
  {
    "image_name": "cafe_highlands_train_197.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Phin Sữa Đá Size L, Freeze Trà Xanh Size M, Bánh Tiramisu, Bánh Mì Que Gà Xé"
  },
  {
    "image_name": "einvoice_viettel_train_256.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "08/06/2026 13:23"
  },
  {
    "image_name": "restaurant_jollibee_train_133.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_train_024.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "image_name": "cafe_highlands_train_041.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE LANDMARK\", \"address\": \"Số 172 Nguyễn Trãi, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"22/05/2026 22:26\", \"total_cost\": \"356,400\", \"items\": [{\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"3\", \"amount\": \"165,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"1\", \"amount\": \"165,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_val_003.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG NGUYỄN VĂN CỪ\", \"address\": \"Số 66 Hai Bà Trưng, Quận 1, Đồng Nai\", \"timestamp\": \"22/05/2026 13:06\", \"total_cost\": \"237,600\", \"items\": [{\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"3\", \"amount\": \"75,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"165,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_175.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "supermarket_lotte_train_095.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "44,000.00"
  },
  {
    "image_name": "restaurant_jollibee_train_100.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"228,800đ\", \"box\": [331, 426, 390, 447]}"
  },
  {
    "image_name": "supermarket_lotte_train_047.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn ướt Bobby 80 tờ là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_110.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Rau muống nước 500g, Đường cát trắng Biên Hòa 1kg, Thịt đùi heo 500g, Trứng gà tươi hộp 10 quả, Hành lá 100g"
  },
  {
    "image_name": "restaurant_kfc_val_064.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "einvoice_viettel_train_114.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "convenience_circlek_train_094.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo cao su Cool Air hũ trên hóa đơn là bao nhiêu?",
    "ground_truth": "22,000"
  },
  {
    "image_name": "supermarket_lotte_train_169.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 251 Nguyễn Trãi, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"12/06/2026 09:42:00\", \"total_cost\": \"164,160.00\", \"items\": [{\"name\": \"Kem đánh răng Colgate\", \"qty\": \"3\", \"amount\": \"96,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"2\", \"amount\": \"56,000.00\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_167.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE PASTEUR\", \"box\": [145, 15, 265, 36]}"
  },
  {
    "image_name": "convenience_7eleven_train_207.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "restaurant_jollibee_train_194.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "70,000"
  },
  {
    "image_name": "minimart_anan_val_060.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kem đánh răng Colgate trên hóa đơn là bao nhiêu?",
    "ground_truth": "64.000"
  },
  {
    "image_name": "minimart_anan_train_253.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "32.000"
  },
  {
    "image_name": "receipt_c45_bb_train_118.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 28 tháng 05 năm 2026"
  },
  {
    "image_name": "minimart_anan_train_041.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "302.500"
  },
  {
    "image_name": "cafe_starbucks_train_054.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "160,000"
  },
  {
    "image_name": "cafe_starbucks_train_020.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cold Brew Coffee M trên hóa đơn là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "cafe_highlands_train_147.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_train_075.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"104,500\", \"box\": [309, 349, 358, 372]}"
  },
  {
    "image_name": "einvoice_vnpt_train_050.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,880,400đ"
  },
  {
    "image_name": "minimart_anan_train_215.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"222.480\", \"box\": [341, 390, 391, 411]}"
  },
  {
    "image_name": "cafe_phuclong_val_061.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LÊ LAI\", \"address\": \"Số 360 Nguyễn Trãi, Quận 1, Đồng Nai\", \"timestamp\": \"25/05/2026 14:49\", \"total_cost\": \"491,400\", \"items\": [{\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"110,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"3\", \"amount\": \"165,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"1\", \"amount\": \"135,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_val_032.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH HẠ LONG\", \"address\": \"Số 240 Lý Thường Kiệt, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"07/06/2026 11:35\", \"total_cost\": \"586.850\", \"items\": [{\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"2\", \"amount\": \"20.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"4\", \"amount\": \"224.000\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"4\", \"amount\": \"116.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"4\", \"amount\": \"152.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"1\", \"amount\": \"3.500\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_028.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"54,000đ\", \"box\": [339, 371, 388, 391]}"
  },
  {
    "image_name": "restaurant_kfc_train_170.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "54,000"
  },
  {
    "image_name": "cafe_phuclong_train_089.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "restaurant_jollibee_val_024.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 156 Lê Lợi, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "convenience_circlek_train_259.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "receipt_c45_bb_train_060.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 63 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 12 tháng 06 năm 2026\", \"total_cost\": \"1.998.000 VND\"}"
  },
  {
    "image_name": "cafe_starbucks_train_003.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Green Tea Latte M, Chocolate Muffin, Caramel Macchiato L, Caffe Americano L, Cold Brew Coffee M, Butter Croissant"
  },
  {
    "image_name": "restaurant_kfc_train_080.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Trứng Egg Tart, Nước ngọt Pepsi L, Gà Giòn Cay 1 miếng, Khoai Tây Chiên L, Burger Tôm"
  },
  {
    "image_name": "cafe_starbucks_train_197.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS HÀN THUYÊN\", \"address\": \"Số 111 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"09/06/2026\", \"total_cost\": \"1,339,200\", \"items\": [{\"name\": \"Butter Croissant\", \"qty\": \"4\", \"amount\": \"160,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"2\", \"amount\": \"90,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"4\", \"amount\": \"340,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"4\", \"amount\": \"260,000\"}, {\"name\": \"Green Tea Latte M\", \"qty\": \"2\", \"amount\": \"150,000\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_195.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 17:25"
  },
  {
    "image_name": "einvoice_vnpt_train_242.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "3,400,000"
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 188 Cách Mạng Tháng Tám, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_highlands_train_047.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_187.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "supermarket_lotte_train_078.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 75 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "image_name": "convenience_gs25_train_035.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 TÔN THẤT THUYẾT\", \"box\": [135, 28, 281, 56]}"
  },
  {
    "image_name": "cafe_phuclong_train_197.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "821,340"
  },
  {
    "image_name": "receipt_c45_bb_train_112.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_221.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Rau muống nước 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "einvoice_viettel_train_109.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"address\": \"Số 227 Lê Lợi, Quận 1, Bình Dương\", \"timestamp\": \"10/06/2026 22:36\", \"total_cost\": \"10,321,560đ\", \"items\": [{\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"3\", \"amount\": \"12,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"2\", \"amount\": \"2,500,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"2\", \"amount\": \"3,000,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_val_009.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"298.650\", \"box\": [338, 506, 390, 527]}"
  },
  {
    "image_name": "restaurant_kfc_val_006.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 15:14"
  },
  {
    "image_name": "einvoice_vnpt_train_011.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "1,800,000"
  },
  {
    "image_name": "convenience_circlek_train_047.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 138 Trần Hưng Đạo, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "cafe_phuclong_train_223.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hồng Trà Sữa M trên hóa đơn là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "einvoice_vnpt_train_245.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "3,700,000"
  },
  {
    "image_name": "einvoice_viettel_train_187.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mực in Canon Cartridge, Bút bi Thiên Long FO-03, Giấy Double A A4 70gsm"
  },
  {
    "image_name": "convenience_circlek_train_070.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "image_name": "convenience_gs25_train_104.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 148 Hai Bà Trưng, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "image_name": "cafe_phuclong_train_106.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "363,000"
  },
  {
    "image_name": "supermarket_lotte_train_023.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "66,000.00"
  },
  {
    "image_name": "cafe_phuclong_val_014.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "688,050"
  },
  {
    "image_name": "restaurant_kfc_train_255.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Trứng Egg Tart trên hóa đơn là bao nhiêu?",
    "ground_truth": "54,000"
  },
  {
    "image_name": "restaurant_kfc_val_039.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt Pepsi L trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "einvoice_viettel_val_044.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Dịch vụ Bảo trì Hệ thống mạng, Mực in Canon Cartridge"
  },
  {
    "image_name": "restaurant_kfc_train_053.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 17 Bùi Thị Xuân, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_winmart_train_203.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "image_name": "cafe_phuclong_val_045.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 212 Trần Hưng Đạo, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "supermarket_lotte_train_207.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"449,280.00\", \"box\": [319, 532, 391, 552]}"
  },
  {
    "image_name": "cafe_highlands_val_034.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_007.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 162 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "restaurant_jollibee_train_221.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "68,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_123.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hành lá 100g trên hóa đơn là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "cafe_highlands_train_161.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "convenience_circlek_val_001.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "restaurant_kfc_val_034.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "supermarket_winmart_train_259.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+HN TRẦN DUY HƯNG\", \"address\": \"Số 286 Trần Hưng Đạo, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"31/05/2026 15:04\", \"total_cost\": \"361.900\", \"items\": [{\"name\": \"Bánh Custas Orion 6P\", \"qty\": \"1\", \"amount\": \"34.000\"}, {\"name\": \"Nước rửa chén Sunlight 750ml\", \"qty\": \"3\", \"amount\": \"87.000\"}, {\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"2\", \"amount\": \"20.000\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"2\", \"amount\": \"76.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"2\", \"amount\": \"112.000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_235.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 97 Cách Mạng Tháng Tám, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "image_name": "restaurant_kfc_val_040.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 10:24"
  },
  {
    "image_name": "supermarket_winmart_train_033.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "425.520"
  },
  {
    "image_name": "convenience_circlek_val_060.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CIRCLE K\", \"box\": [178, 32, 230, 54]}"
  },
  {
    "image_name": "restaurant_jollibee_train_027.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_vnpt_train_144.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "1,800,000"
  },
  {
    "image_name": "restaurant_jollibee_val_036.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "convenience_circlek_train_180.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "22,000"
  },
  {
    "image_name": "restaurant_jollibee_train_234.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "111,100đ"
  },
  {
    "image_name": "convenience_7eleven_train_092.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "80,000"
  },
  {
    "image_name": "minimart_anan_train_174.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_170.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trứng gà tươi hộp 10 quả trên hóa đơn là bao nhiêu?",
    "ground_truth": "128,000"
  },
  {
    "image_name": "minimart_anan_train_123.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"199.100\", \"box\": [343, 306, 395, 327]}"
  },
  {
    "image_name": "convenience_7eleven_val_067.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "102,300"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_232.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_winmart_train_167.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VM+HN TRẦN DUY HƯNG\", \"box\": [133, 26, 272, 49]}"
  },
  {
    "image_name": "restaurant_jollibee_train_116.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE VINCOM\", \"box\": [144, 41, 253, 62]}"
  },
  {
    "image_name": "einvoice_vnpt_train_219.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "290,000"
  },
  {
    "image_name": "cafe_highlands_train_133.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Tiramisu trên hóa đơn là bao nhiêu?",
    "ground_truth": "140,000"
  },
  {
    "image_name": "convenience_gs25_val_054.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"119,880đ\", \"box\": [324, 408, 378, 429]}"
  },
  {
    "image_name": "cafe_starbucks_val_009.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_107.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH\", \"address\": \"Số 415 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"06/06/2026 21:00\", \"total_cost\": \"224,640\", \"items\": [{\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}, {\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"2\", \"amount\": \"150,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"4\", \"amount\": \"32,000\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_095.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "minimart_anan_train_049.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "112.320"
  },
  {
    "image_name": "supermarket_lotte_train_160.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"address\": \"Số 53 Nguyễn Huệ, Quận 1, Bình Dương\", \"timestamp\": \"27/05/2026 13:27:00\", \"total_cost\": \"324,500.00\", \"items\": [{\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"3\", \"amount\": \"195,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"2\", \"amount\": \"56,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"2\", \"amount\": \"44,000.00\"}]}"
  },
  {
    "image_name": "minimart_anan_train_243.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CỬA HÀNG TIỆN LỢI AN AN\", \"address\": \"Số 331 Cách Mạng Tháng Tám, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"03/06/2026 22:47\", \"total_cost\": \"248.400\", \"items\": [{\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"2\", \"amount\": \"44.000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"4\", \"amount\": \"40.000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"3\", \"amount\": \"48.000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"3\", \"amount\": \"18.000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"2\", \"amount\": \"16.000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_196.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà sữa Phúc Long L, Trà Đào Phúc Long L"
  },
  {
    "image_name": "cafe_starbucks_train_079.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "415,800"
  },
  {
    "image_name": "convenience_7eleven_train_211.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "cafe_highlands_train_113.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 403 Cách Mạng Tháng Tám, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "restaurant_jollibee_train_081.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE PASTEUR\", \"address\": \"Số 4 Cách Mạng Tháng Tám, Quận Cầu Giấy, Hà Nội\", \"timestamp\": \"04/06/2026 18:23\", \"total_cost\": \"294,840đ\", \"items\": [{\"name\": \"Bánh Khoai Môn\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"2\", \"amount\": \"76,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"4\", \"amount\": \"140,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"1\", \"amount\": \"17,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_101.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Thịt ba rọi heo 500g, Thịt đùi heo 500g, Đường cát trắng Biên Hòa 1kg, Trứng gà tươi hộp 10 quả"
  },
  {
    "image_name": "restaurant_jollibee_val_058.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước ngọt 7Up L, Mì Ý Sốt Bò Bằm"
  },
  {
    "image_name": "restaurant_jollibee_train_115.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "287,280đ"
  },
  {
    "image_name": "supermarket_lotte_val_011.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước cam ép Twister 1L, Bia Heineken Lon 330ml, Khăn ướt Bobby 80 tờ, Hộp dâu tây Đà Lạt 250g, Kem đánh răng Colgate"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_092.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt đùi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "272,000"
  },
  {
    "image_name": "restaurant_kfc_train_170.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "supermarket_lotte_train_126.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "455,760.00"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_096.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "706,200"
  },
  {
    "image_name": "einvoice_viettel_train_084.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Giấy Double A A4 70gsm, Mực in Canon Cartridge, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "convenience_gs25_train_021.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "minimart_anan_val_031.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "429.840"
  },
  {
    "image_name": "einvoice_vnpt_train_160.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"box\": [59, 82, 372, 123]}"
  },
  {
    "image_name": "supermarket_lotte_train_140.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Kem đánh răng Colgate"
  },
  {
    "image_name": "supermarket_lotte_val_023.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 356 Lê Lợi, Quận 1, Hải Phòng"
  },
  {
    "image_name": "minimart_anan_train_158.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"983.880\", \"box\": [341, 408, 394, 430]}"
  },
  {
    "image_name": "cafe_starbucks_train_039.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "cafe_highlands_train_136.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Mì Que Gà Xé trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "minimart_anan_train_030.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "16.000"
  },
  {
    "image_name": "restaurant_kfc_train_164.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "minimart_anan_train_009.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "32.000"
  },
  {
    "image_name": "convenience_circlek_train_153.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "176,000đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_028.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt ba rọi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "restaurant_jollibee_train_126.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE VINCOM\", \"box\": [147, 32, 257, 55]}"
  },
  {
    "image_name": "restaurant_jollibee_train_157.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE VINCOM\", \"box\": [151, 21, 256, 45]}"
  },
  {
    "image_name": "restaurant_kfc_train_115.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_001.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "18/06/2026 11:22"
  },
  {
    "image_name": "einvoice_viettel_val_028.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "1,607,100đ"
  },
  {
    "image_name": "convenience_7eleven_train_132.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"185,900\", \"box\": [309, 208, 360, 232]}"
  },
  {
    "image_name": "cafe_highlands_train_251.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_train_033.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "340,200"
  },
  {
    "image_name": "cafe_starbucks_val_047.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"1,108,080\", \"box\": [292, 546, 359, 569]}"
  },
  {
    "image_name": "cafe_starbucks_train_079.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "einvoice_viettel_train_103.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mực in Canon Cartridge trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000,000"
  },
  {
    "image_name": "einvoice_viettel_val_051.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "image_name": "cafe_phuclong_train_135.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Croissant Bơ Pháp trên hóa đơn là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "supermarket_winmart_train_172.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 109 Nguyễn Huệ, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_viettel_train_263.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "21,449,880đ"
  },
  {
    "image_name": "einvoice_viettel_val_038.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dịch vụ Bảo trì Hệ thống mạng, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "cafe_highlands_train_068.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cà Phê Đen Đá Size M là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_073.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "232,100"
  },
  {
    "image_name": "restaurant_jollibee_train_119.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "114,000"
  },
  {
    "image_name": "receipt_c45_bb_train_226.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 20 tháng 06 năm 2026"
  },
  {
    "image_name": "convenience_7eleven_train_258.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-Eleven Nguyễn Du"
  },
  {
    "image_name": "cafe_phuclong_val_009.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "restaurant_kfc_train_108.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC VIỆT NAM\", \"box\": [152, 41, 254, 62]}"
  },
  {
    "image_name": "cafe_highlands_train_018.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "einvoice_vnpt_train_071.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "image_name": "supermarket_lotte_val_062.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"SAIGON CO.OP\", \"box\": [160, 15, 250, 36]}"
  },
  {
    "image_name": "restaurant_jollibee_train_178.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "einvoice_vnpt_train_170.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 304 Bùi Thị Xuân, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "cafe_starbucks_train_025.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,117,800"
  },
  {
    "image_name": "einvoice_vnpt_train_180.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "3,700,000"
  },
  {
    "image_name": "restaurant_jollibee_train_259.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Gà Giòn Vui Vẻ 1 miếng, Nước ngọt 7Up L, Bánh Khoai Môn, Khoai Tây Lắc Phô Mai"
  },
  {
    "image_name": "receipt_c45_bb_train_167.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "supermarket_lotte_train_137.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "128,000.00"
  },
  {
    "image_name": "restaurant_kfc_train_099.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Bà Hom\", \"box\": [160, 20, 244, 44]}"
  },
  {
    "image_name": "restaurant_jollibee_train_245.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE VINCOM\", \"address\": \"Số 339 Nguyễn Huệ, Quận 1, Hải Phòng\", \"timestamp\": \"11/06/2026 17:12\", \"total_cost\": \"260,280đ\", \"items\": [{\"name\": \"Nước ngọt 7Up L\", \"qty\": \"2\", \"amount\": \"34,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"4\", \"amount\": \"152,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_046.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Nước uống Aquafina 500ml, Trà xanh Không Độ 500ml, Bánh bao trứng muối"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_253.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH QUẬN 12"
  },
  {
    "image_name": "einvoice_viettel_train_063.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "image_name": "restaurant_jollibee_train_011.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "265,100đ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_230.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH NGUYỄN TRÃI\", \"box\": [104, 51, 278, 75]}"
  },
  {
    "image_name": "convenience_circlek_val_038.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "convenience_circlek_train_210.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 126 Cách Mạng Tháng Tám, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "supermarket_lotte_train_067.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"address\": \"Số 146 Hai Bà Trưng, Quận Tân Bình, TP. Hồ Chí Minh\", \"timestamp\": \"27/05/2026 10:41:00\", \"total_cost\": \"861,840.00\", \"items\": [{\"name\": \"Kem đánh răng Colgate\", \"qty\": \"1\", \"amount\": \"32,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"3\", \"amount\": \"66,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"4\", \"amount\": \"700,000.00\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_135.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kẹo cao su Cool Air hũ, Băng cá nhân Urgo hộp 10 miếng, Trà xanh Không Độ 500ml, Nước uống Aquafina 500ml, Bánh bao trứng muối, Mì Ly ăn liền Modern"
  },
  {
    "image_name": "restaurant_kfc_train_122.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC VIỆT NAM"
  },
  {
    "image_name": "cafe_phuclong_train_213.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Hồng Trà Sữa M, Trà sữa Phúc Long L, Cà phê Latte, Bánh Croissant Bơ Pháp, Trà Đào Phúc Long L"
  },
  {
    "image_name": "receipt_c45_bb_train_108.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "cafe_highlands_train_207.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Sen Vàng Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "restaurant_kfc_train_222.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_169.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "einvoice_viettel_train_219.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Máy in HP LaserJet Pro, Giấy Double A A4 70gsm, Dịch vụ Bảo trì Hệ thống mạng"
  },
  {
    "image_name": "cafe_highlands_train_214.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"HIGHLANDS COFFEE\", \"box\": [127, 23, 240, 50]}"
  },
  {
    "image_name": "einvoice_vnpt_train_260.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 100 Lý Thường Kiệt, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "supermarket_winmart_train_265.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "14/06/2026 18:25"
  },
  {
    "image_name": "convenience_gs25_val_048.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_kfc_val_042.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 14 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "image_name": "minimart_anan_train_208.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ANAN CONVENIENCE STORE\", \"address\": \"Số 21 Trần Hưng Đạo, Quận 1, Bình Dương\", \"timestamp\": \"27/05/2026 12:07\", \"total_cost\": \"798.120\", \"items\": [{\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525.000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"2\", \"amount\": \"18.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"3\", \"amount\": \"96.000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"4\", \"amount\": \"88.000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"2\", \"amount\": \"12.000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_015.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Rau muống nước 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "supermarket_lotte_train_118.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Hộp dâu tây Đà Lạt 250g, Khăn ướt Bobby 80 tờ, Bia Heineken Lon 330ml"
  },
  {
    "image_name": "cafe_phuclong_train_031.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LANDMARK\", \"address\": \"Số 330 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"15/06/2026 22:28\", \"total_cost\": \"198,000\", \"items\": [{\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"3\", \"amount\": \"55,000\"}, {\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"1\", \"amount\": \"75,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"50,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_val_015.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Chiên L trên hóa đơn là bao nhiêu?",
    "ground_truth": "112,000"
  },
  {
    "image_name": "convenience_gs25_train_160.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "convenience_7eleven_train_161.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì xào tương đen, Snack khoai tây Lays 95g, Kẹo dẻo Haribo Goldbears, Trà đào sả Slurpee"
  },
  {
    "image_name": "cafe_phuclong_train_225.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"PHÚC LONG NGUYỄN VĂN CỪ\", \"address\": \"Số 126 Nguyễn Trãi, Quận 1, Cần Thơ\", \"timestamp\": \"14/06/2026 19:40\", \"total_cost\": \"858,600\", \"items\": [{\"name\": \"Cà phê Latte\", \"qty\": \"2\", \"amount\": \"135,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"2\", \"amount\": \"100,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"4\", \"amount\": \"110,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"4\", \"amount\": \"180,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"2\", \"amount\": \"220,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_val_064.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"18,288,720đ\", \"box\": [660, 838, 740, 858]}"
  },
  {
    "image_name": "cafe_highlands_train_081.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE LÊ LỢI\", \"address\": \"Số 248 Nguyễn Huệ, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"17/06/2026 09:56\", \"total_cost\": \"401,940\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"4\", \"amount\": \"39,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"3\", \"amount\": \"220,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"2\", \"amount\": \"57,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"1\", \"amount\": \"90,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_002.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH QUẬN 12\", \"address\": \"Số 155 Lê Lợi, Quận 1, Cần Thơ\", \"timestamp\": \"18/06/2026 12:40\", \"total_cost\": \"637,200\", \"items\": [{\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"3\", \"amount\": \"225,000\"}, {\"name\": \"Thịt đùi heo 500g\", \"qty\": \"4\", \"amount\": \"272,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"2\", \"amount\": \"64,000\"}, {\"name\": \"Hành lá 100g\", \"qty\": \"1\", \"amount\": \"3,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_176.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "14/06/2026 07:01"
  },
  {
    "image_name": "minimart_anan_train_103.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_020.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_gs25_train_083.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "10,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_139.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "290,400"
  },
  {
    "image_name": "convenience_gs25_val_066.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 LANDMARK 81\", \"box\": [145, 15, 262, 36]}"
  },
  {
    "image_name": "minimart_anan_train_050.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ANAN CONVENIENCE STORE\", \"address\": \"Số 88 Lý Thường Kiệt, Quận Ba Đình, Hà Nội\", \"timestamp\": \"19/06/2026 12:02\", \"total_cost\": \"630.300\", \"items\": [{\"name\": \"Kem đánh răng Colgate\", \"qty\": \"3\", \"amount\": \"96.000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"2\", \"amount\": \"12.000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"4\", \"amount\": \"64.000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"3\", \"amount\": \"27.000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24.000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_078.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "restaurant_kfc_val_006.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_194.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "110,000"
  },
  {
    "image_name": "cafe_starbucks_train_161.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "cafe_starbucks_train_200.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "supermarket_winmart_train_213.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "image_name": "cafe_highlands_train_222.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_263.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_val_026.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kem đánh răng Colgate trên hóa đơn là bao nhiêu?",
    "ground_truth": "96,000.00"
  },
  {
    "image_name": "supermarket_winmart_train_223.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn giấy Pulppy 100 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "54.000"
  },
  {
    "image_name": "convenience_circlek_train_243.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 335 Lý Thường Kiệt, Quận 1, Bình Dương"
  },
  {
    "image_name": "cafe_starbucks_train_201.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "682,290"
  },
  {
    "image_name": "einvoice_viettel_train_106.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 15:01"
  },
  {
    "image_name": "restaurant_kfc_train_155.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_gs25_train_181.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 NGUYỄN HUỆ\", \"address\": \"Số 10 Lê Lợi, Quận Hai Bà Trưng, Hà Nội\", \"timestamp\": \"22/05/2026 15:52\", \"total_cost\": \"192,240đ\", \"items\": [{\"name\": \"Sữa chua uống Proby\", \"qty\": \"2\", \"amount\": \"16,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"2\", \"amount\": \"28,000\"}, {\"name\": \"Cơm nắm cá hồi sốt Mayo\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Cafe sữa đá GS25\", \"qty\": \"1\", \"amount\": \"22,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"3\", \"amount\": \"36,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_232.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT"
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Phin Sữa Đá Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "156,000"
  },
  {
    "image_name": "cafe_starbucks_val_055.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"STARBUCKS REX HOTEL\", \"box\": [112, 103, 267, 122]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_118.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước mắm Nam Ngư 750ml, Thịt ba rọi heo 500g"
  },
  {
    "image_name": "cafe_phuclong_train_091.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "357,500"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_204.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước mắm Nam Ngư 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "42,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_238.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hành lá 100g là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_circlek_train_222.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Hai Bà Trưng\", \"address\": \"Số 293 Cách Mạng Tháng Tám, Quận 1, Cần Thơ\", \"timestamp\": \"05/06/2026 22:53\", \"total_cost\": \"213,400đ\", \"items\": [{\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"4\", \"amount\": \"88,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24,000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"1\", \"amount\": \"6,000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"1\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_094.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_train_224.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY CỔ PHẦN CÔNG NGHỆ SAO NAM"
  },
  {
    "image_name": "cafe_starbucks_train_160.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_204.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Hộp dâu tây Đà Lạt 250g, Bia Heineken Lon 330ml, Kem đánh răng Colgate, Khăn ướt Bobby 80 tờ, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "convenience_gs25_train_253.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "convenience_7eleven_train_235.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì xào tương đen, Snack khoai tây Lays 95g, Sandwich xá xíu phô mai, Trà đào sả Slurpee"
  },
  {
    "image_name": "cafe_starbucks_train_167.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "supermarket_lotte_train_068.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_033.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_highlands_train_128.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "07/06/2026 22:47"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_118.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước mắm Nam Ngư 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "convenience_gs25_train_024.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"47,520đ\", \"box\": [336, 356, 388, 378]}"
  },
  {
    "image_name": "convenience_circlek_train_246.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_143.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "image_name": "cafe_starbucks_train_051.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "image_name": "restaurant_kfc_train_193.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_017.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 21:21"
  },
  {
    "image_name": "supermarket_winmart_train_095.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"253.800\", \"box\": [343, 456, 395, 477]}"
  },
  {
    "image_name": "convenience_gs25_train_180.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"196,560đ\", \"box\": [333, 444, 392, 464]}"
  },
  {
    "image_name": "einvoice_vnpt_train_117.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_circlek_train_095.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_179.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "receipt_c45_bb_train_029.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 262 Nguyễn Huệ, Quận 1, Hải Phòng\", \"timestamp\": \"Ngày 08 tháng 06 năm 2026\", \"total_cost\": \"2.090.000 VND\"}"
  },
  {
    "image_name": "convenience_circlek_train_139.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 14:46"
  },
  {
    "image_name": "supermarket_winmart_val_034.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 92 Nguyễn Huệ, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_starbucks_train_003.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Green Tea Latte M, Chocolate Muffin, Caramel Macchiato L, Caffe Americano L, Cold Brew Coffee M, Butter Croissant"
  },
  {
    "image_name": "cafe_starbucks_train_005.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_043.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH LÊ ĐỨC THỌ"
  },
  {
    "image_name": "restaurant_kfc_train_105.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_020.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Sen Vàng Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "cafe_phuclong_train_130.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "10/06/2026 18:36"
  },
  {
    "image_name": "einvoice_vnpt_val_036.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Lắp đặt camera giám sát trên hóa đơn là bao nhiêu?",
    "ground_truth": "2,500,000"
  },
  {
    "image_name": "cafe_starbucks_val_030.png",
    "template": "cafe_starbucks",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 397 Nguyễn Trãi, Quận 1, Bình Dương"
  },
  {
    "image_name": "receipt_c45_bb_val_063.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [85, 27, 364, 42]}"
  },
  {
    "image_name": "einvoice_viettel_train_019.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"8,910,000đ\", \"box\": [664, 698, 735, 717]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_266.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_130.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "110,000"
  },
  {
    "image_name": "minimart_anan_val_016.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "25/05/2026 07:18"
  },
  {
    "image_name": "minimart_anan_train_161.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_val_062.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 356 Nguyễn Trãi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "cafe_highlands_train_124.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "611,600"
  },
  {
    "image_name": "einvoice_viettel_train_003.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "1,500,000"
  },
  {
    "image_name": "supermarket_winmart_train_010.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 336 Bùi Thị Xuân, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_044.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo dẻo Haribo Goldbears trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "minimart_anan_train_166.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "59.400"
  },
  {
    "image_name": "cafe_highlands_val_034.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 73 Trần Hưng Đạo, Quận 1, Cần Thơ"
  },
  {
    "image_name": "einvoice_viettel_val_065.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "supermarket_winmart_train_176.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VM+QNH CẨM PHẢ\", \"box\": [164, 35, 265, 57]}"
  },
  {
    "image_name": "receipt_c45_bb_train_019.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 17 tháng 06 năm 2026"
  },
  {
    "image_name": "convenience_circlek_train_198.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Xúc xích tiệt trùng Ponnie, Băng cá nhân Urgo hộp 10 miếng, Bánh bao trứng muối"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_263.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước mắm Nam Ngư 750ml, Rau muống nước 500g, Đường cát trắng Biên Hòa 1kg, Thịt ba rọi heo 500g"
  },
  {
    "image_name": "receipt_c45_bb_train_116.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "einvoice_viettel_train_073.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_train_242.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bia Heineken Lon 330ml, Hộp dâu tây Đà Lạt 250g, Kem đánh răng Colgate, Khăn ướt Bobby 80 tờ, Nước cam ép Twister 1L"
  },
  {
    "image_name": "restaurant_kfc_train_002.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "112,000"
  },
  {
    "image_name": "einvoice_viettel_train_108.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "receipt_c45_bb_train_242.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 19 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_circlek_train_009.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "57,240đ"
  },
  {
    "image_name": "supermarket_winmart_train_119.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Dầu ăn Simply 1L, Khăn giấy Pulppy 100 tờ, Coca Cola Lon 320ml, Bánh Custas Orion 6P, Nước rửa chén Sunlight 750ml"
  },
  {
    "image_name": "restaurant_kfc_train_153.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt Pepsi L trên hóa đơn là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "restaurant_jollibee_train_084.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 404 Lý Thường Kiệt, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "convenience_circlek_train_043.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"138,600đ\", \"box\": [329, 399, 387, 421]}"
  },
  {
    "image_name": "convenience_gs25_train_082.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"GS25 NGUYỄN HUỆ\", \"address\": \"Số 122 Trần Hưng Đạo, Quận Tân Bình, TP. Hồ Chí Minh\", \"timestamp\": \"24/05/2026 21:51\", \"total_cost\": \"54,000đ\", \"items\": [{\"name\": \"Sữa chua uống Proby\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"3\", \"amount\": \"42,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_022.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "210,100đ"
  },
  {
    "image_name": "convenience_7eleven_train_234.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Snack khoai tây Lays 95g, Mì xào tương đen, Kẹo dẻo Haribo Goldbears, Sandwich xá xíu phô mai, Nước khoáng Dasani 500ml"
  },
  {
    "image_name": "supermarket_winmart_val_032.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_013.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "08/06/2026 22:23"
  },
  {
    "image_name": "receipt_c45_bb_train_036.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "216.000 VND"
  },
  {
    "image_name": "convenience_circlek_train_132.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "27/05/2026 09:18"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_008.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_249.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ly ăn liền Modern trên hóa đơn là bao nhiêu?",
    "ground_truth": "36.000"
  },
  {
    "image_name": "minimart_anan_train_116.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh bao trứng muối, Mì Ly ăn liền Modern, Xúc xích tiệt trùng Ponnie"
  },
  {
    "image_name": "cafe_phuclong_train_173.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "receipt_c45_bb_train_240.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"box\": [90, 28, 348, 47]}"
  },
  {
    "image_name": "cafe_highlands_train_125.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "convenience_gs25_train_179.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 NGUYỄN HUỆ\", \"box\": [153, 35, 263, 59]}"
  },
  {
    "image_name": "supermarket_winmart_train_094.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 274 Lý Thường Kiệt, Quận 1, Cần Thơ"
  },
  {
    "image_name": "receipt_c45_bb_val_029.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 156 Hai Bà Trưng, Quận 1, Cần Thơ\", \"timestamp\": \"Ngày 01 tháng 06 năm 2026\", \"total_cost\": \"2.090.000 VND\"}"
  },
  {
    "image_name": "convenience_gs25_train_220.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh giò thịt băm trứng cút, Xúc xích Đức ăn liền, Mì trộn Indomie đặc biệt, Cơm nắm cá hồi sốt Mayo, Sữa chua uống Proby"
  },
  {
    "image_name": "einvoice_vnpt_train_135.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_132.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "556,600"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_174.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "cafe_starbucks_train_170.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "restaurant_kfc_train_184.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Burger Tôm là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "restaurant_jollibee_train_012.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Khoai Môn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "supermarket_lotte_val_021.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 127 Hai Bà Trưng, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "minimart_anan_val_001.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "image_name": "restaurant_jollibee_train_036.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "convenience_gs25_val_013.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 THỦ ĐỨC"
  },
  {
    "image_name": "supermarket_lotte_val_045.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_192.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt ba rọi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "image_name": "cafe_phuclong_val_039.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "786,500"
  },
  {
    "image_name": "cafe_starbucks_train_036.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "supermarket_winmart_train_041.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Coca Cola Lon 320ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20.000"
  },
  {
    "image_name": "cafe_phuclong_train_005.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG TEA & COFFEE"
  },
  {
    "image_name": "cafe_phuclong_train_117.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_192.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH LÊ ĐỨC THỌ"
  },
  {
    "image_name": "convenience_7eleven_train_032.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Saigon Trade Center\", \"box\": [102, 22, 291, 52]}"
  },
  {
    "image_name": "cafe_phuclong_val_054.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hồng Trà Sữa M trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "restaurant_kfc_train_112.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_7eleven_val_024.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "image_name": "minimart_anan_train_050.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "630.300"
  },
  {
    "image_name": "receipt_c45_bb_val_061.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "1.620.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_181.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 428 Cách Mạng Tháng Tám, Quận 1, Bình Dương"
  },
  {
    "image_name": "cafe_starbucks_val_049.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 299 Bùi Thị Xuân, Quận 1, Hải Phòng\", \"timestamp\": \"17/06/2026\", \"total_cost\": \"135,850\", \"items\": [{\"name\": \"Chocolate Muffin\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_val_046.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"KFC Bà Hom\", \"address\": \"Số 53 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"18/06/2026 19:07\", \"total_cost\": \"144,720\", \"items\": [{\"name\": \"Burger Tôm\", \"qty\": \"2\", \"amount\": \"84,000\"}, {\"name\": \"Bắp Cải Trộn L\", \"qty\": \"1\", \"amount\": \"22,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"1\", \"amount\": \"28,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_080.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE\", \"address\": \"Số 335 Trần Hưng Đạo, Quận 1, Hải Phòng\", \"timestamp\": \"30/05/2026 14:27\", \"total_cost\": \"198,000\", \"items\": [{\"name\": \"Bánh Tiramisu\", \"qty\": \"2\", \"amount\": \"70,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"1\", \"amount\": \"110,000\"}]}"
  },
  {
    "image_name": "convenience_circlek_val_013.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Băng cá nhân Urgo hộp 10 miếng là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "restaurant_jollibee_train_065.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_067.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Thịt đùi heo 500g, Đường cát trắng Biên Hòa 1kg, Rau muống nước 500g, Nước mắm Nam Ngư 750ml, Thịt ba rọi heo 500g, Trứng gà tươi hộp 10 quả"
  },
  {
    "image_name": "receipt_c45_bb_train_024.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 425 Nguyễn Huệ, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 25 tháng 05 năm 2026\", \"total_cost\": \"2.052.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_071.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 228 Nguyễn Trãi, Quận 1, Cần Thơ\", \"timestamp\": \"Ngày 31 tháng 05 năm 2026\", \"total_cost\": \"165.000 VND\"}"
  },
  {
    "image_name": "restaurant_kfc_train_254.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "280,500"
  },
  {
    "image_name": "receipt_c45_bb_val_041.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 68 Cách Mạng Tháng Tám, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"Ngày 18 tháng 06 năm 2026\", \"total_cost\": \"162.000 VND\"}"
  },
  {
    "image_name": "restaurant_jollibee_train_054.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE PASTEUR\", \"address\": \"Số 96 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"12/06/2026 13:47\", \"total_cost\": \"205,200đ\", \"items\": [{\"name\": \"Nước ngọt 7Up L\", \"qty\": \"1\", \"amount\": \"17,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"2\", \"amount\": \"50,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"2\", \"amount\": \"70,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"1\", \"amount\": \"38,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_190.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "378,180"
  },
  {
    "image_name": "restaurant_jollibee_train_119.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước ngọt 7Up L, Khoai Tây Lắc Phô Mai, Gà Giòn Vui Vẻ 1 miếng, Mì Ý Sốt Bò Bằm"
  },
  {
    "image_name": "supermarket_winmart_train_242.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "152.000"
  },
  {
    "image_name": "restaurant_jollibee_train_200.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_val_015.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "supermarket_lotte_train_055.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Hộp dâu tây Đà Lạt 250g, Bia Heineken Lon 330ml, Kem đánh răng Colgate, Nước cam ép Twister 1L, Khăn ướt Bobby 80 tờ"
  },
  {
    "image_name": "restaurant_kfc_train_168.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "04/06/2026 08:53"
  },
  {
    "image_name": "convenience_circlek_train_216.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"123,200đ\", \"box\": [329, 422, 388, 443]}"
  },
  {
    "image_name": "restaurant_jollibee_train_126.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "convenience_gs25_train_092.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Cơm nắm cá hồi sốt Mayo, Xúc xích Đức ăn liền"
  },
  {
    "image_name": "supermarket_lotte_train_172.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 22:17:00"
  },
  {
    "image_name": "einvoice_viettel_val_057.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "cafe_highlands_train_095.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "supermarket_winmart_val_024.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Coca Cola Lon 320ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30.000"
  },
  {
    "image_name": "restaurant_kfc_train_004.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bắp Cải Trộn L, Burger Tôm"
  },
  {
    "image_name": "supermarket_lotte_train_005.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 202 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "image_name": "receipt_c45_bb_train_021.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [91, 31, 357, 51]}"
  },
  {
    "image_name": "restaurant_jollibee_train_036.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_gs25_val_057.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 281 Cách Mạng Tháng Tám, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_phuclong_val_059.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "55,000"
  },
  {
    "image_name": "cafe_highlands_train_075.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"HIGHLANDS COFFEE LÊ LỢI\", \"box\": [112, 28, 277, 54]}"
  },
  {
    "image_name": "cafe_starbucks_train_059.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "einvoice_vnpt_train_059.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_val_019.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Sen Vàng Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "convenience_gs25_train_034.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 LANDMARK 81\", \"box\": [142, 35, 258, 57]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_066.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt đùi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "204,000"
  },
  {
    "image_name": "einvoice_vnpt_train_234.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_lotte_train_020.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "96,000.00"
  },
  {
    "image_name": "convenience_7eleven_train_137.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000"
  },
  {
    "image_name": "cafe_starbucks_train_130.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Chocolate Muffin, Green Tea Latte M, Caffe Americano L, Caramel Macchiato L"
  },
  {
    "image_name": "supermarket_lotte_val_052.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bia Heineken Lon 330ml là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_circlek_train_094.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "22,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_254.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_vnpt_train_189.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Lắp đặt camera giám sát trên hóa đơn là bao nhiêu?",
    "ground_truth": "2,500,000"
  },
  {
    "image_name": "supermarket_winmart_val_023.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Sữa tươi TH True Milk 1L, Coca Cola Lon 320ml"
  },
  {
    "image_name": "einvoice_viettel_val_012.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_gs25_train_185.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "207,360đ"
  },
  {
    "image_name": "cafe_starbucks_train_155.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caffe Americano L là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "einvoice_vnpt_train_182.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "19/06/2026 08:11"
  },
  {
    "image_name": "einvoice_viettel_train_203.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_gs25_train_256.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "14/06/2026 15:35"
  },
  {
    "image_name": "convenience_gs25_train_203.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"GS25 THỦ ĐỨC\", \"address\": \"Số 426 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"27/05/2026 14:10\", \"total_cost\": \"114,480đ\", \"items\": [{\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"3\", \"amount\": \"42,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"4\", \"amount\": \"48,000\"}, {\"name\": \"Sữa chua uống Proby\", \"qty\": \"2\", \"amount\": \"16,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_258.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 378 Bùi Thị Xuân, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "cafe_highlands_train_022.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Phin Sữa Đá Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "156,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_094.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "104,000"
  },
  {
    "image_name": "einvoice_vnpt_val_064.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "09/06/2026 11:55"
  },
  {
    "image_name": "convenience_circlek_train_134.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "supermarket_winmart_train_170.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn giấy Pulppy 100 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "54.000"
  },
  {
    "image_name": "convenience_7eleven_train_161.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì xào tương đen trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "cafe_starbucks_train_055.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"573,480\", \"box\": [311, 554, 363, 575]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_247.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Thịt đùi heo 500g, Trứng gà tươi hộp 10 quả"
  },
  {
    "image_name": "restaurant_kfc_val_022.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Burger Tôm trên hóa đơn là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "restaurant_jollibee_val_001.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt 7Up L trên hóa đơn là bao nhiêu?",
    "ground_truth": "68,000"
  },
  {
    "image_name": "convenience_gs25_val_055.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_val_019.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà sữa Phúc Long L, Trà Đào Phúc Long L, Bánh Croissant Bơ Pháp"
  },
  {
    "image_name": "cafe_starbucks_train_210.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "12/06/2026"
  },
  {
    "image_name": "minimart_anan_train_164.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20.000"
  },
  {
    "image_name": "restaurant_kfc_train_155.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "22,000"
  },
  {
    "image_name": "cafe_phuclong_train_245.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LÊ LAI\", \"address\": \"Số 57 Lê Lợi, Quận 1, Hải Phòng\", \"timestamp\": \"29/05/2026 14:50\", \"total_cost\": \"656,640\", \"items\": [{\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"4\", \"amount\": \"110,000\"}, {\"name\": \"Hồng Trà Sữa M\", \"qty\": \"3\", \"amount\": \"180,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"3\", \"amount\": \"135,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"1\", \"amount\": \"165,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"50,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_120.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_128.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Bà Hom\", \"box\": [161, 23, 245, 44]}"
  },
  {
    "image_name": "supermarket_winmart_val_050.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dầu ăn Simply 1L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "minimart_anan_val_022.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Phin Sữa Đá Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "cafe_starbucks_train_008.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "minimart_anan_train_062.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "40.000"
  },
  {
    "image_name": "cafe_highlands_train_236.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "467,532"
  },
  {
    "image_name": "einvoice_vnpt_val_006.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"20,012,400đ\", \"box\": [644, 816, 725, 840]}"
  },
  {
    "image_name": "einvoice_vnpt_train_236.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "1,800,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_004.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH QUẬN 12\", \"address\": \"Số 374 Nguyễn Trãi, Quận 10, TP. Hồ Chí Minh\", \"timestamp\": \"23/05/2026 12:04\", \"total_cost\": \"116,600\", \"items\": [{\"name\": \"Hành lá 100g\", \"qty\": \"4\", \"amount\": \"12,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"2\", \"amount\": \"16,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"3\", \"amount\": \"78,000\"}]}"
  },
  {
    "image_name": "receipt_c45_bb_train_107.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 166 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 28 tháng 05 năm 2026\", \"total_cost\": \"2.090.000 VND\"}"
  },
  {
    "image_name": "cafe_starbucks_train_175.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "image_name": "einvoice_vnpt_train_263.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 310 Lê Lợi, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "restaurant_kfc_val_031.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "09/06/2026 11:38"
  },
  {
    "image_name": "einvoice_vnpt_train_168.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_gs25_val_033.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sữa chua uống Proby trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "convenience_7eleven_train_132.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Nguyễn Du\", \"box\": [132, 28, 252, 48]}"
  },
  {
    "image_name": "cafe_phuclong_train_009.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"PHÚC LONG NGUYỄN VĂN CỪ\", \"address\": \"Số 84 Lý Thường Kiệt, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"23/05/2026 19:14\", \"total_cost\": \"215,460\", \"items\": [{\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"3\", \"amount\": \"75,000\"}, {\"name\": \"Cà phê Latte\", \"qty\": \"1\", \"amount\": \"135,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_211.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước cam ép Twister 1L là bao nhiêu?",
    "ground_truth": "28,000.00"
  },
  {
    "image_name": "restaurant_jollibee_train_152.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Khoai Môn trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "convenience_7eleven_train_171.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"156,600\", \"box\": [307, 202, 357, 226]}"
  },
  {
    "image_name": "supermarket_lotte_train_012.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "192,500.00"
  },
  {
    "image_name": "receipt_c45_bb_train_219.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 365 Lê Lợi, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "cafe_starbucks_train_206.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Butter Croissant, Green Tea Latte M, Cold Brew Coffee M, Caramel Macchiato L, Caffe Americano L, Chocolate Muffin"
  },
  {
    "image_name": "einvoice_vnpt_val_018.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 07:13"
  },
  {
    "image_name": "cafe_starbucks_train_059.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "restaurant_kfc_train_188.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Burger Tôm là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "receipt_c45_bb_val_060.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"box\": [92, 28, 349, 43]}"
  },
  {
    "image_name": "receipt_c45_bb_train_092.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"box\": [89, 31, 339, 49]}"
  },
  {
    "image_name": "cafe_highlands_train_219.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cà Phê Đen Đá Size M là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_train_084.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE COOPMART\", \"box\": [138, 19, 263, 43]}"
  },
  {
    "image_name": "minimart_anan_train_057.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Mì Ly ăn liền Modern, Bánh bao trứng muối, Xúc xích tiệt trùng Ponnie, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "supermarket_winmart_train_106.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "542.300"
  },
  {
    "image_name": "supermarket_lotte_train_242.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "412,500.00"
  },
  {
    "image_name": "convenience_circlek_val_006.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "einvoice_vnpt_train_245.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "3,700,000"
  },
  {
    "image_name": "cafe_phuclong_val_030.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "convenience_circlek_val_058.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo cao su Cool Air hũ trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "image_name": "supermarket_lotte_train_092.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"161,700.00\", \"box\": [318, 427, 393, 448]}"
  },
  {
    "image_name": "restaurant_kfc_train_026.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC Bà Hom\", \"address\": \"Số 419 Nguyễn Huệ, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"10/06/2026 11:57\", \"total_cost\": \"372,900\", \"items\": [{\"name\": \"Bắp Cải Trộn L\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Nước ngọt Pepsi L\", \"qty\": \"1\", \"amount\": \"19,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"2\", \"amount\": \"56,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"2\", \"amount\": \"78,000\"}, {\"name\": \"Burger Tôm\", \"qty\": \"2\", \"amount\": \"84,000\"}]}"
  },
  {
    "image_name": "minimart_anan_val_040.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "64.000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_030.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"box\": [102, 17, 281, 38]}"
  },
  {
    "image_name": "cafe_starbucks_train_225.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caffe Americano L trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "einvoice_viettel_train_246.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "7,143,120đ"
  },
  {
    "image_name": "minimart_anan_train_041.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"MINIMART ANAN\", \"box\": [156, 15, 254, 36]}"
  },
  {
    "image_name": "einvoice_vnpt_train_098.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_131.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"773,300\", \"box\": [308, 453, 359, 475]}"
  },
  {
    "image_name": "restaurant_kfc_train_208.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt Pepsi L trên hóa đơn là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "convenience_7eleven_train_176.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "80,000"
  },
  {
    "image_name": "supermarket_winmart_train_109.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VM+QNH HẠ LONG"
  },
  {
    "image_name": "convenience_7eleven_train_011.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-ELEVEN BÙI VIỆN\", \"timestamp\": \"09/06/2026 14:32\", \"total_cost\": \"286,200\", \"items\": [{\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"50,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"72,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"10,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"80,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_239.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_221.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "einvoice_vnpt_train_045.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"box\": [37, 59, 306, 89]}"
  },
  {
    "image_name": "convenience_circlek_val_013.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "24/05/2026 20:01"
  },
  {
    "image_name": "restaurant_kfc_train_094.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 315 Trần Hưng Đạo, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_082.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-Eleven Nguyễn Du"
  },
  {
    "image_name": "cafe_highlands_val_022.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Freeze Trà Xanh Size M là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_val_006.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Kem đánh răng Colgate, Hộp dâu tây Đà Lạt 250g"
  },
  {
    "image_name": "cafe_phuclong_train_052.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_gs25_train_202.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Kirin Latte 345ml là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "convenience_7eleven_train_033.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "convenience_circlek_val_061.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "restaurant_jollibee_train_075.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE PASTEUR\", \"address\": \"Số 383 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"29/05/2026 10:30\", \"total_cost\": \"201,960đ\", \"items\": [{\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"1\", \"amount\": \"35,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"4\", \"amount\": \"152,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_176.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "653,400"
  },
  {
    "image_name": "restaurant_kfc_val_019.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_049.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "cafe_highlands_train_050.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà Phê Đen Đá Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "87,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_193.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 327 Bùi Thị Xuân, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "minimart_anan_train_046.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "MINIMART ANAN"
  },
  {
    "image_name": "cafe_starbucks_train_041.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caramel Macchiato L trên hóa đơn là bao nhiêu?",
    "ground_truth": "85,000"
  },
  {
    "image_name": "minimart_anan_train_126.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_starbucks_train_236.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"STARBUCKS REX HOTEL\", \"box\": [109, 82, 271, 103]}"
  },
  {
    "image_name": "cafe_starbucks_train_136.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026"
  },
  {
    "image_name": "cafe_highlands_val_002.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "434,160"
  },
  {
    "image_name": "cafe_starbucks_train_082.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_232.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Vui Vẻ 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "einvoice_viettel_train_141.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_103.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_032.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"628,560\", \"box\": [308, 527, 358, 550]}"
  },
  {
    "image_name": "cafe_starbucks_train_232.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caffe Americano L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_phuclong_train_240.png",
    "template": "cafe_phuclong",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"PHÚC LONG LANDMARK\", \"address\": \"Số 14 Trần Hưng Đạo, Quận Đống Đa, Hà Nội\", \"timestamp\": \"23/05/2026 22:18\", \"total_cost\": \"600,875\", \"items\": [{\"name\": \"Bánh Croissant Bơ Pháp\", \"qty\": \"4\", \"amount\": \"100,000\"}, {\"name\": \"Trà sữa Phúc Long L\", \"qty\": \"1\", \"amount\": \"200,000\"}, {\"name\": \"Trà Đào Phúc Long L\", \"qty\": \"4\", \"amount\": \"55,000\"}, {\"name\": \"Trà Lài Đác Thơm L\", \"qty\": \"1\", \"amount\": \"220,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_038.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Giấy Double A A4 70gsm, Bút bi Thiên Long FO-03, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "supermarket_winmart_train_242.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dầu ăn Simply 1L, Khăn giấy Pulppy 100 tờ, Sữa tươi TH True Milk 1L, Bánh Custas Orion 6P, Mì tôm Hảo Hảo chua cay, Coca Cola Lon 320ml"
  },
  {
    "image_name": "convenience_gs25_train_223.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "15/06/2026 12:34"
  },
  {
    "image_name": "convenience_7eleven_train_252.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì xào tương đen trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "convenience_circlek_train_241.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "minimart_anan_train_005.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "32.000"
  },
  {
    "image_name": "einvoice_viettel_train_069.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "einvoice_viettel_train_198.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 209 Nguyễn Huệ, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "minimart_anan_train_253.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "525.000"
  },
  {
    "image_name": "restaurant_kfc_train_057.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_042.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH QUẬN 12\", \"box\": [108, 24, 263, 53]}"
  },
  {
    "image_name": "convenience_circlek_train_179.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CIRCLE K\", \"address\": \"Số 166 Lý Thường Kiệt, Quận 1, Hải Phòng\", \"timestamp\": \"29/05/2026 12:02\", \"total_cost\": \"129,600đ\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"2\", \"amount\": \"12,000\"}, {\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"4\", \"amount\": \"40,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"1\", \"amount\": \"8,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_142.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_train_204.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caffe Americano L trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "convenience_circlek_train_033.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CIRCLE K\", \"address\": \"Số 3 Lê Lợi, Quận 1, Cần Thơ\", \"timestamp\": \"11/06/2026 13:37\", \"total_cost\": \"127,440đ\", \"items\": [{\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"4\", \"amount\": \"88,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"3\", \"amount\": \"30,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_val_027.png",
    "template": "cafe_highlands",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "image_name": "einvoice_viettel_train_112.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dịch vụ Bảo trì Hệ thống mạng, Giấy Double A A4 70gsm, Bút bi Thiên Long FO-03, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "cafe_highlands_val_002.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 19:24"
  },
  {
    "image_name": "cafe_starbucks_train_172.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Butter Croissant là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "receipt_c45_bb_train_063.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 160 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 25 tháng 05 năm 2026\", \"total_cost\": \"378.000 VND\"}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_002.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Thịt đùi heo 500g là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "receipt_c45_bb_train_098.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 01 tháng 06 năm 2026"
  },
  {
    "image_name": "cafe_starbucks_train_163.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cold Brew Coffee M là bao nhiêu?",
    "ground_truth": "130,000"
  },
  {
    "image_name": "supermarket_lotte_train_087.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bia Heineken Lon 330ml là bao nhiêu?",
    "ground_truth": "78,000.00"
  },
  {
    "image_name": "supermarket_winmart_val_037.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "40.000"
  },
  {
    "image_name": "convenience_circlek_train_101.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "cafe_starbucks_train_229.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "supermarket_winmart_train_023.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "19/06/2026 17:12"
  },
  {
    "image_name": "cafe_phuclong_train_215.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LÊ LAI"
  },
  {
    "image_name": "cafe_phuclong_train_198.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Lài Đác Thơm L, Trà Đào Phúc Long L, Trà sữa Phúc Long L, Bánh Croissant Bơ Pháp, Hồng Trà Sữa M"
  },
  {
    "image_name": "cafe_starbucks_train_024.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caffe Americano L là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "convenience_gs25_train_135.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sữa chua uống Proby trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "convenience_circlek_train_264.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_winmart_train_180.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Sữa tươi TH True Milk 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "152.000"
  },
  {
    "image_name": "supermarket_winmart_val_058.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "26/05/2026 08:53"
  },
  {
    "image_name": "convenience_7eleven_train_202.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "18/06/2026 22:33"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_005.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "cafe_highlands_train_025.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 42 Lý Thường Kiệt, Quận 1, Cần Thơ\", \"timestamp\": \"15/06/2026 21:05\", \"total_cost\": \"384,560\", \"items\": [{\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"4\", \"amount\": \"110,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"2\", \"amount\": \"180,000\"}, {\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"1\", \"amount\": \"78,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_133.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Burger Tôm là bao nhiêu?",
    "ground_truth": "42,000"
  },
  {
    "image_name": "supermarket_lotte_train_092.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OPMART LÝ THƯỜNG KIỆT\", \"address\": \"Số 446 Lê Lợi, Quận 1, Hải Phòng\", \"timestamp\": \"04/06/2026 15:52:00\", \"total_cost\": \"161,700.00\", \"items\": [{\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"3\", \"amount\": \"81,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"3\", \"amount\": \"66,000.00\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_068.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"box\": [49, 72, 343, 118]}"
  },
  {
    "image_name": "cafe_highlands_train_160.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà Phê Đen Đá Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "58,000"
  },
  {
    "image_name": "receipt_c45_bb_train_031.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 08 tháng 06 năm 2026"
  },
  {
    "image_name": "cafe_starbucks_train_134.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_126.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 345 Bùi Thị Xuân, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_7eleven_train_036.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-ELEVEN BÙI VIỆN\", \"timestamp\": \"08/06/2026 14:02\", \"total_cost\": \"315,700\", \"items\": [{\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"15,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"76,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"100,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_045.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Nước cam ép Twister 1L, Kem đánh răng Colgate, Hộp dâu tây Đà Lạt 250g, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "cafe_phuclong_val_049.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Đào Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "convenience_circlek_train_197.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "58,320đ"
  },
  {
    "image_name": "convenience_7eleven_train_109.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "cafe_starbucks_train_082.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caramel Macchiato L trên hóa đơn là bao nhiêu?",
    "ground_truth": "340,000"
  },
  {
    "image_name": "einvoice_viettel_train_245.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2,910,600đ"
  },
  {
    "image_name": "supermarket_winmart_train_045.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "04/06/2026 20:22"
  },
  {
    "image_name": "restaurant_kfc_val_062.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Gà Giòn Cay 1 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "39,000"
  },
  {
    "image_name": "cafe_phuclong_train_095.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "minimart_anan_train_196.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "9.000"
  },
  {
    "image_name": "restaurant_kfc_train_041.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước ngọt Pepsi L, Bắp Cải Trộn L, Khoai Tây Chiên L, Bánh Trứng Egg Tart"
  },
  {
    "image_name": "supermarket_lotte_train_158.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 319 Hai Bà Trưng, Quận 1, Cần Thơ\", \"timestamp\": \"02/06/2026 09:35:00\", \"total_cost\": \"723,600.00\", \"items\": [{\"name\": \"Hộp dâu tây Đà Lạt 250g\", \"qty\": \"3\", \"amount\": \"195,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"2\", \"amount\": \"39,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"1\", \"amount\": \"32,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"2\", \"amount\": \"350,000.00\"}, {\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"2\", \"amount\": \"54,000.00\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_076.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "einvoice_vnpt_train_240.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_071.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 44 Cách Mạng Tháng Tám, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "convenience_gs25_train_255.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "129,600đ"
  },
  {
    "image_name": "minimart_anan_train_086.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "02/06/2026 16:20"
  },
  {
    "image_name": "restaurant_kfc_val_006.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC VIỆT NAM"
  },
  {
    "image_name": "einvoice_viettel_train_190.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "1,500,000"
  },
  {
    "image_name": "supermarket_winmart_val_043.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì tôm Hảo Hảo chua cay trên hóa đơn là bao nhiêu?",
    "ground_truth": "7.000"
  },
  {
    "image_name": "einvoice_vnpt_train_257.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,735,200đ"
  },
  {
    "image_name": "cafe_starbucks_train_149.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Caffe Americano L, Butter Croissant"
  },
  {
    "image_name": "cafe_phuclong_val_055.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_051.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "96,000"
  },
  {
    "image_name": "cafe_starbucks_train_163.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "convenience_7eleven_train_021.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "image_name": "cafe_highlands_val_046.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE\", \"address\": \"Số 340 Lý Thường Kiệt, Quận 1, Đồng Nai\", \"timestamp\": \"30/05/2026 12:27\", \"total_cost\": \"324,216\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"1\", \"amount\": \"156,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"3\", \"amount\": \"55,000\"}, {\"name\": \"Bánh Tiramisu\", \"qty\": \"1\", \"amount\": \"105,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_158.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_train_082.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "55,000"
  },
  {
    "image_name": "supermarket_winmart_train_243.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VINCOMMERCE\", \"box\": [160, 38, 237, 57]}"
  },
  {
    "image_name": "einvoice_viettel_train_167.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "image_name": "cafe_phuclong_train_036.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_055.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "8.000"
  },
  {
    "image_name": "restaurant_jollibee_train_235.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 116 Lê Lợi, Quận 1, Bình Dương"
  },
  {
    "image_name": "restaurant_jollibee_val_026.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"195,480đ\", \"box\": [324, 405, 383, 427]}"
  },
  {
    "image_name": "supermarket_winmart_train_177.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"162.000\", \"box\": [340, 400, 390, 423]}"
  },
  {
    "image_name": "convenience_gs25_train_091.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "22/05/2026 08:02"
  },
  {
    "image_name": "convenience_gs25_train_113.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 LANDMARK 81\", \"address\": \"Số 164 Bùi Thị Xuân, Quận 1, Hải Phòng\", \"timestamp\": \"20/06/2026 16:06\", \"total_cost\": \"41,040đ\", \"items\": [{\"name\": \"Sữa chua uống Proby\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Cơm nắm cá hồi sốt Mayo\", \"qty\": \"2\", \"amount\": \"30,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_234.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Rau muống nước 500g, Thịt đùi heo 500g, Nước mắm Nam Ngư 750ml, Trứng gà tươi hộp 10 quả, Thịt ba rọi heo 500g, Đường cát trắng Biên Hòa 1kg"
  },
  {
    "image_name": "einvoice_vnpt_train_140.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_123.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "373,680"
  },
  {
    "image_name": "restaurant_kfc_train_207.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_train_019.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN CÔNG NGHỆ SAO NAM\", \"address\": \"Số 140 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"17/06/2026 14:42\", \"total_cost\": \"8,910,000đ\", \"items\": [{\"name\": \"Mực in Canon Cartridge\", \"qty\": \"3\", \"amount\": \"3,750,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"3\", \"amount\": \"4,500,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_val_017.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS REX HOTEL\", \"address\": \"Số 342 Nguyễn Huệ, Quận Đống Đa, Hà Nội\", \"timestamp\": \"07/06/2026\", \"total_cost\": \"418,000\", \"items\": [{\"name\": \"Green Tea Latte M\", \"qty\": \"1\", \"amount\": \"75,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"1\", \"amount\": \"65,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"4\", \"amount\": \"180,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_105.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Burger Tôm trên hóa đơn là bao nhiêu?",
    "ground_truth": "126,000"
  },
  {
    "image_name": "convenience_7eleven_val_048.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 12:17"
  },
  {
    "image_name": "cafe_starbucks_train_106.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "minimart_anan_train_097.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "image_name": "convenience_7eleven_train_016.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "54,000"
  },
  {
    "image_name": "cafe_starbucks_train_009.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "13/06/2026"
  },
  {
    "image_name": "cafe_phuclong_train_010.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Đào Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "supermarket_winmart_train_074.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_173.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_258.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "255,200"
  },
  {
    "image_name": "receipt_c45_bb_train_122.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"1.620.000 VND\", \"box\": [128, 257, 243, 273]}"
  },
  {
    "image_name": "einvoice_vnpt_train_039.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "290,000"
  },
  {
    "image_name": "convenience_circlek_train_118.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "restaurant_kfc_train_164.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"KFC Nguyễn Thái Học\", \"box\": [128, 25, 285, 51]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_130.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hành lá 100g là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "cafe_starbucks_val_063.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_132.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Khoai Môn là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_075.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_133.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Rau muống nước 500g, Hành lá 100g, Thịt ba rọi heo 500g, Nước mắm Nam Ngư 750ml"
  },
  {
    "image_name": "restaurant_kfc_train_203.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_lotte_train_257.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "cafe_phuclong_train_014.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Lài Đác Thơm L trên hóa đơn là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "supermarket_winmart_train_047.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "58.000"
  },
  {
    "image_name": "einvoice_viettel_train_189.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "4,000"
  },
  {
    "image_name": "einvoice_viettel_train_181.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "130,000"
  },
  {
    "image_name": "restaurant_kfc_train_156.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Gà Giòn Cay 1 miếng, Khoai Tây Chiên L, Bánh Trứng Egg Tart"
  },
  {
    "image_name": "convenience_circlek_train_253.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 406 Lê Lợi, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_196.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "11/06/2026 15:47"
  },
  {
    "image_name": "einvoice_viettel_train_239.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "cafe_starbucks_train_260.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "cafe_phuclong_val_058.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 409 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_gs25_train_169.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_001.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước khoáng Dasani 500ml là bao nhiêu?",
    "ground_truth": "5,000"
  },
  {
    "image_name": "einvoice_vnpt_val_024.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_190.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "convenience_circlek_train_138.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"50,600đ\", \"box\": [340, 395, 391, 417]}"
  },
  {
    "image_name": "supermarket_winmart_train_131.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 216 Lê Lợi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_starbucks_val_062.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "convenience_circlek_train_069.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "einvoice_vnpt_train_148.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 12:48"
  },
  {
    "image_name": "einvoice_vnpt_train_029.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "15/06/2026 11:09"
  },
  {
    "image_name": "cafe_highlands_train_251.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "cafe_starbucks_train_068.png",
    "template": "cafe_starbucks",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 86 Lê Lợi, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "cafe_starbucks_train_263.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caffe Americano L là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_train_079.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Giấy Double A A4 70gsm trên hóa đơn là bao nhiêu?",
    "ground_truth": "65,000"
  },
  {
    "image_name": "supermarket_lotte_train_064.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "01/06/2026 18:01:00"
  },
  {
    "image_name": "convenience_7eleven_train_102.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-Eleven Saigon Trade Center\", \"timestamp\": \"07/06/2026 08:26\", \"total_cost\": \"199,100\", \"items\": [{\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"20,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"80,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_val_038.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "einvoice_vnpt_train_154.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 375 Nguyễn Trãi, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "restaurant_jollibee_val_038.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Khoai Môn là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_vnpt_train_140.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"11,620,800đ\", \"box\": [648, 794, 725, 815]}"
  },
  {
    "image_name": "minimart_anan_train_247.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_train_260.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 30 Trần Hưng Đạo, Quận 10, TP. Hồ Chí Minh"
  },
  {
    "image_name": "cafe_starbucks_train_240.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "restaurant_kfc_train_190.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 242 Cách Mạng Tháng Tám, Quận 1, Đồng Nai"
  },
  {
    "image_name": "einvoice_vnpt_val_065.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "3,245,000đ"
  },
  {
    "image_name": "minimart_anan_val_006.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20.000"
  },
  {
    "image_name": "minimart_anan_train_063.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "16.000"
  },
  {
    "image_name": "restaurant_jollibee_train_173.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "einvoice_vnpt_val_015.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "15/06/2026 12:50"
  },
  {
    "image_name": "cafe_starbucks_train_172.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_gs25_train_226.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "GS25 NGUYỄN HUỆ"
  },
  {
    "image_name": "cafe_phuclong_val_017.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"PHÚC LONG NGUYỄN VĂN CỪ\", \"box\": [102, 19, 267, 48]}"
  },
  {
    "image_name": "convenience_gs25_train_204.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà sữa Kirin Latte 345ml, Cơm nắm cá hồi sốt Mayo, Xúc xích Đức ăn liền"
  },
  {
    "image_name": "cafe_phuclong_train_101.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"PHÚC LONG TEA & COFFEE\", \"box\": [113, 19, 270, 48]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_265.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "221,100"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_022.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 252 Cách Mạng Tháng Tám, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "restaurant_kfc_train_032.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_viettel_train_169.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT"
  },
  {
    "image_name": "cafe_phuclong_train_144.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_viettel_val_011.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 188 Nguyễn Trãi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_winmart_train_021.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "10.000"
  },
  {
    "image_name": "supermarket_lotte_val_019.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước cam ép Twister 1L, Bột giặt Ariel 3.2kg, Bia Heineken Lon 330ml, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "einvoice_vnpt_train_031.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_kfc_train_021.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC Nguyễn Ảnh Thủ\", \"address\": \"Số 35 Cách Mạng Tháng Tám, Quận 1, Cần Thơ\", \"timestamp\": \"15/06/2026 10:12\", \"total_cost\": \"379,080\", \"items\": [{\"name\": \"Bắp Cải Trộn L\", \"qty\": \"1\", \"amount\": \"22,000\"}, {\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"4\", \"amount\": \"72,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"2\", \"amount\": \"56,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"3\", \"amount\": \"117,000\"}, {\"name\": \"Burger Tôm\", \"qty\": \"2\", \"amount\": \"84,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_135.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 LANDMARK 81\", \"address\": \"Số 165 Hai Bà Trưng, Quận 1, Đồng Nai\", \"timestamp\": \"16/06/2026 09:56\", \"total_cost\": \"234,300đ\", \"items\": [{\"name\": \"Cơm nắm cá hồi sốt Mayo\", \"qty\": \"3\", \"amount\": \"45,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"3\", \"amount\": \"42,000\"}, {\"name\": \"Sữa chua uống Proby\", \"qty\": \"4\", \"amount\": \"32,000\"}, {\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"1\", \"amount\": \"16,000\"}, {\"name\": \"Cafe sữa đá GS25\", \"qty\": \"3\", \"amount\": \"66,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_val_066.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "10/06/2026 14:10"
  },
  {
    "image_name": "restaurant_jollibee_train_065.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "image_name": "convenience_gs25_train_266.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "13/06/2026 11:45"
  },
  {
    "image_name": "minimart_anan_train_196.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "14/06/2026 11:11"
  },
  {
    "image_name": "cafe_highlands_train_196.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "35,000"
  },
  {
    "image_name": "supermarket_lotte_train_202.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"286,200.00\", \"box\": [313, 549, 388, 572]}"
  },
  {
    "image_name": "cafe_highlands_val_048.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "einvoice_viettel_train_236.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mực in Canon Cartridge trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000,000"
  },
  {
    "image_name": "cafe_phuclong_val_050.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "220,000"
  },
  {
    "image_name": "cafe_phuclong_train_167.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Croissant Bơ Pháp trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "convenience_gs25_train_188.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "25/05/2026 12:07"
  },
  {
    "image_name": "einvoice_vnpt_train_160.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 218 Cách Mạng Tháng Tám, Quận 1, Đồng Nai"
  },
  {
    "image_name": "cafe_starbucks_val_035.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "792,000"
  },
  {
    "image_name": "convenience_gs25_train_022.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_169.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "supermarket_lotte_train_093.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 114 Nguyễn Trãi, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"30/05/2026 13:46:00\", \"total_cost\": \"739,200.00\", \"items\": [{\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"2\", \"amount\": \"44,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"2\", \"amount\": \"64,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"2\", \"amount\": \"39,000.00\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_119.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026"
  },
  {
    "image_name": "restaurant_jollibee_train_031.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "114,000"
  },
  {
    "image_name": "convenience_7eleven_val_036.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"232,200\", \"box\": [305, 264, 355, 288]}"
  },
  {
    "image_name": "supermarket_lotte_train_160.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "324,500.00"
  },
  {
    "image_name": "supermarket_lotte_train_130.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CO.OP FOOD NGUYỄN ĐÌNH CHIỂU"
  },
  {
    "image_name": "einvoice_viettel_train_179.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Giấy Double A A4 70gsm, Mực in Canon Cartridge, Máy in HP LaserJet Pro, Bút bi Thiên Long FO-03"
  },
  {
    "image_name": "convenience_circlek_train_230.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "supermarket_winmart_train_142.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_016.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "22/05/2026 16:36"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_109.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 385 Lê Lợi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_lotte_train_072.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 22:53:00"
  },
  {
    "image_name": "convenience_7eleven_train_038.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "04/06/2026 11:26"
  },
  {
    "image_name": "supermarket_lotte_train_090.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt đùi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "204,000"
  },
  {
    "image_name": "convenience_gs25_train_039.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 LANDMARK 81\", \"box\": [144, 15, 263, 36]}"
  },
  {
    "image_name": "convenience_gs25_train_188.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_042.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "136.000"
  },
  {
    "image_name": "convenience_gs25_train_063.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 404 Bùi Thị Xuân, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_264.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"246,400\", \"box\": [309, 576, 360, 598]}"
  },
  {
    "image_name": "supermarket_lotte_train_160.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Hộp dâu tây Đà Lạt 250g, Nước cam ép Twister 1L, Bánh quy Cosy Kinh Đô"
  },
  {
    "image_name": "einvoice_viettel_train_240.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "einvoice_viettel_val_067.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT"
  },
  {
    "image_name": "convenience_circlek_train_077.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Băng cá nhân Urgo hộp 10 miếng, Kẹo cao su Cool Air hũ, Nước uống Aquafina 500ml, Mì Ly ăn liền Modern"
  },
  {
    "image_name": "convenience_7eleven_val_059.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước khoáng Dasani 500ml, Snack khoai tây Lays 95g, Mì xào tương đen, Kẹo dẻo Haribo Goldbears"
  },
  {
    "image_name": "convenience_gs25_train_108.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì trộn Indomie đặc biệt trên hóa đơn là bao nhiêu?",
    "ground_truth": "48,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_195.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"230,040\", \"box\": [313, 372, 365, 393]}"
  },
  {
    "image_name": "restaurant_jollibee_val_018.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "JOLLIBEE VINCOM"
  },
  {
    "image_name": "cafe_phuclong_val_002.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_gs25_train_126.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 212 Trần Hưng Đạo, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "convenience_circlek_train_004.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"Circle K Phạm Ngũ Lão\", \"box\": [135, 20, 271, 43]}"
  },
  {
    "image_name": "einvoice_viettel_train_209.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 344 Bùi Thị Xuân, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "convenience_circlek_train_008.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "minimart_anan_train_113.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "minimart_anan_val_035.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_030.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bút bi Thiên Long FO-03, Giấy Double A A4 70gsm, Dịch vụ Bảo trì Hệ thống mạng"
  },
  {
    "image_name": "convenience_7eleven_train_221.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "convenience_circlek_val_017.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "18/06/2026 12:11"
  },
  {
    "image_name": "supermarket_winmart_train_097.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+HN TRẦN DUY HƯNG\", \"address\": \"Số 352 Lý Thường Kiệt, Quận 1, Hải Phòng\", \"timestamp\": \"07/06/2026 10:54\", \"total_cost\": \"352.080\", \"items\": [{\"name\": \"Bánh Custas Orion 6P\", \"qty\": \"3\", \"amount\": \"102.000\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"4\", \"amount\": \"72.000\"}, {\"name\": \"Coca Cola Lon 320ml\", \"qty\": \"4\", \"amount\": \"40.000\"}, {\"name\": \"Dầu ăn Simply 1L\", \"qty\": \"2\", \"amount\": \"112.000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_val_040.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bia Heineken Lon 330ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_gs25_train_050.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Kirin Latte 345ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_7eleven_train_168.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"155,520\", \"box\": [313, 234, 365, 258]}"
  },
  {
    "image_name": "cafe_highlands_train_263.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Thạch Đào Size L, Freeze Trà Xanh Size M, Trà Sen Vàng Size M, Phin Sữa Đá Size L, Bánh Tiramisu"
  },
  {
    "image_name": "einvoice_vnpt_train_071.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bàn phím cơ Dareu EK87 trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,350,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_069.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH QUẬN 12\", \"address\": \"Số 446 Hai Bà Trưng, Quận 1, Hải Phòng\", \"timestamp\": \"08/06/2026 19:04\", \"total_cost\": \"110,000\", \"items\": [{\"name\": \"Hành lá 100g\", \"qty\": \"4\", \"amount\": \"12,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"2\", \"amount\": \"64,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"3\", \"amount\": \"24,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Freeze Trà Xanh Size M là bao nhiêu?",
    "ground_truth": "165,000"
  },
  {
    "image_name": "cafe_phuclong_train_220.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Đào Phúc Long L, Bánh Croissant Bơ Pháp, Trà Lài Đác Thơm L, Hồng Trà Sữa M, Trà sữa Phúc Long L, Cà phê Latte"
  },
  {
    "image_name": "cafe_starbucks_train_209.png",
    "template": "cafe_starbucks",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 444 Hai Bà Trưng, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "convenience_gs25_train_258.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 07:18"
  },
  {
    "image_name": "restaurant_jollibee_train_240.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_089.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CỬA HÀNG TIỆN LỢI AN AN\", \"address\": \"Số 96 Cách Mạng Tháng Tám, Quận 1, Đồng Nai\", \"timestamp\": \"30/05/2026 20:04\", \"total_cost\": \"483.840\", \"items\": [{\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"3\", \"amount\": \"66.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"1\", \"amount\": \"32.000\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"2\", \"amount\": \"350.000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_001.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 10:38"
  },
  {
    "image_name": "restaurant_kfc_train_256.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_vnpt_train_028.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "10,000,000"
  },
  {
    "image_name": "einvoice_viettel_train_072.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Máy in HP LaserJet Pro trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,400,000"
  },
  {
    "image_name": "cafe_highlands_val_009.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Tiramisu trên hóa đơn là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_056.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "convenience_circlek_val_039.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Băng cá nhân Urgo hộp 10 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_048.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "300,000"
  },
  {
    "image_name": "restaurant_kfc_val_057.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"245,160\", \"box\": [321, 469, 378, 491]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_198.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH NGUYỄN TRÃI\", \"box\": [96, 15, 284, 36]}"
  },
  {
    "image_name": "minimart_anan_train_064.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"210.600\", \"box\": [337, 385, 385, 406]}"
  },
  {
    "image_name": "cafe_highlands_train_027.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Thạch Đào Size L, Bánh Tiramisu"
  },
  {
    "image_name": "receipt_c45_bb_train_228.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"box\": [84, 28, 332, 43]}"
  },
  {
    "image_name": "supermarket_lotte_train_239.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "700,000.00"
  },
  {
    "image_name": "convenience_gs25_train_127.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "48,000"
  },
  {
    "image_name": "restaurant_jollibee_val_067.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE COOPMART\", \"address\": \"Số 91 Nguyễn Huệ, Quận 1, Cần Thơ\", \"timestamp\": \"24/05/2026 16:30\", \"total_cost\": \"418,000đ\", \"items\": [{\"name\": \"Bánh Khoai Môn\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"3\", \"amount\": \"105,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"2\", \"amount\": \"50,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"3\", \"amount\": \"114,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"3\", \"amount\": \"51,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_061.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước mắm Nam Ngư 750ml là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "supermarket_lotte_train_097.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh quy Cosy Kinh Đô trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000.00"
  },
  {
    "image_name": "receipt_c45_bb_val_039.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "minimart_anan_train_176.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "27.000"
  },
  {
    "image_name": "supermarket_lotte_train_137.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hộp dâu tây Đà Lạt 250g trên hóa đơn là bao nhiêu?",
    "ground_truth": "195,000.00"
  },
  {
    "image_name": "cafe_phuclong_train_111.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "image_name": "einvoice_vnpt_train_131.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "1,700,000"
  },
  {
    "image_name": "cafe_phuclong_val_025.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Croissant Bơ Pháp, Trà sữa Phúc Long L, Trà Lài Đác Thơm L, Hồng Trà Sữa M"
  },
  {
    "image_name": "restaurant_kfc_val_012.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "409,200"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_008.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"800,280\", \"box\": [304, 584, 356, 605]}"
  },
  {
    "image_name": "supermarket_winmart_train_040.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VINCOMMERCE\", \"address\": \"Số 142 Cách Mạng Tháng Tám, Quận 1, Đồng Nai\", \"timestamp\": \"25/05/2026 20:24\", \"total_cost\": \"126.900\", \"items\": [{\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"1\", \"amount\": \"3.500\"}, {\"name\": \"Sữa tươi TH True Milk 1L\", \"qty\": \"3\", \"amount\": \"114.000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_val_012.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 365 Lê Lợi, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "restaurant_kfc_train_214.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 283 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "restaurant_kfc_train_184.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Trứng Egg Tart, Gà Giòn Cay 1 miếng, Burger Tôm, Nước ngọt Pepsi L, Khoai Tây Chiên L"
  },
  {
    "image_name": "convenience_circlek_val_015.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "receipt_c45_bb_train_004.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 378 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 22 tháng 05 năm 2026\", \"total_cost\": \"432.000 VND\"}"
  },
  {
    "image_name": "convenience_circlek_train_217.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 08:54"
  },
  {
    "image_name": "einvoice_viettel_train_121.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "supermarket_lotte_train_170.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"612,360.00\", \"box\": [315, 481, 387, 504]}"
  },
  {
    "image_name": "cafe_starbucks_train_260.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "einvoice_vnpt_train_174.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 12 Trần Hưng Đạo, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "convenience_gs25_train_149.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"138,600đ\", \"box\": [330, 387, 389, 410]}"
  },
  {
    "image_name": "supermarket_lotte_val_012.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 421 Lê Lợi, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_lotte_train_092.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn ướt Bobby 80 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "81,000.00"
  },
  {
    "image_name": "einvoice_vnpt_train_043.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "09/06/2026 12:10"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_018.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Đường cát trắng Biên Hòa 1kg là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"330,000\", \"box\": [313, 358, 365, 379]}"
  },
  {
    "image_name": "cafe_highlands_train_054.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 448 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "restaurant_jollibee_val_065.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 203 Bùi Thị Xuân, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_gs25_train_178.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 446 Bùi Thị Xuân, Quận 1, Cần Thơ"
  },
  {
    "image_name": "restaurant_kfc_train_136.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "KFC Nguyễn Ảnh Thủ"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_018.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH\", \"address\": \"Số 289 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"20/06/2026 14:55\", \"total_cost\": \"528,120\", \"items\": [{\"name\": \"Rau muống nước 500g\", \"qty\": \"4\", \"amount\": \"32,000\"}, {\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"3\", \"amount\": \"225,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"4\", \"amount\": \"104,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"4\", \"amount\": \"128,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_239.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "image_name": "convenience_7eleven_val_063.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "72,000"
  },
  {
    "image_name": "cafe_phuclong_train_098.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_017.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_7eleven_train_208.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-Eleven Nguyễn Du\", \"box\": [134, 37, 254, 57]}"
  },
  {
    "image_name": "receipt_c45_bb_val_066.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 10 tháng 06 năm 2026"
  },
  {
    "image_name": "restaurant_kfc_val_004.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "304,700"
  },
  {
    "image_name": "einvoice_viettel_train_204.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,507,360đ"
  },
  {
    "image_name": "cafe_highlands_train_067.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_104.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 16:28:00"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_068.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_012.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_124.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "convenience_circlek_train_166.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 386 Nguyễn Trãi, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "cafe_starbucks_train_108.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 231 Trần Hưng Đạo, Quận 1, Cần Thơ\", \"timestamp\": \"23/05/2026\", \"total_cost\": \"405,000\", \"items\": [{\"name\": \"Butter Croissant\", \"qty\": \"4\", \"amount\": \"160,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"2\", \"amount\": \"130,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_062.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "28/05/2026 22:29"
  },
  {
    "image_name": "convenience_circlek_train_103.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,000"
  },
  {
    "image_name": "cafe_phuclong_train_039.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 271 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "restaurant_kfc_val_042.png",
    "template": "restaurant_kfc",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"210,600\", \"box\": [325, 417, 384, 438]}"
  },
  {
    "image_name": "minimart_anan_val_028.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CỬA HÀNG TIỆN LỢI AN AN\", \"address\": \"Số 277 Lê Lợi, Quận 1, Đồng Nai\", \"timestamp\": \"22/05/2026 22:04\", \"total_cost\": \"266.200\", \"items\": [{\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"4\", \"amount\": \"32.000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"3\", \"amount\": \"30.000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24.000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"2\", \"amount\": \"44.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"3\", \"amount\": \"96.000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_075.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Sữa chua uống Proby, Cafe sữa đá GS25, Mì trộn Indomie đặc biệt"
  },
  {
    "image_name": "einvoice_viettel_train_254.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 18:07"
  },
  {
    "image_name": "einvoice_vnpt_train_103.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_val_066.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 15 Lý Thường Kiệt, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "cafe_highlands_train_184.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Tiramisu, Bánh Mì Que Gà Xé, Trà Thạch Đào Size L, Trà Sen Vàng Size M"
  },
  {
    "image_name": "cafe_starbucks_val_013.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "09/06/2026"
  },
  {
    "image_name": "cafe_starbucks_train_257.png",
    "template": "cafe_starbucks",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"743,850\", \"box\": [306, 511, 355, 533]}"
  },
  {
    "image_name": "cafe_highlands_train_157.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "19,000"
  },
  {
    "image_name": "supermarket_winmart_train_085.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Custas Orion 6P trên hóa đơn là bao nhiêu?",
    "ground_truth": "136.000"
  },
  {
    "image_name": "cafe_highlands_train_040.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Phin Sữa Đá Size L trên hóa đơn là bao nhiêu?",
    "ground_truth": "117,000"
  },
  {
    "image_name": "receipt_c45_bb_train_011.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 347 Nguyễn Trãi, Quận 1, Cần Thơ\", \"timestamp\": \"Ngày 11 tháng 06 năm 2026\", \"total_cost\": \"1.870.000 VND\"}"
  },
  {
    "image_name": "supermarket_winmart_val_025.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 386 Hai Bà Trưng, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "supermarket_winmart_train_060.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 377 Trần Hưng Đạo, Quận 1, TP. Hồ Chí Minh"
  },
  {
    "image_name": "receipt_c45_bb_train_185.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 12 Lý Thường Kiệt, Quận Hải Châu, Đà Nẵng"
  },
  {
    "image_name": "supermarket_lotte_val_023.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"280,800.00\", \"box\": [318, 485, 393, 506]}"
  },
  {
    "image_name": "einvoice_vnpt_train_206.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "02/06/2026 10:46"
  },
  {
    "image_name": "supermarket_winmart_train_237.png",
    "template": "supermarket_winmart",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"VM+QNH CẨM PHẢ\", \"address\": \"Số 440 Hai Bà Trưng, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"23/05/2026 18:19\", \"total_cost\": \"106.150\", \"items\": [{\"name\": \"Bánh Custas Orion 6P\", \"qty\": \"2\", \"amount\": \"68.000\"}, {\"name\": \"Mì tôm Hảo Hảo chua cay\", \"qty\": \"3\", \"amount\": \"10.500\"}, {\"name\": \"Khăn giấy Pulppy 100 tờ\", \"qty\": \"1\", \"amount\": \"18.000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_064.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Khoai Tây Lắc Phô Mai, Nước ngọt 7Up L, Mì Ý Sốt Bò Bằm, Bánh Khoai Môn"
  },
  {
    "image_name": "restaurant_kfc_train_003.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_viettel_train_061.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026 17:10"
  },
  {
    "image_name": "convenience_7eleven_val_029.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-Eleven Saigon Trade Center\", \"timestamp\": \"16/06/2026 17:30\", \"total_cost\": \"174,960\", \"items\": [{\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"20,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"72,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_022.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "18/06/2026 16:45"
  },
  {
    "image_name": "minimart_anan_val_006.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "436.320"
  },
  {
    "image_name": "cafe_starbucks_train_191.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Green Tea Latte M trên hóa đơn là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "einvoice_viettel_train_187.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"4,128,840đ\", \"box\": [667, 764, 742, 785]}"
  },
  {
    "image_name": "supermarket_winmart_train_237.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "68.000"
  },
  {
    "image_name": "cafe_highlands_train_226.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Thạch Đào Size L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_phuclong_train_012.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "minimart_anan_train_134.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Kẹo cao su Cool Air hũ, Bột giặt Ariel 3.2kg, Nước uống Aquafina 500ml"
  },
  {
    "image_name": "convenience_7eleven_val_008.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kẹo dẻo Haribo Goldbears, Mì xào tương đen, Nước khoáng Dasani 500ml, Sandwich xá xíu phô mai, Snack khoai tây Lays 95g"
  },
  {
    "image_name": "minimart_anan_train_257.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_076.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Mì Que Gà Xé trên hóa đơn là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "supermarket_lotte_train_039.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 257 Nguyễn Trãi, Quận 1, Bình Dương"
  },
  {
    "image_name": "cafe_highlands_train_004.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "cafe_highlands_train_117.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 268 Nguyễn Trãi, Quận 1, Cần Thơ"
  },
  {
    "image_name": "receipt_c45_bb_train_134.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [91, 32, 358, 51]}"
  },
  {
    "image_name": "supermarket_winmart_train_185.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước rửa chén Sunlight 750ml, Coca Cola Lon 320ml, Khăn giấy Pulppy 100 tờ, Dầu ăn Simply 1L, Bánh Custas Orion 6P, Sữa tươi TH True Milk 1L"
  },
  {
    "image_name": "restaurant_kfc_train_081.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC VIỆT NAM\", \"address\": \"Số 146 Lý Thường Kiệt, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"26/05/2026 09:12\", \"total_cost\": \"421,300\", \"items\": [{\"name\": \"Bắp Cải Trộn L\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Burger Tôm\", \"qty\": \"3\", \"amount\": \"126,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"3\", \"amount\": \"117,000\"}, {\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"1\", \"amount\": \"18,000\"}, {\"name\": \"Khoai Tây Chiên L\", \"qty\": \"2\", \"amount\": \"56,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_val_060.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"172,800\", \"box\": [308, 362, 360, 383]}"
  },
  {
    "image_name": "cafe_highlands_train_255.png",
    "template": "cafe_highlands",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 398 Lý Thường Kiệt, Quận 1, Cần Thơ"
  },
  {
    "image_name": "minimart_anan_train_089.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 20:04"
  },
  {
    "image_name": "supermarket_lotte_train_142.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 08:21:00"
  },
  {
    "image_name": "supermarket_winmart_train_170.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 199 Lê Lợi, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_phuclong_val_047.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "55,000"
  },
  {
    "image_name": "restaurant_jollibee_train_094.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "119,880đ"
  },
  {
    "image_name": "supermarket_lotte_val_064.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Khăn ướt Bobby 80 tờ, Nước cam ép Twister 1L, Kem đánh răng Colgate, Bột giặt Ariel 3.2kg"
  },
  {
    "image_name": "convenience_circlek_val_047.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"92,400đ\", \"box\": [340, 418, 391, 440]}"
  },
  {
    "image_name": "convenience_gs25_val_043.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "06/06/2026 09:30"
  },
  {
    "image_name": "cafe_starbucks_val_059.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "641,250"
  },
  {
    "image_name": "supermarket_winmart_train_033.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dầu ăn Simply 1L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_val_043.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 13 Bùi Thị Xuân, Quận 1, Bình Dương"
  },
  {
    "image_name": "cafe_phuclong_train_013.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG TEA & COFFEE"
  },
  {
    "image_name": "restaurant_jollibee_train_224.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt 7Up L trên hóa đơn là bao nhiêu?",
    "ground_truth": "51,000"
  },
  {
    "image_name": "restaurant_kfc_train_055.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 13:47"
  },
  {
    "image_name": "minimart_anan_train_075.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "16.000"
  },
  {
    "image_name": "convenience_gs25_train_071.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 TÔN THẤT THUYẾT"
  },
  {
    "image_name": "supermarket_lotte_train_114.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 112 Lê Lợi, Quận 1, Đồng Nai\", \"timestamp\": \"18/06/2026 21:19:00\", \"total_cost\": \"977,400.00\", \"items\": [{\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"2\", \"amount\": \"54,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"4\", \"amount\": \"700,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"2\", \"amount\": \"39,000.00\"}, {\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"4\", \"amount\": \"112,000.00\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_213.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_216.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "supermarket_winmart_train_137.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 13:56"
  },
  {
    "image_name": "cafe_starbucks_train_239.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS REX HOTEL\", \"address\": \"Số 256 Cách Mạng Tháng Tám, Quận 1, Bình Dương\", \"timestamp\": \"24/05/2026\", \"total_cost\": \"690,120\", \"items\": [{\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"1\", \"amount\": \"40,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Green Tea Latte M\", \"qty\": \"3\", \"amount\": \"225,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_val_034.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Lài Đác Thơm L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_lotte_train_098.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bột giặt Ariel 3.2kg, Khăn ướt Bobby 80 tờ, Nước cam ép Twister 1L, Hộp dâu tây Đà Lạt 250g, Kem đánh răng Colgate"
  },
  {
    "image_name": "convenience_gs25_val_020.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_gs25_train_095.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 NGUYỄN HUỆ\", \"box\": [150, 36, 255, 57]}"
  },
  {
    "image_name": "supermarket_lotte_train_120.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 235 Bùi Thị Xuân, Quận 1, Hải Phòng"
  },
  {
    "image_name": "cafe_phuclong_train_196.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "11/06/2026 14:03"
  },
  {
    "image_name": "einvoice_vnpt_train_012.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dịch vụ Lắp đặt camera giám sát, Chuột không dây Logitech, Cáp mạng Cat6 UTP 305m, Bàn phím cơ Dareu EK87, Bộ phát Wifi TP-Link Archer"
  },
  {
    "image_name": "convenience_gs25_val_019.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh giò thịt băm trứng cút là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "receipt_c45_bb_train_002.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 305 Lê Lợi, Quận 1, Đồng Nai\", \"timestamp\": \"Ngày 22 tháng 05 năm 2026\", \"total_cost\": \"440.000 VND\"}"
  },
  {
    "image_name": "convenience_gs25_train_229.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_lotte_train_191.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "receipt_c45_bb_val_024.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"box\": [85, 27, 346, 42]}"
  },
  {
    "image_name": "einvoice_vnpt_train_069.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "image_name": "einvoice_vnpt_train_245.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bàn phím cơ Dareu EK87, Cáp mạng Cat6 UTP 305m, Chuột không dây Logitech, Dịch vụ Lắp đặt camera giám sát, Bộ phát Wifi TP-Link Archer"
  },
  {
    "image_name": "einvoice_vnpt_train_163.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,850,000"
  },
  {
    "image_name": "cafe_highlands_val_043.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"611,600\", \"box\": [310, 409, 361, 431]}"
  },
  {
    "image_name": "cafe_highlands_val_031.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE LÊ LỢI\", \"address\": \"Số 274 Hai Bà Trưng, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"01/06/2026 20:08\", \"total_cost\": \"585,846\", \"items\": [{\"name\": \"Phin Sữa Đá Size L\", \"qty\": \"3\", \"amount\": \"156,000\"}, {\"name\": \"Bánh Mì Que Gà Xé\", \"qty\": \"2\", \"amount\": \"57,000\"}, {\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"3\", \"amount\": \"58,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"3\", \"amount\": \"165,000\"}, {\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"1\", \"amount\": \"135,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_104.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước uống Aquafina 500ml là bao nhiêu?",
    "ground_truth": "6.000"
  },
  {
    "image_name": "convenience_circlek_train_120.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Băng cá nhân Urgo hộp 10 miếng trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_246.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "image_name": "einvoice_viettel_val_049.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN CÔNG NGHỆ SAO NAM\", \"box\": [55, 99, 287, 121]}"
  },
  {
    "image_name": "cafe_starbucks_train_135.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "80,000"
  },
  {
    "image_name": "convenience_gs25_train_252.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_circlek_train_002.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "56,160đ"
  },
  {
    "image_name": "cafe_phuclong_train_128.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 97 Lý Thường Kiệt, Quận 1, Cần Thơ"
  },
  {
    "image_name": "supermarket_winmart_train_096.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "72.000"
  },
  {
    "image_name": "receipt_c45_bb_train_194.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 126 Bùi Thị Xuân, Quận 1, Hải Phòng\", \"timestamp\": \"Ngày 26 tháng 05 năm 2026\", \"total_cost\": \"162.000 VND\"}"
  },
  {
    "image_name": "supermarket_lotte_train_145.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_phuclong_train_062.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_circlek_train_026.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 181 Hai Bà Trưng, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "image_name": "minimart_anan_train_177.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 22:57"
  },
  {
    "image_name": "cafe_highlands_train_001.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "39,000"
  },
  {
    "image_name": "convenience_7eleven_train_245.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-Eleven Nguyễn Du\", \"timestamp\": \"16/06/2026 08:38\", \"total_cost\": \"254,880\", \"items\": [{\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"100,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"40,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_177.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 09:28"
  },
  {
    "image_name": "cafe_highlands_val_001.png",
    "template": "cafe_highlands",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"HIGHLANDS COFFEE TRẦN HƯNG ĐẠO\", \"address\": \"Số 116 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"31/05/2026 16:41\", \"total_cost\": \"796,068\", \"items\": [{\"name\": \"Trà Sen Vàng Size M\", \"qty\": \"2\", \"amount\": \"135,000\"}, {\"name\": \"Cà Phê Đen Đá Size M\", \"qty\": \"2\", \"amount\": \"58,000\"}, {\"name\": \"Trà Thạch Đào Size L\", \"qty\": \"4\", \"amount\": \"110,000\"}, {\"name\": \"Bánh Tiramisu\", \"qty\": \"4\", \"amount\": \"140,000\"}, {\"name\": \"Freeze Trà Xanh Size M\", \"qty\": \"4\", \"amount\": \"220,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_146.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn ướt Bobby 80 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "27,000.00"
  },
  {
    "image_name": "convenience_circlek_train_094.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 261 Cách Mạng Tháng Tám, Thành phố Thủ Đức, TP. Hồ Chí Minh"
  },
  {
    "image_name": "convenience_gs25_train_252.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì trộn Indomie đặc biệt, Bánh giò thịt băm trứng cút, Cơm nắm cá hồi sốt Mayo, Trà sữa Kirin Latte 345ml, Sữa chua uống Proby, Xúc xích Đức ăn liền"
  },
  {
    "image_name": "einvoice_vnpt_train_033.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH"
  },
  {
    "image_name": "restaurant_jollibee_train_229.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "238,700đ"
  },
  {
    "image_name": "convenience_gs25_train_014.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh giò thịt băm trứng cút là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_128.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Khoai Môn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "cafe_starbucks_train_203.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Mì Que Gà Xé, Freeze Trà Xanh Size M, Cà Phê Đen Đá Size M, Bánh Tiramisu, Trà Thạch Đào Size L"
  },
  {
    "image_name": "restaurant_kfc_train_072.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026 13:50"
  },
  {
    "image_name": "convenience_7eleven_train_241.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "supermarket_winmart_train_135.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "40.000"
  },
  {
    "image_name": "cafe_phuclong_train_203.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Cà phê Latte, Trà Đào Phúc Long L, Trà Lài Đác Thơm L, Hồng Trà Sữa M, Bánh Croissant Bơ Pháp, Trà sữa Phúc Long L"
  },
  {
    "image_name": "cafe_phuclong_train_138.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "restaurant_kfc_train_155.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Chiên L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "receipt_c45_bb_train_057.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"1.870.000 VND\", \"box\": [122, 260, 239, 275]}"
  },
  {
    "image_name": "cafe_starbucks_train_096.png",
    "template": "cafe_starbucks",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Butter Croissant, Green Tea Latte M"
  },
  {
    "image_name": "convenience_circlek_train_151.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 127 Hai Bà Trưng, Quận 1, Đồng Nai"
  },
  {
    "image_name": "einvoice_vnpt_train_260.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bộ phát Wifi TP-Link Archer trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,700,000"
  },
  {
    "image_name": "convenience_circlek_train_020.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_starbucks_val_002.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_243.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"729,300\", \"box\": [308, 565, 356, 585]}"
  },
  {
    "image_name": "supermarket_lotte_train_051.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "31/05/2026 11:39:00"
  },
  {
    "image_name": "convenience_gs25_train_124.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 TÔN THẤT THUYẾT\", \"address\": \"Số 214 Hai Bà Trưng, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"20/06/2026 11:56\", \"total_cost\": \"129,600đ\", \"items\": [{\"name\": \"Xúc xích Đức ăn liền\", \"qty\": \"2\", \"amount\": \"20,000\"}, {\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"3\", \"amount\": \"48,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"2\", \"amount\": \"28,000\"}, {\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"2\", \"amount\": \"24,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_val_067.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "170,000"
  },
  {
    "image_name": "restaurant_kfc_train_144.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Burger Tôm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "minimart_anan_train_160.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 216 Bùi Thị Xuân, Quận Thanh Xuân, Hà Nội"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_133.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_173.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_highlands_train_114.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "convenience_gs25_val_006.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cơm nắm cá hồi sốt Mayo trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "restaurant_kfc_val_013.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Trứng Egg Tart là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_vnpt_train_121.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 17:48"
  },
  {
    "image_name": "receipt_c45_bb_train_083.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "convenience_circlek_val_017.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 29 Trần Hưng Đạo, Quận Hải Châu, Đà Nẵng\", \"timestamp\": \"18/06/2026 12:11\", \"total_cost\": \"100,440đ\", \"items\": [{\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"4\", \"amount\": \"40,000\"}, {\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"3\", \"amount\": \"45,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_108.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "88.000"
  },
  {
    "image_name": "cafe_starbucks_train_218.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS REX HOTEL"
  },
  {
    "image_name": "convenience_gs25_train_209.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh giò thịt băm trứng cút là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_lotte_train_072.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 22:53:00"
  },
  {
    "image_name": "supermarket_lotte_train_099.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "895,320.00"
  },
  {
    "image_name": "minimart_anan_val_057.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "66.000"
  },
  {
    "image_name": "convenience_gs25_train_181.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh giò thịt băm trứng cút là bao nhiêu?",
    "ground_truth": "28,000"
  },
  {
    "image_name": "einvoice_vnpt_train_203.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH"
  },
  {
    "image_name": "einvoice_vnpt_train_123.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Chuột không dây Logitech, Dịch vụ Lắp đặt camera giám sát, Cáp mạng Cat6 UTP 305m"
  },
  {
    "image_name": "cafe_starbucks_train_223.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "509,850"
  },
  {
    "image_name": "minimart_anan_train_085.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "16.000"
  },
  {
    "image_name": "convenience_circlek_val_012.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "28/05/2026 21:40"
  },
  {
    "image_name": "einvoice_viettel_train_079.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "17,011,500đ"
  },
  {
    "image_name": "restaurant_kfc_val_067.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Nguyễn Ảnh Thủ"
  },
  {
    "image_name": "supermarket_lotte_train_065.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Khăn ướt Bobby 80 tờ, Kem đánh răng Colgate, Hộp dâu tây Đà Lạt 250g, Bia Heineken Lon 330ml"
  },
  {
    "image_name": "cafe_starbucks_train_152.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Butter Croissant là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "cafe_starbucks_train_168.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "cafe_starbucks_train_080.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "image_name": "supermarket_winmart_train_099.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "226.800"
  },
  {
    "image_name": "convenience_gs25_val_011.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 387 Hai Bà Trưng, Quận 1, Cần Thơ"
  },
  {
    "image_name": "minimart_anan_train_263.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Xúc xích tiệt trùng Ponnie, Bột giặt Ariel 3.2kg, Kẹo cao su Cool Air hũ"
  },
  {
    "image_name": "receipt_c45_bb_train_073.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "restaurant_kfc_train_099.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "image_name": "convenience_7eleven_train_239.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "cafe_starbucks_val_026.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cold Brew Coffee M trên hóa đơn là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "convenience_7eleven_train_196.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 15:47"
  },
  {
    "image_name": "cafe_highlands_train_073.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "einvoice_viettel_train_077.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bút bi Thiên Long FO-03, Giấy Double A A4 70gsm, Mực in Canon Cartridge, Máy in HP LaserJet Pro, Dịch vụ Bảo trì Hệ thống mạng"
  },
  {
    "image_name": "supermarket_winmart_train_013.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước rửa chén Sunlight 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "87.000"
  },
  {
    "image_name": "cafe_starbucks_train_249.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "cafe_starbucks_train_062.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Chocolate Muffin trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "minimart_anan_train_173.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"226.800\", \"box\": [336, 347, 386, 369]}"
  },
  {
    "image_name": "einvoice_vnpt_train_179.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"address\": \"Số 360 Nguyễn Huệ, Quận 7, TP. Hồ Chí Minh\", \"timestamp\": \"12/06/2026 08:54\", \"total_cost\": \"15,271,200đ\", \"items\": [{\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"2\", \"amount\": \"3,700,000\"}, {\"name\": \"Bàn phím cơ Dareu EK87\", \"qty\": \"4\", \"amount\": \"1,800,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"1\", \"amount\": \"850,000\"}, {\"name\": \"Chuột không dây Logitech\", \"qty\": \"1\", \"amount\": \"290,000\"}, {\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"3\", \"amount\": \"7,500,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_181.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "406,080"
  },
  {
    "image_name": "cafe_highlands_train_063.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "97,200"
  },
  {
    "image_name": "convenience_gs25_train_165.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_train_016.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"address\": \"Số 117 Lê Lợi, Quận 1, Cần Thơ\", \"timestamp\": \"10/06/2026 07:33\", \"total_cost\": \"10,828,080đ\", \"items\": [{\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"4\", \"amount\": \"16,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"4\", \"amount\": \"260,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"4\", \"amount\": \"6,000,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"3\", \"amount\": \"3,750,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_train_036.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "VM+QNH HẠ LONG"
  },
  {
    "image_name": "cafe_starbucks_train_188.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Caramel Macchiato L trên hóa đơn là bao nhiêu?",
    "ground_truth": "85,000"
  },
  {
    "image_name": "convenience_circlek_train_227.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_002.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "225,000"
  },
  {
    "image_name": "supermarket_lotte_train_056.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 13:07:00"
  },
  {
    "image_name": "convenience_circlek_train_141.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "convenience_circlek_train_174.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Hai Bà Trưng"
  },
  {
    "image_name": "cafe_highlands_train_034.png",
    "template": "cafe_highlands",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 12:46"
  },
  {
    "image_name": "restaurant_kfc_train_173.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "213,840"
  },
  {
    "image_name": "cafe_phuclong_val_006.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "12/06/2026 21:04"
  },
  {
    "image_name": "restaurant_kfc_train_096.png",
    "template": "restaurant_kfc",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "579,700"
  },
  {
    "image_name": "restaurant_kfc_train_085.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Burger Tôm trên hóa đơn là bao nhiêu?",
    "ground_truth": "42,000"
  },
  {
    "image_name": "einvoice_viettel_train_089.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 364 Lý Thường Kiệt, Quận 1, Hải Phòng"
  },
  {
    "image_name": "einvoice_vnpt_train_193.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Lắp đặt camera giám sát trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,000,000"
  },
  {
    "image_name": "convenience_circlek_val_024.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Băng cá nhân Urgo hộp 10 miếng là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_gs25_train_175.png",
    "template": "convenience_gs25",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "28/05/2026 13:32"
  },
  {
    "image_name": "cafe_phuclong_train_127.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cà phê Latte là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_lotte_val_039.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bia Heineken Lon 330ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "78,000.00"
  },
  {
    "image_name": "supermarket_winmart_train_180.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì tôm Hảo Hảo chua cay là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_vnpt_train_185.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "1,800,000"
  },
  {
    "image_name": "minimart_anan_train_076.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ly ăn liền Modern trên hóa đơn là bao nhiêu?",
    "ground_truth": "27.000"
  },
  {
    "image_name": "supermarket_winmart_train_028.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Coca Cola Lon 320ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20.000"
  },
  {
    "image_name": "convenience_7eleven_train_234.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sandwich xá xíu phô mai là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_054.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH QUẬN 12\", \"box\": [117, 32, 270, 61]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_238.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 393 Nguyễn Huệ, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_180.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo dẻo Haribo Goldbears trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "einvoice_vnpt_train_079.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT\", \"address\": \"Số 209 Nguyễn Huệ, Quận 1, Cần Thơ\", \"timestamp\": \"03/06/2026 15:05\", \"total_cost\": \"14,344,000đ\", \"items\": [{\"name\": \"Chuột không dây Logitech\", \"qty\": \"1\", \"amount\": \"290,000\"}, {\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"1\", \"amount\": \"1,850,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"4\", \"amount\": \"3,400,000\"}, {\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"3\", \"amount\": \"7,500,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_013.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Hành lá 100g trên hóa đơn là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_148.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "436,700"
  },
  {
    "image_name": "restaurant_kfc_train_193.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC Nguyễn Ảnh Thủ\", \"address\": \"Số 333 Lý Thường Kiệt, Quận 1, Đồng Nai\", \"timestamp\": \"06/06/2026 17:19\", \"total_cost\": \"129,600\", \"items\": [{\"name\": \"Bắp Cải Trộn L\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"3\", \"amount\": \"54,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_193.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 15:20"
  },
  {
    "image_name": "convenience_gs25_train_059.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "193,600đ"
  },
  {
    "image_name": "convenience_gs25_val_030.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "34,100đ"
  },
  {
    "image_name": "convenience_gs25_train_157.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa chua uống Proby là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "einvoice_viettel_train_249.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 210 Lý Thường Kiệt, Quận 1, Bình Dương"
  },
  {
    "image_name": "cafe_phuclong_train_076.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_7eleven_val_013.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"7-ELEVEN BÙI VIỆN\", \"box\": [130, 17, 249, 38]}"
  },
  {
    "image_name": "einvoice_vnpt_train_213.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"12,595,000đ\", \"box\": [645, 761, 725, 781]}"
  },
  {
    "image_name": "convenience_7eleven_train_037.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-Eleven Nguyễn Du\", \"timestamp\": \"27/05/2026 09:34\", \"total_cost\": \"142,560\", \"items\": [{\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"18,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"19,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"10,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_262.png",
    "template": "restaurant_kfc",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 12:23"
  },
  {
    "image_name": "cafe_highlands_train_134.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "397,062"
  },
  {
    "image_name": "supermarket_lotte_train_100.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "44,000.00"
  },
  {
    "image_name": "supermarket_lotte_train_075.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CO.OPMART CỐNG QUỲNH\", \"box\": [132, 40, 280, 63]}"
  },
  {
    "image_name": "convenience_7eleven_train_062.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "cafe_highlands_val_039.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"162,800\", \"box\": [300, 369, 350, 392]}"
  },
  {
    "image_name": "cafe_phuclong_train_139.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà phê Latte trên hóa đơn là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "image_name": "convenience_7eleven_train_152.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "image_name": "einvoice_viettel_train_169.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Máy in HP LaserJet Pro trên hóa đơn là bao nhiêu?",
    "ground_truth": "15,400,000"
  },
  {
    "image_name": "einvoice_vnpt_train_186.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "2,550,000"
  },
  {
    "image_name": "supermarket_winmart_train_262.png",
    "template": "supermarket_winmart",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "195.800"
  },
  {
    "image_name": "restaurant_jollibee_train_261.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 19:48"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_033.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH\", \"box\": [146, 37, 241, 59]}"
  },
  {
    "image_name": "supermarket_winmart_train_147.png",
    "template": "supermarket_winmart",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 394 Nguyễn Huệ, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "einvoice_vnpt_train_027.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH"
  },
  {
    "image_name": "einvoice_vnpt_train_222.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "580,000"
  },
  {
    "image_name": "einvoice_viettel_train_226.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Máy in HP LaserJet Pro trên hóa đơn là bao nhiêu?",
    "ground_truth": "11,550,000"
  },
  {
    "image_name": "einvoice_viettel_train_036.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "25,147,100đ"
  },
  {
    "image_name": "restaurant_kfc_train_087.png",
    "template": "restaurant_kfc",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"KFC Nguyễn Ảnh Thủ\", \"address\": \"Số 291 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"03/06/2026 18:11\", \"total_cost\": \"500,040\", \"items\": [{\"name\": \"Bánh Trứng Egg Tart\", \"qty\": \"3\", \"amount\": \"54,000\"}, {\"name\": \"Gà Giòn Cay 1 miếng\", \"qty\": \"4\", \"amount\": \"156,000\"}, {\"name\": \"Nước ngọt Pepsi L\", \"qty\": \"1\", \"amount\": \"19,000\"}, {\"name\": \"Burger Tôm\", \"qty\": \"4\", \"amount\": \"168,000\"}, {\"name\": \"Bắp Cải Trộn L\", \"qty\": \"3\", \"amount\": \"66,000\"}]}"
  },
  {
    "image_name": "restaurant_kfc_train_195.png",
    "template": "restaurant_kfc",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Burger Tôm trên hóa đơn là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "einvoice_viettel_train_075.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "4,000"
  },
  {
    "image_name": "einvoice_viettel_train_124.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_winmart_train_190.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "152.000"
  },
  {
    "image_name": "cafe_starbucks_train_021.png",
    "template": "cafe_starbucks",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "STARBUCKS NEW WORLD"
  },
  {
    "image_name": "restaurant_jollibee_train_060.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "cafe_highlands_val_008.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "371,250"
  },
  {
    "image_name": "restaurant_kfc_train_202.png",
    "template": "restaurant_kfc",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Khoai Tây Chiên L, Bắp Cải Trộn L"
  },
  {
    "image_name": "cafe_starbucks_train_048.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "578,340"
  },
  {
    "image_name": "cafe_phuclong_train_098.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Phúc Long L trên hóa đơn là bao nhiêu?",
    "ground_truth": "200,000"
  },
  {
    "image_name": "restaurant_jollibee_train_164.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "receipt_c45_bb_train_031.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1.650.000 VND"
  },
  {
    "image_name": "minimart_anan_train_065.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"140.800\", \"box\": [340, 305, 392, 326]}"
  },
  {
    "image_name": "cafe_starbucks_train_133.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS HÀN THUYÊN\", \"address\": \"Số 394 Trần Hưng Đạo, Quận 1, Bình Dương\", \"timestamp\": \"19/06/2026\", \"total_cost\": \"481,140\", \"items\": [{\"name\": \"Cold Brew Coffee M\", \"qty\": \"2\", \"amount\": \"130,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"4\", \"amount\": \"160,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"2\", \"amount\": \"120,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"1\", \"amount\": \"85,000\"}]}"
  },
  {
    "image_name": "supermarket_winmart_val_033.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 09:52"
  },
  {
    "image_name": "restaurant_jollibee_train_064.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE COOPMART\", \"address\": \"Số 32 Trần Hưng Đạo, Quận 1, Bình Dương\", \"timestamp\": \"04/06/2026 18:15\", \"total_cost\": \"271,080đ\", \"items\": [{\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"3\", \"amount\": \"75,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"1\", \"amount\": \"17,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"3\", \"amount\": \"114,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"3\", \"amount\": \"45,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_176.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 164 Nguyễn Huệ, Quận Thanh Khê, Đà Nẵng\", \"timestamp\": \"16/06/2026\", \"total_cost\": \"858,600\", \"items\": [{\"name\": \"Caffe Americano L\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"2\", \"amount\": \"170,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Chocolate Muffin\", \"qty\": \"3\", \"amount\": \"135,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"4\", \"amount\": \"160,000\"}]}"
  },
  {
    "image_name": "receipt_c45_bb_train_135.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 02 tháng 06 năm 2026"
  },
  {
    "image_name": "einvoice_viettel_train_191.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 07:45"
  },
  {
    "image_name": "einvoice_vnpt_train_058.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bàn phím cơ Dareu EK87, Bộ phát Wifi TP-Link Archer, Chuột không dây Logitech"
  },
  {
    "image_name": "minimart_anan_train_121.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà xanh Không Độ 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "40.000"
  },
  {
    "image_name": "convenience_gs25_train_167.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh giò thịt băm trứng cút trên hóa đơn là bao nhiêu?",
    "ground_truth": "14,000"
  },
  {
    "image_name": "convenience_gs25_train_257.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 THỦ ĐỨC"
  },
  {
    "image_name": "convenience_7eleven_train_089.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "05/06/2026 11:17"
  },
  {
    "image_name": "einvoice_vnpt_val_032.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH"
  },
  {
    "image_name": "convenience_gs25_train_236.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích Đức ăn liền trên hóa đơn là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "restaurant_jollibee_train_187.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 41 Lý Thường Kiệt, Quận 1, Hải Phòng"
  },
  {
    "image_name": "einvoice_viettel_train_188.png",
    "template": "einvoice_viettel",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "22,999,680đ"
  },
  {
    "image_name": "cafe_highlands_train_151.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà Sen Vàng Size M trên hóa đơn là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "supermarket_winmart_train_010.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "29.000"
  },
  {
    "image_name": "einvoice_vnpt_train_076.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,120,000đ"
  },
  {
    "image_name": "supermarket_lotte_train_100.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CO.OP FOOD NGUYỄN ĐÌNH CHIỂU\", \"address\": \"Số 363 Cách Mạng Tháng Tám, Quận 1, Hải Phòng\", \"timestamp\": \"11/06/2026 10:21:00\", \"total_cost\": \"167,200.00\", \"items\": [{\"name\": \"Khăn ướt Bobby 80 tờ\", \"qty\": \"4\", \"amount\": \"108,000.00\"}, {\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"2\", \"amount\": \"44,000.00\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_val_040.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dịch vụ Lắp đặt camera giám sát trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,000,000"
  },
  {
    "image_name": "restaurant_jollibee_train_119.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_027.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "08/06/2026 19:55"
  },
  {
    "image_name": "supermarket_winmart_train_035.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Dầu ăn Simply 1L trên hóa đơn là bao nhiêu?",
    "ground_truth": "168.000"
  },
  {
    "image_name": "convenience_gs25_train_116.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"195,800đ\", \"box\": [333, 452, 392, 473]}"
  },
  {
    "image_name": "supermarket_winmart_train_187.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Custas Orion 6P trên hóa đơn là bao nhiêu?",
    "ground_truth": "136.000"
  },
  {
    "image_name": "cafe_phuclong_train_266.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Croissant Bơ Pháp trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "restaurant_kfc_train_253.png",
    "template": "restaurant_kfc",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC VIỆT NAM"
  },
  {
    "image_name": "restaurant_jollibee_val_064.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 191 Bùi Thị Xuân, Quận 1, Cần Thơ"
  },
  {
    "image_name": "minimart_anan_train_022.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_highlands_train_005.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_circlek_train_233.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 345 Lý Thường Kiệt, Quận 1, Bình Dương"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_054.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hành lá 100g là bao nhiêu?",
    "ground_truth": "6,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_139.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "290,400"
  },
  {
    "image_name": "einvoice_vnpt_train_068.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bộ phát Wifi TP-Link Archer, Bàn phím cơ Dareu EK87"
  },
  {
    "image_name": "einvoice_vnpt_train_190.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_train_158.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh Khoai Môn trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "supermarket_lotte_train_046.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"257,400.00\", \"box\": [318, 485, 393, 506]}"
  },
  {
    "image_name": "convenience_7eleven_train_079.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước khoáng Dasani 500ml là bao nhiêu?",
    "ground_truth": "15,000"
  },
  {
    "image_name": "convenience_gs25_val_020.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cơm nắm cá hồi sốt Mayo là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_067.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH LÊ ĐỨC THỌ"
  },
  {
    "image_name": "minimart_anan_train_157.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "16.000"
  },
  {
    "image_name": "restaurant_kfc_train_266.png",
    "template": "restaurant_kfc",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 277 Bùi Thị Xuân, Quận 1, Cần Thơ"
  },
  {
    "image_name": "einvoice_viettel_train_255.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "18/06/2026 20:49"
  },
  {
    "image_name": "cafe_starbucks_train_226.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "311,850"
  },
  {
    "image_name": "einvoice_vnpt_val_052.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "3,700,000"
  },
  {
    "image_name": "cafe_starbucks_train_248.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caffe Americano L là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_189.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "BÁCH HÓA XANH LÊ ĐỨC THỌ"
  },
  {
    "image_name": "supermarket_lotte_train_195.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bia Heineken Lon 330ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "39,000.00"
  },
  {
    "image_name": "convenience_gs25_val_043.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì trộn Indomie đặc biệt trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "einvoice_viettel_val_012.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"3,239,500đ\", \"box\": [671, 750, 745, 772]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_172.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Thịt ba rọi heo 500g trên hóa đơn là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "image_name": "supermarket_winmart_train_232.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_jollibee_val_066.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Gà Giòn Vui Vẻ 1 miếng, Mì Ý Sốt Bò Bằm, Khoai Tây Lắc Phô Mai"
  },
  {
    "image_name": "einvoice_viettel_train_084.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "1,250,000"
  },
  {
    "image_name": "supermarket_winmart_val_059.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 20:02"
  },
  {
    "image_name": "restaurant_jollibee_val_028.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt 7Up L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_019.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"address\": \"Số 383 Hai Bà Trưng, Quận 3, TP. Hồ Chí Minh\", \"timestamp\": \"19/06/2026 20:40\", \"total_cost\": \"826,100\", \"items\": [{\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"3\", \"amount\": \"225,000\"}, {\"name\": \"Thịt đùi heo 500g\", \"qty\": \"4\", \"amount\": \"272,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"4\", \"amount\": \"128,000\"}, {\"name\": \"Nước mắm Nam Ngư 750ml\", \"qty\": \"3\", \"amount\": \"126,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_046.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 20:29:00"
  },
  {
    "image_name": "restaurant_kfc_train_013.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_kfc_train_028.png",
    "template": "restaurant_kfc",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Cay 1 miếng là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_train_203.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "cafe_highlands_train_165.png",
    "template": "cafe_highlands",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"132,840\", \"box\": [304, 368, 355, 390]}"
  },
  {
    "image_name": "receipt_c45_bb_train_176.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 51 Cách Mạng Tháng Tám, Quận 1, Hải Phòng"
  },
  {
    "image_name": "einvoice_viettel_val_021.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"box\": [34, 66, 366, 92]}"
  },
  {
    "image_name": "convenience_7eleven_train_191.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Mì xào tương đen, Trà đào sả Slurpee, Snack khoai tây Lays 95g, Nước khoáng Dasani 500ml, Kẹo dẻo Haribo Goldbears"
  },
  {
    "image_name": "supermarket_winmart_val_003.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"229.500\", \"box\": [342, 484, 393, 506]}"
  },
  {
    "image_name": "convenience_gs25_train_251.png",
    "template": "convenience_gs25",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Xúc xích Đức ăn liền, Cơm nắm cá hồi sốt Mayo"
  },
  {
    "image_name": "minimart_anan_train_081.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"246.240\", \"box\": [340, 382, 390, 404]}"
  },
  {
    "image_name": "minimart_anan_train_179.png",
    "template": "minimart_anan",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "39.960"
  },
  {
    "image_name": "convenience_gs25_val_063.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì trộn Indomie đặc biệt trên hóa đơn là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "convenience_circlek_train_129.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 276 Hai Bà Trưng, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"05/06/2026 19:18\", \"total_cost\": \"195,480đ\", \"items\": [{\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"2\", \"amount\": \"44,000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24,000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"2\", \"amount\": \"32,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"3\", \"amount\": \"24,000\"}, {\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"2\", \"amount\": \"30,000\"}]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_211.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "03/06/2026 13:56"
  },
  {
    "image_name": "supermarket_winmart_train_022.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "receipt_c45_bb_train_142.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2.052.000 VND"
  },
  {
    "image_name": "convenience_gs25_train_138.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_vnpt_train_159.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "23/05/2026 12:34"
  },
  {
    "image_name": "convenience_circlek_train_107.png",
    "template": "convenience_circlek",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "Circle K Phạm Ngũ Lão"
  },
  {
    "image_name": "cafe_phuclong_train_099.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "convenience_circlek_train_086.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "restaurant_jollibee_train_207.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "354,200đ"
  },
  {
    "image_name": "cafe_phuclong_train_045.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"648,000\", \"box\": [312, 450, 364, 472]}"
  },
  {
    "image_name": "convenience_circlek_val_050.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "30/05/2026 13:12"
  },
  {
    "image_name": "supermarket_lotte_val_064.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"502,200.00\", \"box\": [311, 534, 386, 558]}"
  },
  {
    "image_name": "cafe_starbucks_train_003.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "supermarket_winmart_train_087.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn giấy Pulppy 100 tờ là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_circlek_train_231.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 315 Bùi Thị Xuân, Quận Thanh Xuân, Hà Nội"
  },
  {
    "image_name": "einvoice_vnpt_train_053.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"4,730,000đ\", \"box\": [659, 740, 732, 762]}"
  },
  {
    "image_name": "minimart_anan_train_103.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh bao trứng muối trên hóa đơn là bao nhiêu?",
    "ground_truth": "48.000"
  },
  {
    "image_name": "cafe_phuclong_train_247.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 420 Lý Thường Kiệt, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "supermarket_lotte_train_094.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 345 Cách Mạng Tháng Tám, Thành phố Thủ Đức, TP. Hồ Chí Minh"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_030.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước mắm Nam Ngư 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "168,000"
  },
  {
    "image_name": "einvoice_vnpt_val_033.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH\", \"box\": [60, 64, 371, 90]}"
  },
  {
    "image_name": "supermarket_winmart_train_163.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Custas Orion 6P là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "einvoice_viettel_train_051.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 224 Bùi Thị Xuân, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "einvoice_vnpt_train_244.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT\", \"box\": [53, 62, 376, 93]}"
  },
  {
    "image_name": "einvoice_viettel_train_204.png",
    "template": "einvoice_viettel",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "17/06/2026 12:12"
  },
  {
    "image_name": "einvoice_vnpt_train_217.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "580,000"
  },
  {
    "image_name": "cafe_phuclong_train_245.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà phê Latte là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "image_name": "convenience_gs25_val_042.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "image_name": "cafe_phuclong_train_032.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "PHÚC LONG NGUYỄN VĂN CỪ"
  },
  {
    "image_name": "supermarket_lotte_val_058.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OP FOOD NGUYỄN ĐÌNH CHIỂU"
  },
  {
    "image_name": "supermarket_lotte_train_158.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 09:35:00"
  },
  {
    "image_name": "einvoice_vnpt_train_176.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "1,700,000"
  },
  {
    "image_name": "convenience_gs25_train_138.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 23 Cách Mạng Tháng Tám, Quận 1, Đồng Nai"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_210.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trứng gà tươi hộp 10 quả là bao nhiêu?",
    "ground_truth": "128,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_179.png",
    "template": "supermarket_bachhoaxanh",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH QUẬN 12"
  },
  {
    "image_name": "supermarket_winmart_train_120.png",
    "template": "supermarket_winmart",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "01/06/2026 10:35"
  },
  {
    "image_name": "receipt_c45_bb_train_167.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2.090.000 VND"
  },
  {
    "image_name": "supermarket_lotte_train_214.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "96,000.00"
  },
  {
    "image_name": "convenience_7eleven_val_048.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "171,600"
  },
  {
    "image_name": "cafe_phuclong_train_240.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 14 Trần Hưng Đạo, Quận Đống Đa, Hà Nội"
  },
  {
    "image_name": "convenience_circlek_train_187.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"262,440đ\", \"box\": [328, 497, 387, 517]}"
  },
  {
    "image_name": "minimart_anan_train_265.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ANAN CONVENIENCE STORE\", \"box\": [123, 15, 288, 36]}"
  },
  {
    "image_name": "supermarket_lotte_train_141.png",
    "template": "supermarket_lotte",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 17:09:00"
  },
  {
    "image_name": "cafe_starbucks_train_057.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Green Tea Latte M là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_highlands_train_240.png",
    "template": "cafe_highlands",
    "field": "ITEM_QTY",
    "question": "Số lượng của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_winmart_val_044.png",
    "template": "supermarket_winmart",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"VINCOMMERCE\", \"box\": [166, 33, 244, 55]}"
  },
  {
    "image_name": "minimart_anan_train_090.png",
    "template": "minimart_anan",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ANAN CONVENIENCE STORE\", \"address\": \"Số 230 Lý Thường Kiệt, Quận Sơn Trà, Đà Nẵng\", \"timestamp\": \"31/05/2026 20:44\", \"total_cost\": \"771.100\", \"items\": [{\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"4\", \"amount\": \"32.000\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525.000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"1\", \"amount\": \"16.000\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"4\", \"amount\": \"128.000\"}]}"
  },
  {
    "image_name": "convenience_circlek_train_039.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "image_name": "einvoice_viettel_train_049.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Giấy Double A A4 70gsm, Dịch vụ Bảo trì Hệ thống mạng, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "convenience_gs25_train_186.png",
    "template": "convenience_gs25",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 TÔN THẤT THUYẾT"
  },
  {
    "image_name": "supermarket_winmart_train_210.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Sữa tươi TH True Milk 1L là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_starbucks_val_012.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caffe Americano L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "einvoice_vnpt_train_163.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "450,000"
  },
  {
    "image_name": "cafe_highlands_train_021.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "140,000"
  },
  {
    "image_name": "convenience_gs25_train_136.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà sữa Kirin Latte 345ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "image_name": "convenience_circlek_val_029.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 147 Nguyễn Huệ, Quận 1, Bình Dương\", \"timestamp\": \"09/06/2026 16:54\", \"total_cost\": \"58,320đ\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"3\", \"amount\": \"18,000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"4\", \"amount\": \"36,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_227.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Sandwich xá xíu phô mai là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "image_name": "minimart_anan_val_032.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước uống Aquafina 500ml, Kẹo cao su Cool Air hũ, Xúc xích tiệt trùng Ponnie, Kem đánh răng Colgate, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "minimart_anan_train_052.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_lotte_train_204.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hộp dâu tây Đà Lạt 250g là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "cafe_highlands_train_083.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cà Phê Đen Đá Size M là bao nhiêu?",
    "ground_truth": "87,000"
  },
  {
    "image_name": "einvoice_vnpt_train_088.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"17,413,000đ\", \"box\": [671, 846, 753, 867]}"
  },
  {
    "image_name": "cafe_phuclong_train_229.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "restaurant_jollibee_train_217.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ý Sốt Bò Bằm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "receipt_c45_bb_train_013.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [88, 29, 361, 46]}"
  },
  {
    "image_name": "einvoice_viettel_val_061.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_7eleven_train_185.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "99,360"
  },
  {
    "image_name": "convenience_circlek_train_237.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "125,400đ"
  },
  {
    "image_name": "restaurant_jollibee_train_102.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước ngọt 7Up L, Khoai Tây Lắc Phô Mai, Mì Ý Sốt Bò Bằm, Bánh Khoai Môn, Gà Giòn Vui Vẻ 1 miếng"
  },
  {
    "image_name": "convenience_7eleven_train_144.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "164,160"
  },
  {
    "image_name": "convenience_gs25_train_161.png",
    "template": "convenience_gs25",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"GS25 THỦ ĐỨC\", \"address\": \"Số 196 Cách Mạng Tháng Tám, Quận 1, Đồng Nai\", \"timestamp\": \"21/05/2026 19:20\", \"total_cost\": \"280,800đ\", \"items\": [{\"name\": \"Mì trộn Indomie đặc biệt\", \"qty\": \"3\", \"amount\": \"36,000\"}, {\"name\": \"Cơm nắm cá hồi sốt Mayo\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Trà sữa Kirin Latte 345ml\", \"qty\": \"3\", \"amount\": \"48,000\"}, {\"name\": \"Bánh giò thịt băm trứng cút\", \"qty\": \"3\", \"amount\": \"42,000\"}, {\"name\": \"Cafe sữa đá GS25\", \"qty\": \"4\", \"amount\": \"88,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_train_073.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bàn phím cơ Dareu EK87, Dịch vụ Lắp đặt camera giám sát, Cáp mạng Cat6 UTP 305m"
  },
  {
    "image_name": "supermarket_lotte_train_082.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "image_name": "restaurant_jollibee_train_230.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "einvoice_viettel_train_229.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN CÔNG NGHỆ SAO NAM\", \"address\": \"Số 78 Lê Lợi, Quận 1, Hải Phòng\", \"timestamp\": \"10/06/2026 16:23\", \"total_cost\": \"19,000,300đ\", \"items\": [{\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"2\", \"amount\": \"8,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"2\", \"amount\": \"7,700,000\"}, {\"name\": \"Giấy Double A A4 70gsm\", \"qty\": \"1\", \"amount\": \"65,000\"}, {\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"3\", \"amount\": \"4,500,000\"}, {\"name\": \"Mực in Canon Cartridge\", \"qty\": \"4\", \"amount\": \"5,000,000\"}]}"
  },
  {
    "image_name": "cafe_starbucks_train_059.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Butter Croissant trên hóa đơn là bao nhiêu?",
    "ground_truth": "120,000"
  },
  {
    "image_name": "minimart_anan_val_001.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CỬA HÀNG TIỆN LỢI AN AN\", \"box\": [123, 28, 288, 57]}"
  },
  {
    "image_name": "restaurant_jollibee_train_169.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "152,000"
  },
  {
    "image_name": "cafe_phuclong_train_168.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Đào Phúc Long L là bao nhiêu?",
    "ground_truth": "55,000"
  },
  {
    "image_name": "einvoice_viettel_val_056.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mực in Canon Cartridge là bao nhiêu?",
    "ground_truth": "1,250,000"
  },
  {
    "image_name": "einvoice_vnpt_train_101.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "3,400,000"
  },
  {
    "image_name": "cafe_phuclong_train_110.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cà phê Latte là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_7eleven_val_061.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "einvoice_viettel_train_116.png",
    "template": "einvoice_viettel",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT\", \"box\": [46, 60, 359, 103]}"
  },
  {
    "image_name": "supermarket_winmart_train_076.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh Custas Orion 6P, Dầu ăn Simply 1L, Mì tôm Hảo Hảo chua cay, Coca Cola Lon 320ml, Sữa tươi TH True Milk 1L"
  },
  {
    "image_name": "restaurant_jollibee_train_022.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"210,100đ\", \"box\": [334, 393, 393, 415]}"
  },
  {
    "image_name": "convenience_circlek_train_062.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"Circle K Phạm Ngũ Lão\", \"box\": [139, 39, 267, 61]}"
  },
  {
    "image_name": "convenience_7eleven_train_058.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "130,680"
  },
  {
    "image_name": "cafe_highlands_train_127.png",
    "template": "cafe_highlands",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "469,908"
  },
  {
    "image_name": "convenience_gs25_train_053.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 THỦ ĐỨC\", \"box\": [160, 15, 250, 36]}"
  },
  {
    "image_name": "restaurant_jollibee_val_067.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "418,000đ"
  },
  {
    "image_name": "convenience_gs25_train_122.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì trộn Indomie đặc biệt là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "convenience_7eleven_train_234.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "cafe_starbucks_train_189.png",
    "template": "cafe_starbucks",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "334,400"
  },
  {
    "image_name": "convenience_circlek_train_243.png",
    "template": "convenience_circlek",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 21:08"
  },
  {
    "image_name": "einvoice_vnpt_train_003.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT"
  },
  {
    "image_name": "convenience_7eleven_val_050.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"201,960\", \"box\": [311, 294, 363, 317]}"
  },
  {
    "image_name": "restaurant_jollibee_train_074.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_winmart_train_189.png",
    "template": "supermarket_winmart",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Coca Cola Lon 320ml, Sữa tươi TH True Milk 1L, Bánh Custas Orion 6P, Mì tôm Hảo Hảo chua cay, Nước rửa chén Sunlight 750ml, Khăn giấy Pulppy 100 tờ"
  },
  {
    "image_name": "einvoice_viettel_train_230.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 357 Cách Mạng Tháng Tám, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_train_021.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "2,500,000"
  },
  {
    "image_name": "cafe_phuclong_val_018.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "03/06/2026 20:13"
  },
  {
    "image_name": "convenience_gs25_train_199.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "cafe_starbucks_train_087.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 315 Nguyễn Huệ, Quận 1, Đồng Nai\", \"timestamp\": \"26/05/2026\", \"total_cost\": \"830,500\", \"items\": [{\"name\": \"Chocolate Muffin\", \"qty\": \"2\", \"amount\": \"90,000\"}, {\"name\": \"Green Tea Latte M\", \"qty\": \"1\", \"amount\": \"75,000\"}, {\"name\": \"Cold Brew Coffee M\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Caffe Americano L\", \"qty\": \"1\", \"amount\": \"60,000\"}, {\"name\": \"Caramel Macchiato L\", \"qty\": \"3\", \"amount\": \"255,000\"}]}"
  },
  {
    "image_name": "minimart_anan_val_022.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 13 Lê Lợi, Quận Thanh Xuân, Hà Nội"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_258.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "255,200"
  },
  {
    "image_name": "convenience_gs25_train_007.png",
    "template": "convenience_gs25",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 73 Lê Lợi, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "convenience_7eleven_train_260.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "image_name": "minimart_anan_val_018.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ANAN CONVENIENCE STORE\", \"box\": [122, 24, 280, 45]}"
  },
  {
    "image_name": "convenience_7eleven_train_068.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "72,000"
  },
  {
    "image_name": "minimart_anan_train_144.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Xúc xích tiệt trùng Ponnie trên hóa đơn là bao nhiêu?",
    "ground_truth": "8.000"
  },
  {
    "image_name": "supermarket_lotte_train_181.png",
    "template": "supermarket_lotte",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Khăn ướt Bobby 80 tờ, Hộp dâu tây Đà Lạt 250g, Kem đánh răng Colgate, Nước cam ép Twister 1L, Bia Heineken Lon 330ml"
  },
  {
    "image_name": "cafe_starbucks_val_010.png",
    "template": "cafe_starbucks",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "85,000"
  },
  {
    "image_name": "cafe_phuclong_train_174.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG TEA & COFFEE"
  },
  {
    "image_name": "supermarket_lotte_train_099.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước cam ép Twister 1L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_val_067.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Mì Ý Sốt Bò Bằm trên hóa đơn là bao nhiêu?",
    "ground_truth": "114,000"
  },
  {
    "image_name": "receipt_c45_bb_val_047.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "supermarket_lotte_train_166.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "509,760.00"
  },
  {
    "image_name": "convenience_circlek_train_116.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước uống Aquafina 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,000"
  },
  {
    "image_name": "supermarket_lotte_val_006.png",
    "template": "supermarket_lotte",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 428 Nguyễn Trãi, Quận Cầu Giấy, Hà Nội"
  },
  {
    "image_name": "convenience_gs25_train_018.png",
    "template": "convenience_gs25",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"GS25 TÔN THẤT THUYẾT\", \"box\": [125, 28, 272, 50]}"
  },
  {
    "image_name": "restaurant_jollibee_val_050.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "cafe_phuclong_val_059.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"485,925\", \"box\": [311, 450, 361, 471]}"
  },
  {
    "image_name": "supermarket_winmart_train_221.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VM+QNH HẠ LONG"
  },
  {
    "image_name": "einvoice_vnpt_train_017.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_val_060.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "266,200"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_055.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 22:27"
  },
  {
    "image_name": "convenience_7eleven_train_152.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"272,160\", \"box\": [313, 234, 365, 258]}"
  },
  {
    "image_name": "restaurant_jollibee_val_028.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước ngọt 7Up L trên hóa đơn là bao nhiêu?",
    "ground_truth": "68,000"
  },
  {
    "image_name": "cafe_phuclong_train_174.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "convenience_7eleven_train_101.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước khoáng Dasani 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "einvoice_viettel_train_102.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_025.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước mắm Nam Ngư 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "84,000"
  },
  {
    "image_name": "einvoice_vnpt_train_236.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "25/05/2026 22:45"
  },
  {
    "image_name": "supermarket_lotte_val_044.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "image_name": "restaurant_jollibee_train_155.png",
    "template": "restaurant_jollibee",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "image_name": "supermarket_lotte_train_079.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "223,300.00"
  },
  {
    "image_name": "einvoice_vnpt_train_258.png",
    "template": "einvoice_vnpt",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bộ phát Wifi TP-Link Archer, Bàn phím cơ Dareu EK87"
  },
  {
    "image_name": "cafe_starbucks_train_119.png",
    "template": "cafe_starbucks",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "29/05/2026"
  },
  {
    "image_name": "supermarket_winmart_val_032.png",
    "template": "supermarket_winmart",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VM+QNH HẠ LONG"
  },
  {
    "image_name": "einvoice_viettel_train_204.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Dịch vụ Bảo trì Hệ thống mạng, Giấy Double A A4 70gsm, Mực in Canon Cartridge, Bút bi Thiên Long FO-03, Máy in HP LaserJet Pro"
  },
  {
    "image_name": "supermarket_lotte_train_055.png",
    "template": "supermarket_lotte",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OP FOOD NGUYỄN ĐÌNH CHIỂU"
  },
  {
    "image_name": "supermarket_lotte_train_105.png",
    "template": "supermarket_lotte",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"729,000.00\", \"box\": [315, 648, 388, 668]}"
  },
  {
    "image_name": "einvoice_viettel_val_057.png",
    "template": "einvoice_viettel",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 313 Bùi Thị Xuân, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "image_name": "minimart_anan_train_222.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "64.000"
  },
  {
    "image_name": "cafe_starbucks_train_127.png",
    "template": "cafe_starbucks",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"STARBUCKS NEW WORLD\", \"address\": \"Số 50 Nguyễn Huệ, Quận 1, Cần Thơ\", \"timestamp\": \"31/05/2026\", \"total_cost\": \"370,975\", \"items\": [{\"name\": \"Cold Brew Coffee M\", \"qty\": \"3\", \"amount\": \"195,000\"}, {\"name\": \"Butter Croissant\", \"qty\": \"4\", \"amount\": \"160,000\"}]}"
  },
  {
    "image_name": "restaurant_jollibee_train_080.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "21/05/2026 22:48"
  },
  {
    "image_name": "supermarket_lotte_val_020.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CO.OPMART CỐNG QUỲNH\", \"address\": \"Số 8 Nguyễn Huệ, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"18/06/2026 18:01:00\", \"total_cost\": \"700,150.00\", \"items\": [{\"name\": \"Nước cam ép Twister 1L\", \"qty\": \"1\", \"amount\": \"28,000.00\"}, {\"name\": \"Bia Heineken Lon 330ml\", \"qty\": \"1\", \"amount\": \"19,500.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"2\", \"amount\": \"64,000.00\"}, {\"name\": \"Bột giặt Ariel 3.2kg\", \"qty\": \"3\", \"amount\": \"525,000.00\"}]}"
  },
  {
    "image_name": "convenience_circlek_val_058.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước uống Aquafina 500ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "18,000"
  },
  {
    "image_name": "cafe_starbucks_train_122.png",
    "template": "cafe_starbucks",
    "field": "ITEM_QTY",
    "question": "Số lượng của Caramel Macchiato L là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "convenience_circlek_train_103.png",
    "template": "convenience_circlek",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "81,400đ"
  },
  {
    "image_name": "convenience_gs25_train_090.png",
    "template": "convenience_gs25",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bánh giò thịt băm trứng cút trên hóa đơn là bao nhiêu?",
    "ground_truth": "42,000"
  },
  {
    "image_name": "minimart_anan_train_041.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 415 Bùi Thị Xuân, Quận 10, TP. Hồ Chí Minh"
  },
  {
    "image_name": "restaurant_jollibee_train_232.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"JOLLIBEE VINCOM\", \"address\": \"Số 407 Cách Mạng Tháng Tám, Quận 1, Đồng Nai\", \"timestamp\": \"31/05/2026 13:31\", \"total_cost\": \"190,300đ\", \"items\": [{\"name\": \"Bánh Khoai Môn\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"1\", \"amount\": \"38,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"3\", \"amount\": \"105,000\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_233.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Phin Sữa Đá Size L là bao nhiêu?",
    "ground_truth": "78,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_246.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"address\": \"Số 292 Hai Bà Trưng, Quận 1, Cần Thơ\", \"timestamp\": \"02/06/2026 14:11\", \"total_cost\": \"481,800\", \"items\": [{\"name\": \"Thịt đùi heo 500g\", \"qty\": \"3\", \"amount\": \"204,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"3\", \"amount\": \"78,000\"}, {\"name\": \"Thịt ba rọi heo 500g\", \"qty\": \"2\", \"amount\": \"150,000\"}, {\"name\": \"Hành lá 100g\", \"qty\": \"2\", \"amount\": \"6,000\"}]}"
  },
  {
    "image_name": "supermarket_lotte_train_025.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước cam ép Twister 1L là bao nhiêu?",
    "ground_truth": "56,000.00"
  },
  {
    "image_name": "cafe_highlands_train_168.png",
    "template": "cafe_highlands",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Mì Que Gà Xé là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "supermarket_lotte_train_029.png",
    "template": "supermarket_lotte",
    "field": "ITEM_QTY",
    "question": "Số lượng của Khăn ướt Bobby 80 tờ là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_val_026.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"JOLLIBEE COOPMART\", \"address\": \"Số 133 Nguyễn Trãi, Quận 1, Cần Thơ\", \"timestamp\": \"24/05/2026 19:18\", \"total_cost\": \"195,480đ\", \"items\": [{\"name\": \"Bánh Khoai Môn\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"3\", \"amount\": \"51,000\"}, {\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"4\", \"amount\": \"100,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_163.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_train_056.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 07:54"
  },
  {
    "image_name": "supermarket_lotte_train_091.png",
    "template": "supermarket_lotte",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"SAIGON CO.OP\", \"address\": \"Số 370 Hai Bà Trưng, Quận 10, TP. Hồ Chí Minh\", \"timestamp\": \"04/06/2026 20:17:00\", \"total_cost\": \"129,600.00\", \"items\": [{\"name\": \"Bánh quy Cosy Kinh Đô\", \"qty\": \"4\", \"amount\": \"88,000.00\"}, {\"name\": \"Kem đánh răng Colgate\", \"qty\": \"1\", \"amount\": \"32,000.00\"}]}"
  },
  {
    "image_name": "cafe_highlands_train_046.png",
    "template": "cafe_highlands",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Tiramisu, Freeze Trà Xanh Size M"
  },
  {
    "image_name": "convenience_circlek_train_147.png",
    "template": "convenience_circlek",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"Circle K Lê Lợi\", \"box\": [160, 21, 255, 43]}"
  },
  {
    "image_name": "convenience_gs25_train_051.png",
    "template": "convenience_gs25",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "116,600đ"
  },
  {
    "image_name": "convenience_circlek_val_063.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "minimart_anan_train_145.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kem đánh răng Colgate trên hóa đơn là bao nhiêu?",
    "ground_truth": "96.000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_191.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "supermarket_lotte_train_228.png",
    "template": "supermarket_lotte",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khăn ướt Bobby 80 tờ trên hóa đơn là bao nhiêu?",
    "ground_truth": "108,000.00"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_017.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước mắm Nam Ngư 750ml là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "receipt_c45_bb_train_056.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 84 Lý Thường Kiệt, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "supermarket_winmart_train_230.png",
    "template": "supermarket_winmart",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "30.000"
  },
  {
    "image_name": "convenience_circlek_val_056.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 59 Lý Thường Kiệt, Quận Cầu Giấy, Hà Nội\", \"timestamp\": \"18/06/2026 09:49\", \"total_cost\": \"134,200đ\", \"items\": [{\"name\": \"Băng cá nhân Urgo hộp 10 miếng\", \"qty\": \"4\", \"amount\": \"60,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"4\", \"amount\": \"32,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"3\", \"amount\": \"30,000\"}]}"
  },
  {
    "image_name": "einvoice_viettel_train_258.png",
    "template": "einvoice_viettel",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN\", \"address\": \"Số 78 Lê Lợi, Quận 3, TP. Hồ Chí Minh\", \"timestamp\": \"28/05/2026 22:07\", \"total_cost\": \"5,786,640đ\", \"items\": [{\"name\": \"Dịch vụ Bảo trì Hệ thống mạng\", \"qty\": \"1\", \"amount\": \"1,500,000\"}, {\"name\": \"Máy in HP LaserJet Pro\", \"qty\": \"1\", \"amount\": \"3,850,000\"}, {\"name\": \"Bút bi Thiên Long FO-03\", \"qty\": \"2\", \"amount\": \"8,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_097.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "receipt_c45_bb_train_256.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 134 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 12 tháng 06 năm 2026\", \"total_cost\": \"440.000 VND\"}"
  },
  {
    "image_name": "einvoice_viettel_train_068.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Máy in HP LaserJet Pro, Bút bi Thiên Long FO-03, Dịch vụ Bảo trì Hệ thống mạng, Mực in Canon Cartridge"
  },
  {
    "image_name": "convenience_circlek_val_066.png",
    "template": "convenience_circlek",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "supermarket_lotte_train_011.png",
    "template": "supermarket_lotte",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "1,150,740.00"
  },
  {
    "image_name": "cafe_phuclong_train_038.png",
    "template": "cafe_phuclong",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 20:10"
  },
  {
    "image_name": "einvoice_vnpt_train_048.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_QTY",
    "question": "Số lượng của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "4"
  },
  {
    "image_name": "restaurant_jollibee_train_036.png",
    "template": "restaurant_jollibee",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"JOLLIBEE VINCOM\", \"address\": \"Số 237 Cách Mạng Tháng Tám, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"05/06/2026 18:54\", \"total_cost\": \"239,800đ\", \"items\": [{\"name\": \"Khoai Tây Lắc Phô Mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Mì Ý Sốt Bò Bằm\", \"qty\": \"2\", \"amount\": \"76,000\"}, {\"name\": \"Bánh Khoai Môn\", \"qty\": \"2\", \"amount\": \"30,000\"}, {\"name\": \"Nước ngọt 7Up L\", \"qty\": \"1\", \"amount\": \"17,000\"}, {\"name\": \"Gà Giòn Vui Vẻ 1 miếng\", \"qty\": \"2\", \"amount\": \"70,000\"}]}"
  },
  {
    "image_name": "convenience_gs25_train_237.png",
    "template": "convenience_gs25",
    "field": "ITEM_QTY",
    "question": "Số lượng của Cafe sữa đá GS25 là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "receipt_c45_bb_train_216.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 228 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh\", \"timestamp\": \"Ngày 17 tháng 06 năm 2026\", \"total_cost\": \"55.000 VND\"}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_157.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH QUẬN 12\", \"address\": \"Số 278 Lý Thường Kiệt, Quận Thanh Xuân, Hà Nội\", \"timestamp\": \"19/06/2026 08:36\", \"total_cost\": \"160,600\", \"items\": [{\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"1\", \"amount\": \"32,000\"}, {\"name\": \"Thịt đùi heo 500g\", \"qty\": \"1\", \"amount\": \"68,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Hành lá 100g\", \"qty\": \"4\", \"amount\": \"12,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_019.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "76,000"
  },
  {
    "image_name": "minimart_anan_train_219.png",
    "template": "minimart_anan",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"MINIMART ANAN\", \"box\": [162, 35, 254, 57]}"
  },
  {
    "image_name": "einvoice_viettel_train_006.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Máy in HP LaserJet Pro là bao nhiêu?",
    "ground_truth": "11,550,000"
  },
  {
    "image_name": "convenience_circlek_val_020.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 442 Lê Lợi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "einvoice_viettel_train_025.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "260,000"
  },
  {
    "image_name": "restaurant_jollibee_val_029.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước ngọt 7Up L, Khoai Tây Lắc Phô Mai, Gà Giòn Vui Vẻ 1 miếng, Bánh Khoai Môn"
  },
  {
    "image_name": "convenience_circlek_train_056.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "supermarket_winmart_train_259.png",
    "template": "supermarket_winmart",
    "field": "ITEM_QTY",
    "question": "Số lượng của Nước rửa chén Sunlight 750ml là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "cafe_phuclong_train_127.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "restaurant_jollibee_train_264.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_QTY",
    "question": "Số lượng của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "einvoice_viettel_train_088.png",
    "template": "einvoice_viettel",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mực in Canon Cartridge, Giấy Double A A4 70gsm, Máy in HP LaserJet Pro, Dịch vụ Bảo trì Hệ thống mạng, Bút bi Thiên Long FO-03"
  },
  {
    "image_name": "einvoice_viettel_val_066.png",
    "template": "einvoice_viettel",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Bút bi Thiên Long FO-03 trên hóa đơn là bao nhiêu?",
    "ground_truth": "4,000"
  },
  {
    "image_name": "einvoice_vnpt_val_026.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Cáp mạng Cat6 UTP 305m là bao nhiêu?",
    "ground_truth": "7,400,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_083.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 183 Lê Lợi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "einvoice_vnpt_train_130.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "11/06/2026 09:19"
  },
  {
    "image_name": "cafe_phuclong_train_167.png",
    "template": "cafe_phuclong",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 237 Lý Thường Kiệt, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "convenience_7eleven_train_077.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "345,600"
  },
  {
    "image_name": "einvoice_vnpt_train_030.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TỔNG CÔNG TY DỊCH VỤ VIỄN THÔNG VNPT"
  },
  {
    "image_name": "convenience_circlek_train_119.png",
    "template": "convenience_circlek",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Xúc xích tiệt trùng Ponnie, Trà xanh Không Độ 500ml"
  },
  {
    "image_name": "receipt_c45_bb_train_044.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"216.000 VND\", \"box\": [122, 260, 221, 275]}"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_022.png",
    "template": "supermarket_bachhoaxanh",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"BÁCH HÓA XANH LÊ ĐỨC THỌ\", \"address\": \"Số 252 Cách Mạng Tháng Tám, Quận Hai Bà Trưng, Hà Nội\", \"timestamp\": \"18/06/2026 16:45\", \"total_cost\": \"693,360\", \"items\": [{\"name\": \"Thịt đùi heo 500g\", \"qty\": \"3\", \"amount\": \"204,000\"}, {\"name\": \"Trứng gà tươi hộp 10 quả\", \"qty\": \"4\", \"amount\": \"128,000\"}, {\"name\": \"Rau muống nước 500g\", \"qty\": \"1\", \"amount\": \"8,000\"}, {\"name\": \"Đường cát trắng Biên Hòa 1kg\", \"qty\": \"1\", \"amount\": \"26,000\"}, {\"name\": \"Nước mắm Nam Ngư 750ml\", \"qty\": \"3\", \"amount\": \"126,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_val_033.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cà phê Latte trên hóa đơn là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "image_name": "cafe_phuclong_train_112.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Đào Phúc Long L, Trà sữa Phúc Long L, Cà phê Latte, Bánh Croissant Bơ Pháp, Trà Lài Đác Thơm L, Hồng Trà Sữa M"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_216.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Nước mắm Nam Ngư 750ml trên hóa đơn là bao nhiêu?",
    "ground_truth": "126,000"
  },
  {
    "image_name": "einvoice_viettel_train_166.png",
    "template": "einvoice_viettel",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "image_name": "convenience_circlek_train_107.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 417 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "minimart_anan_val_014.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "18.000"
  },
  {
    "image_name": "einvoice_viettel_val_067.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Dịch vụ Bảo trì Hệ thống mạng là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "convenience_circlek_train_205.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo cao su Cool Air hũ trên hóa đơn là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "image_name": "convenience_circlek_train_120.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo cao su Cool Air hũ là bao nhiêu?",
    "ground_truth": "44,000"
  },
  {
    "image_name": "minimart_anan_train_245.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "einvoice_viettel_train_088.png",
    "template": "einvoice_viettel",
    "field": "ITEM_QTY",
    "question": "Số lượng của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "cafe_phuclong_train_168.png",
    "template": "cafe_phuclong",
    "field": "ITEM_QTY",
    "question": "Số lượng của Bánh Croissant Bơ Pháp là bao nhiêu?",
    "ground_truth": "1"
  },
  {
    "image_name": "restaurant_jollibee_train_189.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE VINCOM\", \"box\": [146, 23, 253, 47]}"
  },
  {
    "image_name": "convenience_circlek_train_225.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CIRCLE K\", \"address\": \"Số 111 Nguyễn Trãi, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"02/06/2026 21:59\", \"total_cost\": \"225,720đ\", \"items\": [{\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"4\", \"amount\": \"24,000\"}, {\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"3\", \"amount\": \"27,000\"}, {\"name\": \"Bánh bao trứng muối\", \"qty\": \"4\", \"amount\": \"64,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"4\", \"amount\": \"40,000\"}, {\"name\": \"Xúc xích tiệt trùng Ponnie\", \"qty\": \"4\", \"amount\": \"32,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_110.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bột giặt Ariel 3.2kg là bao nhiêu?",
    "ground_truth": "175.000"
  },
  {
    "image_name": "cafe_phuclong_val_059.png",
    "template": "cafe_phuclong",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "100,000"
  },
  {
    "image_name": "convenience_circlek_train_085.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"Circle K Phạm Ngũ Lão\", \"address\": \"Số 399 Lý Thường Kiệt, Quận 1, Bình Dương\", \"timestamp\": \"29/05/2026 18:34\", \"total_cost\": \"155,100đ\", \"items\": [{\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"3\", \"amount\": \"27,000\"}, {\"name\": \"Kẹo cao su Cool Air hũ\", \"qty\": \"3\", \"amount\": \"66,000\"}, {\"name\": \"Trà xanh Không Độ 500ml\", \"qty\": \"3\", \"amount\": \"30,000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"3\", \"amount\": \"18,000\"}]}"
  },
  {
    "image_name": "cafe_phuclong_train_128.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"328,320\", \"box\": [307, 368, 358, 391]}"
  },
  {
    "image_name": "einvoice_vnpt_train_188.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dịch vụ Lắp đặt camera giám sát là bao nhiêu?",
    "ground_truth": "2,500,000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_val_046.png",
    "template": "supermarket_bachhoaxanh",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"BÁCH HÓA XANH QUẬN 12\", \"box\": [111, 15, 267, 37]}"
  },
  {
    "image_name": "restaurant_jollibee_train_201.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "108,000đ"
  },
  {
    "image_name": "minimart_anan_train_047.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 60 Bùi Thị Xuân, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_train_066.png",
    "template": "einvoice_vnpt",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "10,724,400đ"
  },
  {
    "image_name": "restaurant_jollibee_train_143.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "213,400đ"
  },
  {
    "image_name": "receipt_c45_bb_train_237.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "einvoice_vnpt_train_187.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"10,703,000đ\", \"box\": [668, 803, 750, 825]}"
  },
  {
    "image_name": "receipt_c45_bb_train_148.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "2.052.000 VND"
  },
  {
    "image_name": "cafe_phuclong_train_001.png",
    "template": "cafe_phuclong",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "227,700"
  },
  {
    "image_name": "convenience_circlek_train_197.png",
    "template": "convenience_circlek",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"CIRCLE K\", \"address\": \"Số 283 Nguyễn Trãi, Quận 1, Đồng Nai\", \"timestamp\": \"29/05/2026 17:41\", \"total_cost\": \"58,320đ\", \"items\": [{\"name\": \"Mì Ly ăn liền Modern\", \"qty\": \"4\", \"amount\": \"36,000\"}, {\"name\": \"Nước uống Aquafina 500ml\", \"qty\": \"3\", \"amount\": \"18,000\"}]}"
  },
  {
    "image_name": "minimart_anan_train_051.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "36.000"
  },
  {
    "image_name": "restaurant_jollibee_val_039.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 338 Nguyễn Trãi, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_7eleven_val_002.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "minimart_anan_train_028.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "30.000"
  },
  {
    "image_name": "convenience_circlek_train_133.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 45 Nguyễn Huệ, Quận Ba Đình, Hà Nội"
  },
  {
    "image_name": "convenience_circlek_val_019.png",
    "template": "convenience_circlek",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà xanh Không Độ 500ml là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "convenience_circlek_train_002.png",
    "template": "convenience_circlek",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 445 Trần Hưng Đạo, Quận 1, Bình Dương"
  },
  {
    "image_name": "minimart_anan_train_266.png",
    "template": "minimart_anan",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "32.000"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_194.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 23 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_014.png",
    "template": "supermarket_bachhoaxanh",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "31/05/2026 21:25"
  },
  {
    "image_name": "cafe_phuclong_train_187.png",
    "template": "cafe_phuclong",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Trà Lài Đác Thơm L, Trà Đào Phúc Long L, Hồng Trà Sữa M, Cà phê Latte"
  },
  {
    "image_name": "minimart_anan_train_054.png",
    "template": "minimart_anan",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"437.800\", \"box\": [336, 406, 386, 427]}"
  },
  {
    "image_name": "einvoice_vnpt_train_102.png",
    "template": "einvoice_vnpt",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"CÔNG TY CỔ PHẦN THIẾT BỊ VĂN PHÒNG HOÀNG MINH\", \"address\": \"Số 322 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"15/06/2026 18:39\", \"total_cost\": \"13,230,000đ\", \"items\": [{\"name\": \"Dịch vụ Lắp đặt camera giám sát\", \"qty\": \"2\", \"amount\": \"5,000,000\"}, {\"name\": \"Bộ phát Wifi TP-Link Archer\", \"qty\": \"2\", \"amount\": \"1,700,000\"}, {\"name\": \"Cáp mạng Cat6 UTP 305m\", \"qty\": \"3\", \"amount\": \"5,550,000\"}]}"
  },
  {
    "image_name": "receipt_c45_bb_train_177.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"box\": [94, 33, 334, 54]}"
  },
  {
    "image_name": "cafe_phuclong_train_159.png",
    "template": "cafe_phuclong",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"334,400\", \"box\": [308, 374, 361, 395]}"
  },
  {
    "image_name": "cafe_phuclong_train_029.png",
    "template": "cafe_phuclong",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LANDMARK"
  },
  {
    "image_name": "einvoice_vnpt_train_141.png",
    "template": "einvoice_vnpt",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "image_name": "supermarket_bachhoaxanh_train_124.png",
    "template": "supermarket_bachhoaxanh",
    "field": "ITEM_QTY",
    "question": "Số lượng của Hành lá 100g là bao nhiêu?",
    "ground_truth": "2"
  },
  {
    "image_name": "einvoice_vnpt_train_073.png",
    "template": "einvoice_vnpt",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"18,252,000đ\", \"box\": [663, 740, 742, 761]}"
  },
  {
    "image_name": "einvoice_vnpt_train_253.png",
    "template": "einvoice_vnpt",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "04/06/2026 13:42"
  },
  {
    "image_name": "einvoice_vnpt_train_194.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "5,550,000"
  },
  {
    "image_name": "restaurant_jollibee_train_017.png",
    "template": "restaurant_jollibee",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"JOLLIBEE PASTEUR\", \"box\": [146, 20, 265, 43]}"
  },
  {
    "image_name": "einvoice_vnpt_train_245.png",
    "template": "einvoice_vnpt",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 237 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "convenience_7eleven_train_073.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-ELEVEN BÙI VIỆN\", \"timestamp\": \"27/05/2026 19:15\", \"total_cost\": \"232,100\", \"items\": [{\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"76,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"30,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"75,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"20,000\"}, {\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"10,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_012.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kẹo dẻo Haribo Goldbears, Nước khoáng Dasani 500ml, Sandwich xá xíu phô mai, Snack khoai tây Lays 95g, Trà đào sả Slurpee, Mì xào tương đen"
  },
  {
    "image_name": "convenience_7eleven_train_221.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-Eleven Saigon Trade Center\", \"timestamp\": \"24/05/2026 09:29\", \"total_cost\": \"148,500\", \"items\": [{\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"50,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"40,000\"}]}"
  },
  {
    "image_name": "einvoice_vnpt_val_036.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bàn phím cơ Dareu EK87 là bao nhiêu?",
    "ground_truth": "450,000"
  },
  {
    "image_name": "restaurant_jollibee_val_055.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 266 Lý Thường Kiệt, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "einvoice_vnpt_train_167.png",
    "template": "einvoice_vnpt",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Cáp mạng Cat6 UTP 305m trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,850,000"
  },
  {
    "image_name": "restaurant_jollibee_train_235.png",
    "template": "restaurant_jollibee",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì Ý Sốt Bò Bằm, Khoai Tây Lắc Phô Mai, Nước ngọt 7Up L"
  },
  {
    "image_name": "receipt_c45_bb_train_009.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 23 tháng 05 năm 2026"
  },
  {
    "image_name": "minimart_anan_train_064.png",
    "template": "minimart_anan",
    "field": "ITEM_QTY",
    "question": "Số lượng của Mì Ly ăn liền Modern là bao nhiêu?",
    "ground_truth": "3"
  },
  {
    "image_name": "convenience_7eleven_val_063.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "183,700"
  },
  {
    "image_name": "minimart_anan_train_162.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "04/06/2026 19:36"
  },
  {
    "image_name": "convenience_7eleven_train_244.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "minimart_anan_val_016.png",
    "template": "minimart_anan",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 370 Trần Hưng Đạo, Quận 1, Đồng Nai"
  },
  {
    "image_name": "convenience_7eleven_train_213.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "223,300"
  },
  {
    "image_name": "convenience_7eleven_train_115.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 07:53"
  },
  {
    "image_name": "convenience_7eleven_train_170.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "image_name": "restaurant_jollibee_train_040.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "12/06/2026 18:13"
  },
  {
    "image_name": "receipt_c45_bb_val_033.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "restaurant_jollibee_train_245.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 339 Nguyễn Huệ, Quận 1, Hải Phòng"
  },
  {
    "image_name": "minimart_anan_val_021.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "image_name": "restaurant_jollibee_train_035.png",
    "template": "restaurant_jollibee",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "20/06/2026 22:43"
  },
  {
    "image_name": "receipt_c45_bb_val_033.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 64 Lê Lợi, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"Ngày 30 tháng 05 năm 2026\", \"total_cost\": \"378.000 VND\"}"
  },
  {
    "image_name": "minimart_anan_train_095.png",
    "template": "minimart_anan",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "ANAN CONVENIENCE STORE"
  },
  {
    "image_name": "minimart_anan_train_165.png",
    "template": "minimart_anan",
    "field": "ITEMS_LIST",
    "question": "Hóa đơn này bao gồm những sản phẩm / dịch vụ nào?",
    "ground_truth": "Nước uống Aquafina 500ml, Kẹo cao su Cool Air hũ, Mì Ly ăn liền Modern, Xúc xích tiệt trùng Ponnie, Kem đánh răng Colgate, Bột giặt Ariel 3.2kg"
  },
  {
    "image_name": "minimart_anan_val_067.png",
    "template": "minimart_anan",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 19:14"
  },
  {
    "image_name": "convenience_7eleven_train_206.png",
    "template": "convenience_7eleven",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước khoáng Dasani 500ml, Trà đào sả Slurpee, Mì xào tương đen, Snack khoai tây Lays 95g, Sandwich xá xíu phô mai, Kẹo dẻo Haribo Goldbears"
  },
  {
    "image_name": "restaurant_jollibee_train_214.png",
    "template": "restaurant_jollibee",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 424 Nguyễn Huệ, Quận 1, Cần Thơ"
  },
  {
    "image_name": "restaurant_jollibee_train_261.png",
    "template": "restaurant_jollibee",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "277,200đ"
  },
  {
    "image_name": "restaurant_jollibee_train_186.png",
    "template": "restaurant_jollibee",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Khoai Tây Lắc Phô Mai trên hóa đơn là bao nhiêu?",
    "ground_truth": "25,000"
  },
  {
    "image_name": "convenience_7eleven_train_160.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kẹo dẻo Haribo Goldbears là bao nhiêu?",
    "ground_truth": "30,000"
  },
  {
    "image_name": "receipt_c45_bb_train_219.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 365 Lê Lợi, Quận Hoàn Kiếm, Hà Nội\", \"timestamp\": \"Ngày 22 tháng 05 năm 2026\", \"total_cost\": \"54.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_175.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 14 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_val_018.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "54.000 VND"
  },
  {
    "image_name": "convenience_7eleven_train_249.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "14/06/2026 09:43"
  },
  {
    "image_name": "convenience_7eleven_train_181.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Snack khoai tây Lays 95g trên hóa đơn là bao nhiêu?",
    "ground_truth": "57,000"
  },
  {
    "image_name": "receipt_c45_bb_val_054.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "55.000 VND"
  },
  {
    "image_name": "convenience_7eleven_train_222.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"136,080\", \"box\": [313, 205, 365, 229]}"
  },
  {
    "image_name": "receipt_c45_bb_train_097.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [87, 27, 357, 49]}"
  },
  {
    "image_name": "convenience_7eleven_train_060.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "convenience_7eleven_train_016.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "07/06/2026 10:25"
  },
  {
    "image_name": "convenience_7eleven_train_214.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Kẹo dẻo Haribo Goldbears trên hóa đơn là bao nhiêu?",
    "ground_truth": "60,000"
  },
  {
    "image_name": "convenience_7eleven_train_192.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "279,720"
  },
  {
    "image_name": "receipt_c45_bb_train_091.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 285 Lê Lợi, Quận 1, Đồng Nai\", \"timestamp\": \"Ngày 02 tháng 06 năm 2026\", \"total_cost\": \"1.650.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_val_064.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 26 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_088.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 231 Trần Hưng Đạo, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"Ngày 26 tháng 05 năm 2026\", \"total_cost\": \"1.890.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_197.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 03 tháng 06 năm 2026"
  },
  {
    "image_name": "convenience_7eleven_train_060.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"7-Eleven Nguyễn Du\", \"timestamp\": \"28/05/2026 18:31\", \"total_cost\": \"177,120\", \"items\": [{\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Mì xào tương đen\", \"qty\": \"1\", \"amount\": \"20,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"38,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"25,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_145.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "54,000"
  },
  {
    "image_name": "receipt_c45_bb_train_073.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 334 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "image_name": "convenience_7eleven_train_084.png",
    "template": "convenience_7eleven",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "23/05/2026 07:44"
  },
  {
    "image_name": "receipt_c45_bb_train_115.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 01 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_160.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "convenience_7eleven_train_139.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "image_name": "convenience_7eleven_train_225.png",
    "template": "convenience_7eleven",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "image_name": "receipt_c45_bb_train_037.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 154 Lý Thường Kiệt, Quận 1, Cần Thơ\", \"timestamp\": \"Ngày 24 tháng 05 năm 2026\", \"total_cost\": \"1.998.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_137.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"box\": [88, 31, 334, 48]}"
  },
  {
    "image_name": "convenience_7eleven_train_107.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"75,900\", \"box\": [320, 176, 365, 200]}"
  },
  {
    "image_name": "receipt_c45_bb_val_054.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "convenience_7eleven_train_065.png",
    "template": "convenience_7eleven",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"7-ELEVEN BÙI VIỆN\", \"timestamp\": \"06/06/2026 13:30\", \"total_cost\": \"163,900\", \"items\": [{\"name\": \"Nước khoáng Dasani 500ml\", \"qty\": \"1\", \"amount\": \"5,000\"}, {\"name\": \"Sandwich xá xíu phô mai\", \"qty\": \"1\", \"amount\": \"25,000\"}, {\"name\": \"Kẹo dẻo Haribo Goldbears\", \"qty\": \"1\", \"amount\": \"45,000\"}, {\"name\": \"Snack khoai tây Lays 95g\", \"qty\": \"1\", \"amount\": \"38,000\"}, {\"name\": \"Trà đào sả Slurpee\", \"qty\": \"1\", \"amount\": \"36,000\"}]}"
  },
  {
    "image_name": "convenience_7eleven_train_027.png",
    "template": "convenience_7eleven",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "157,680"
  },
  {
    "image_name": "convenience_7eleven_train_182.png",
    "template": "convenience_7eleven",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"169,560\", \"box\": [313, 234, 365, 258]}"
  },
  {
    "image_name": "receipt_c45_bb_train_025.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "convenience_7eleven_train_034.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Giá / Phí của Trà đào sả Slurpee trên hóa đơn là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "image_name": "convenience_7eleven_train_077.png",
    "template": "convenience_7eleven",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Snack khoai tây Lays 95g là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "image_name": "receipt_c45_bb_train_254.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"216.000 VND\", \"box\": [131, 264, 229, 279]}"
  },
  {
    "image_name": "receipt_c45_bb_train_069.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 10 Lý Thường Kiệt, Quận 10, TP. Hồ Chí Minh"
  },
  {
    "image_name": "receipt_c45_bb_train_215.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "receipt_c45_bb_train_210.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 13 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_val_011.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 396 Nguyễn Trãi, Quận Ngũ Hành Sơn, Đà Nẵng\", \"timestamp\": \"Ngày 14 tháng 06 năm 2026\", \"total_cost\": \"1.836.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_213.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "54.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_212.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "2.052.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_151.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "receipt_c45_bb_train_053.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 9 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "image_name": "receipt_c45_bb_train_142.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 19 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_164.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 21 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_145.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"box\": [85, 27, 346, 42]}"
  },
  {
    "image_name": "receipt_c45_bb_train_092.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 24 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_152.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 26 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_127.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "receipt_c45_bb_val_035.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "receipt_c45_bb_train_104.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 82 Cách Mạng Tháng Tám, Quận Sơn Trà, Đà Nẵng"
  },
  {
    "image_name": "receipt_c45_bb_val_001.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"box\": [98, 35, 342, 52]}"
  },
  {
    "image_name": "receipt_c45_bb_train_112.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 131 Cách Mạng Tháng Tám, Quận 1, Đồng Nai"
  },
  {
    "image_name": "receipt_c45_bb_train_023.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 20 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_050.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"2.052.000 VND\", \"box\": [133, 265, 245, 280]}"
  },
  {
    "image_name": "receipt_c45_bb_train_190.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 31 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_val_060.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 199 Nguyễn Trãi, Quận 1, Đồng Nai\", \"timestamp\": \"Ngày 19 tháng 06 năm 2026\", \"total_cost\": \"2.035.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_val_043.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 16 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_val_054.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [87, 29, 361, 47]}"
  },
  {
    "image_name": "receipt_c45_bb_train_008.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "receipt_c45_bb_train_196.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "2.090.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_195.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 281 Lê Lợi, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"Ngày 04 tháng 06 năm 2026\", \"total_cost\": \"216.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_val_017.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "1.674.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_070.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "receipt_c45_bb_train_144.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 06 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_258.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 14 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_val_004.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "receipt_c45_bb_train_063.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"378.000 VND\", \"box\": [127, 255, 222, 268]}"
  },
  {
    "image_name": "receipt_c45_bb_train_065.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 188 Hai Bà Trưng, Quận 1, Cần Thơ"
  },
  {
    "image_name": "receipt_c45_bb_train_224.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 42 Nguyễn Trãi, Quận Bình Thạnh, TP. Hồ Chí Minh"
  },
  {
    "image_name": "receipt_c45_bb_val_025.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 27 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_185.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "receipt_c45_bb_train_128.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 52 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "image_name": "receipt_c45_bb_train_263.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "image_name": "receipt_c45_bb_val_031.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY\", \"address\": \"Số 440 Lê Lợi, Quận 1, TP. Hồ Chí Minh\", \"timestamp\": \"Ngày 12 tháng 06 năm 2026\", \"total_cost\": \"385.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_217.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 320 Hai Bà Trưng, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "image_name": "receipt_c45_bb_train_001.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Hóa đơn này được phát hành bởi công ty / cửa hàng nào?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "image_name": "receipt_c45_bb_val_019.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 379 Nguyễn Trãi, Quận Đống Đa, Hà Nội\", \"timestamp\": \"Ngày 30 tháng 05 năm 2026\", \"total_cost\": \"1.815.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_209.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Hóa đơn này được xuất vào ngày tháng năm nào?",
    "ground_truth": "Ngày 12 tháng 06 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_060.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 63 Nguyễn Trãi, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 12 tháng 06 năm 2026\", \"total_cost\": \"1.998.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_088.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Cửa hàng / công ty phát hành hóa đơn nằm ở địa chỉ nào?",
    "ground_truth": "Số 231 Trần Hưng Đạo, Quận 1, TP. Hồ Chí Minh"
  },
  {
    "image_name": "receipt_c45_bb_train_137.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Xuất cấu trúc JSON chi tiết của hóa đơn gồm bên bán, tổng tiền, ngày lập và danh sách mặt hàng?",
    "ground_truth": "{\"seller\": \"TRUNG TÂM Y TẾ HUYỆN GIA LÂM\", \"address\": \"Số 71 Bùi Thị Xuân, Quận 1, Đồng Nai\", \"timestamp\": \"Ngày 16 tháng 06 năm 2026\", \"total_cost\": \"2.035.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_111.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 311 Trần Hưng Đạo, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "image_name": "receipt_c45_bb_train_082.png",
    "template": "receipt_c45_bb",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "ỦY BAN NHÂN DÂN QUẬN CẦU GIẤY"
  },
  {
    "image_name": "receipt_c45_bb_val_003.png",
    "template": "receipt_c45_bb",
    "field": "FULL_JSON",
    "question": "Trích xuất toàn bộ thông tin hóa đơn dưới dạng JSON?",
    "ground_truth": "{\"seller\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"address\": \"Số 306 Hai Bà Trưng, Quận 1, Bình Dương\", \"timestamp\": \"Ngày 17 tháng 06 năm 2026\", \"total_cost\": \"1.870.000 VND\"}"
  },
  {
    "image_name": "receipt_c45_bb_train_038.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Khách hàng phải thanh toán tổng cộng bao nhiêu tiền?",
    "ground_truth": "2.052.000 VND"
  },
  {
    "image_name": "receipt_c45_bb_train_161.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 239 Nguyễn Huệ, Quận 1, Hải Phòng"
  },
  {
    "image_name": "receipt_c45_bb_train_179.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_SELLER",
    "question": "Tìm và định vị vùng chứa tên đơn vị bán hàng trên hóa đơn?",
    "ground_truth": "{\"text\": \"TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI\", \"box\": [92, 29, 365, 47]}"
  },
  {
    "image_name": "receipt_c45_bb_train_040.png",
    "template": "receipt_c45_bb",
    "field": "GROUNDING_TOTAL",
    "question": "Tìm và định vị vùng chứa tổng tiền thanh toán trên hóa đơn?",
    "ground_truth": "{\"text\": \"378.000 VND\", \"box\": [132, 259, 227, 274]}"
  },
  {
    "image_name": "receipt_c45_bb_val_015.png",
    "template": "receipt_c45_bb",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 114 Lê Lợi, Quận 3, TP. Hồ Chí Minh"
  },
  {
    "image_name": "receipt_c45_bb_train_103.png",
    "template": "receipt_c45_bb",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 22 tháng 05 năm 2026"
  },
  {
    "image_name": "receipt_c45_bb_train_172.png",
    "template": "receipt_c45_bb",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "2.052.000 VND"
  }
]
SYSTEM_INSTRUCTION = "Bạn là trợ lý AI kế toán chuyên trích xuất hóa đơn. Hãy đọc ảnh và trả lời câu hỏi thật ngắn gọn, chính xác tuyệt đối giá trị thực thể, không thêm bất kỳ lời chào hay giải thích nào."

train_items = []
for item in raw_train_data:
    img_name = item.get("image_name", "")
    if img_name in image_map:
        item["full_image_path"] = image_map[img_name]
        train_items.append(item)

print(f"🎯 Khớp thành công {len(train_items)} mẫu huấn luyện nâng cao!")

class OptimizedDocVQADataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        item = self.items[idx]
        image = Image.open(item["full_image_path"]).convert("RGB")
        messages = [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": item["question"]}]},
            {"role": "assistant", "content": [{"type": "text", "text": item["ground_truth"]}]}
        ]
        prompt_only = [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": item["question"]}]}
        ]
        
        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_text = self.processor.apply_chat_template(prompt_only, tokenize=False, add_generation_prompt=True)
        
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[full_text], images=image_inputs, videos=video_inputs, padding=False, return_tensors="pt")
        prompt_inputs = self.processor(text=[prompt_text], images=image_inputs, videos=video_inputs, padding=False, return_tensors="pt")
        
        input_ids = inputs.input_ids[0]
        prompt_len = prompt_inputs.input_ids.shape[1]
        
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": inputs.attention_mask[0],
            "pixel_values": inputs.pixel_values if "pixel_values" in inputs else None,
            "image_grid_thw": inputs.image_grid_thw if "image_grid_thw" in inputs else None,
            "labels": labels
        }


In [ ]:
# 5. Vòng lặp Huấn luyện Tối ưu hóa
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
dataset = OptimizedDocVQADataset(train_items, processor)

num_epochs = 3
grad_accum_steps = 16
batch_size = 1
total_steps = (len(dataset) // (batch_size * grad_accum_steps)) * num_epochs
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN NÂNG CAO ({len(dataset)} MẪU)...")

model.train()
step_count = 0

for epoch in range(num_epochs):
    t_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad()
    
    for idx in range(len(dataset)):
        item = dataset[idx]
        input_ids = item["input_ids"].unsqueeze(0).to("cuda")
        attention_mask = item["attention_mask"].unsqueeze(0).to("cuda")
        labels = item["labels"].unsqueeze(0).to("cuda")
        
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        if item["pixel_values"] is not None:
            kwargs["pixel_values"] = item["pixel_values"].to("cuda")
        if item["image_grid_thw"] is not None:
            kwargs["image_grid_thw"] = item["image_grid_thw"].to("cuda")
            
        outputs = model(**kwargs)
        loss = outputs.loss / grad_accum_steps
        loss.backward()
        running_loss += outputs.loss.item()
        
        if (idx + 1) % grad_accum_steps == 0 or (idx + 1) == len(dataset):
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            step_count += 1
            
            if step_count % 15 == 0:
                cur_lr = optimizer.param_groups[0]["lr"]
                vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
                print(f"Epoch [{epoch+1}/{num_epochs}] Step {step_count} | Loss: {outputs.loss.item():.4f} | LR: {cur_lr:.2e} | VRAM: {vram_gb:.2f} GB")
                
    avg_loss = running_loss / len(dataset)
    print(f"✅ Hoàn thành Epoch {epoch+1}/{num_epochs} trong {time.time() - t_start:.1f}s | Avg Loss: {avg_loss:.4f}")

output_adapter_dir = "/kaggle/working/qwen2_5_vl_lora_adapters"
model.save_pretrained(output_adapter_dir)
processor.save_pretrained(output_adapter_dir)
!cd /kaggle/working && zip -r qwen2_5_vl_lora_adapters.zip qwen2_5_vl_lora_adapters


In [ ]:
# 6. Đánh giá Tối ưu hóa kèm Bộ lọc Hậu xử lý (Post-Processing)
print("=" * 85)
print("🚀 BẮT ĐẦU ĐÁNH GIÁ CHUẨN ĐỊNH LƯỢNG TỐI ƯU HÓA TRÊN 174 CÂU HỎI...")
print("=" * 85)

validation_samples = [
  {
    "id": 1,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT"
  },
  {
    "id": 2,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,389,200đ"
  },
  {
    "id": 3,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "13/06/2026 15:43"
  },
  {
    "id": 4,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 322 Lý Thường Kiệt, Quận Ba Đình, Hà Nội"
  },
  {
    "id": 5,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bút bi Thiên Long FO-03, Giấy Double A A4 70gsm, Dịch vụ Bảo trì Hệ thống mạng, Máy in HP LaserJet Pro, Mực in Canon Cartridge"
  },
  {
    "id": 6,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bút bi Thiên Long FO-03 là bao nhiêu?",
    "ground_truth": "12,000"
  },
  {
    "id": 7,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY TNHH THƯƠNG MẠI & DỊCH VỤ KHÁNH AN"
  },
  {
    "id": 8,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "16,393,300đ"
  },
  {
    "id": 9,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 09:04"
  },
  {
    "id": 10,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 375 Nguyễn Trãi, Quận Bình Thạnh, TP. Hồ Chí Minh"
  },
  {
    "id": 11,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Giấy Double A A4 70gsm, Bút bi Thiên Long FO-03, Dịch vụ Bảo trì Hệ thống mạng, Mực in Canon Cartridge, Máy in HP LaserJet Pro"
  },
  {
    "id": 12,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Giấy Double A A4 70gsm là bao nhiêu?",
    "ground_truth": "195,000"
  },
  {
    "id": 13,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "id": 14,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "7,326,000đ"
  },
  {
    "id": 15,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "06/06/2026 11:59"
  },
  {
    "id": 16,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 99 Nguyễn Huệ, Quận 7, TP. Hồ Chí Minh"
  },
  {
    "id": 17,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Chuột không dây Logitech, Bàn phím cơ Dareu EK87, Dịch vụ Lắp đặt camera giám sát, Bộ phát Wifi TP-Link Archer"
  },
  {
    "id": 18,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chuột không dây Logitech là bao nhiêu?",
    "ground_truth": "1,160,000"
  },
  {
    "id": 19,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "id": 20,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "11,167,200đ"
  },
  {
    "id": 21,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "20/06/2026 18:23"
  },
  {
    "id": 22,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 19 Lý Thường Kiệt, Quận 1, Hải Phòng"
  },
  {
    "id": 23,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bộ phát Wifi TP-Link Archer, Chuột không dây Logitech, Dịch vụ Lắp đặt camera giám sát, Cáp mạng Cat6 UTP 305m, Bàn phím cơ Dareu EK87"
  },
  {
    "id": 24,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bộ phát Wifi TP-Link Archer là bao nhiêu?",
    "ground_truth": "3,400,000"
  },
  {
    "id": 25,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "id": 26,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "165.000 VND"
  },
  {
    "id": 27,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 17 tháng 06 năm 2026"
  },
  {
    "id": 28,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 255 Bùi Thị Xuân, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "id": 29,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRƯỜNG ĐẠI HỌC BÁCH KHOA HÀ NỘI"
  },
  {
    "id": 30,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1.705.000 VND"
  },
  {
    "id": 31,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 15 tháng 06 năm 2026"
  },
  {
    "id": 32,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 173 Nguyễn Huệ, Quận 1, Bình Dương"
  },
  {
    "id": 33,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "id": 34,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "94.050"
  },
  {
    "id": 35,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "12/06/2026 13:46"
  },
  {
    "id": 36,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 102 Hai Bà Trưng, Quận 1, Hải Phòng"
  },
  {
    "id": 37,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Coca Cola Lon 320ml, Nước rửa chén Sunlight 750ml, Khăn giấy Pulppy 100 tờ, Mì tôm Hảo Hảo chua cay"
  },
  {
    "id": 38,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Coca Cola Lon 320ml là bao nhiêu?",
    "ground_truth": "10.000"
  },
  {
    "id": 39,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VM+HN TRẦN DUY HƯNG"
  },
  {
    "id": 40,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "407.160"
  },
  {
    "id": 41,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "06/06/2026 17:28"
  },
  {
    "id": 42,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 220 Lý Thường Kiệt, Quận Hoàn Kiếm, Hà Nội"
  },
  {
    "id": 43,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Dầu ăn Simply 1L, Mì tôm Hảo Hảo chua cay, Bánh Custas Orion 6P, Coca Cola Lon 320ml"
  },
  {
    "id": 44,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Dầu ăn Simply 1L là bao nhiêu?",
    "ground_truth": "224.000"
  },
  {
    "id": 45,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "id": 46,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "759,550.00"
  },
  {
    "id": 47,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 18:55:00"
  },
  {
    "id": 48,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 274 Cách Mạng Tháng Tám, Quận 1, Cần Thơ"
  },
  {
    "id": 49,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bia Heineken Lon 330ml, Khăn ướt Bobby 80 tờ, Bột giặt Ariel 3.2kg, Hộp dâu tây Đà Lạt 250g"
  },
  {
    "id": 50,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bia Heineken Lon 330ml là bao nhiêu?",
    "ground_truth": "19,500.00"
  },
  {
    "id": 51,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CO.OPMART CỐNG QUỲNH"
  },
  {
    "id": 52,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "1,015,200.00"
  },
  {
    "id": 53,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 09:44:00"
  },
  {
    "id": 54,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 32 Nguyễn Huệ, Quận Cầu Giấy, Hà Nội"
  },
  {
    "id": 55,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh quy Cosy Kinh Đô, Hộp dâu tây Đà Lạt 250g, Nước cam ép Twister 1L, Khăn ướt Bobby 80 tờ, Bột giặt Ariel 3.2kg"
  },
  {
    "id": 56,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh quy Cosy Kinh Đô là bao nhiêu?",
    "ground_truth": "44,000.00"
  },
  {
    "id": 57,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "id": 58,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "233,200"
  },
  {
    "id": 59,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 13:25"
  },
  {
    "id": 60,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 127 Hai Bà Trưng, Quận 1, Bình Dương"
  },
  {
    "id": 61,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Rau muống nước 500g, Thịt đùi heo 500g"
  },
  {
    "id": 62,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Rau muống nước 500g là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "id": 63,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "id": 64,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "108,900"
  },
  {
    "id": 65,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "08/06/2026 19:12"
  },
  {
    "id": 66,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 189 Nguyễn Huệ, Quận 1, TP. Hồ Chí Minh"
  },
  {
    "id": 67,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Thịt ba rọi heo 500g, Rau muống nước 500g"
  },
  {
    "id": 68,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Thịt ba rọi heo 500g là bao nhiêu?",
    "ground_truth": "75,000"
  },
  {
    "id": 69,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CIRCLE K"
  },
  {
    "id": 70,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "92,400đ"
  },
  {
    "id": 71,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "07/06/2026 14:03"
  },
  {
    "id": 72,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 395 Trần Hưng Đạo, Quận Hai Bà Trưng, Hà Nội"
  },
  {
    "id": 73,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Xúc xích tiệt trùng Ponnie, Trà xanh Không Độ 500ml, Kẹo cao su Cool Air hũ"
  },
  {
    "id": 74,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích tiệt trùng Ponnie là bao nhiêu?",
    "ground_truth": "8,000"
  },
  {
    "id": 75,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "Circle K Lê Lợi"
  },
  {
    "id": 76,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "129,800đ"
  },
  {
    "id": 77,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "16/06/2026 18:08"
  },
  {
    "id": 78,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 24 Lý Thường Kiệt, Quận Ba Đình, Hà Nội"
  },
  {
    "id": 79,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh bao trứng muối, Trà xanh Không Độ 500ml, Băng cá nhân Urgo hộp 10 miếng"
  },
  {
    "id": 80,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "48,000"
  },
  {
    "id": 81,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 TÔN THẤT THUYẾT"
  },
  {
    "id": 82,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "117,720đ"
  },
  {
    "id": 83,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 16:59"
  },
  {
    "id": 84,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 242 Bùi Thị Xuân, Quận 1, Đồng Nai"
  },
  {
    "id": 85,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Xúc xích Đức ăn liền, Bánh giò thịt băm trứng cút, Trà sữa Kirin Latte 345ml, Cơm nắm cá hồi sốt Mayo"
  },
  {
    "id": 86,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Xúc xích Đức ăn liền là bao nhiêu?",
    "ground_truth": "20,000"
  },
  {
    "id": 87,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 NGUYỄN HUỆ"
  },
  {
    "id": 88,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "77,760đ"
  },
  {
    "id": 89,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "19/06/2026 19:05"
  },
  {
    "id": 90,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 238 Lê Lợi, Quận 1, Cần Thơ"
  },
  {
    "id": 91,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà sữa Kirin Latte 345ml, Xúc xích Đức ăn liền"
  },
  {
    "id": 92,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Kirin Latte 345ml là bao nhiêu?",
    "ground_truth": "32,000"
  },
  {
    "id": 93,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "id": 94,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "id": 95,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 22:37"
  },
  {
    "id": 96,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Mì xào tương đen, Nước khoáng Dasani 500ml"
  },
  {
    "id": 97,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Mì xào tương đen là bao nhiêu?",
    "ground_truth": "40,000"
  },
  {
    "id": 98,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-ELEVEN BÙI VIỆN"
  },
  {
    "id": 99,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "321,840"
  },
  {
    "id": 100,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "10/06/2026 21:34"
  },
  {
    "id": 101,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà đào sả Slurpee, Kẹo dẻo Haribo Goldbears, Snack khoai tây Lays 95g, Sandwich xá xíu phô mai, Nước khoáng Dasani 500ml, Mì xào tương đen"
  },
  {
    "id": 102,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà đào sả Slurpee là bao nhiêu?",
    "ground_truth": "36,000"
  },
  {
    "id": 103,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "id": 104,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "796,068"
  },
  {
    "id": 105,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 16:41"
  },
  {
    "id": 106,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 116 Bùi Thị Xuân, Thành phố Thủ Đức, TP. Hồ Chí Minh"
  },
  {
    "id": 107,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà Sen Vàng Size M, Cà Phê Đen Đá Size M, Trà Thạch Đào Size L, Bánh Tiramisu, Freeze Trà Xanh Size M, Phin Sữa Đá Size L"
  },
  {
    "id": 108,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà Sen Vàng Size M là bao nhiêu?",
    "ground_truth": "135,000"
  },
  {
    "id": 109,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE"
  },
  {
    "id": 110,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "434,160"
  },
  {
    "id": 111,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 19:24"
  },
  {
    "id": 112,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 398 Nguyễn Trãi, Quận 1, Đồng Nai"
  },
  {
    "id": 113,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh Tiramisu, Trà Sen Vàng Size M, Phin Sữa Đá Size L"
  },
  {
    "id": 114,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh Tiramisu là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "id": 115,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG NGUYỄN VĂN CỪ"
  },
  {
    "id": 116,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "506,825"
  },
  {
    "id": 117,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "01/06/2026 21:19"
  },
  {
    "id": 118,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 62 Lý Thường Kiệt, Quận 1, Hải Phòng"
  },
  {
    "id": 119,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Hồng Trà Sữa M, Trà Đào Phúc Long L, Trà Lài Đác Thơm L, Bánh Croissant Bơ Pháp, Trà sữa Phúc Long L, Cà phê Latte"
  },
  {
    "id": 120,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Hồng Trà Sữa M là bao nhiêu?",
    "ground_truth": "45,000"
  },
  {
    "id": 121,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG LÊ LAI"
  },
  {
    "id": 122,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "306,900"
  },
  {
    "id": 123,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "14/06/2026 14:56"
  },
  {
    "id": 124,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 436 Nguyễn Trãi, Quận 1, Hải Phòng"
  },
  {
    "id": 125,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Trà sữa Phúc Long L, Hồng Trà Sữa M, Bánh Croissant Bơ Pháp"
  },
  {
    "id": 126,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Trà sữa Phúc Long L là bao nhiêu?",
    "ground_truth": "150,000"
  },
  {
    "id": 127,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "id": 128,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "841,500"
  },
  {
    "id": 129,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026"
  },
  {
    "id": 130,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 152 Trần Hưng Đạo, Quận Ngũ Hành Sơn, Đà Nẵng"
  },
  {
    "id": 131,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Caffe Americano L, Chocolate Muffin, Butter Croissant, Caramel Macchiato L, Cold Brew Coffee M, Green Tea Latte M"
  },
  {
    "id": 132,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Caffe Americano L là bao nhiêu?",
    "ground_truth": "180,000"
  },
  {
    "id": 133,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "id": 134,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "502,740"
  },
  {
    "id": 135,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "19/06/2026"
  },
  {
    "id": 136,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 48 Lê Lợi, Thành phố Thủ Đức, TP. Hồ Chí Minh"
  },
  {
    "id": 137,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Chocolate Muffin, Cold Brew Coffee M, Butter Croissant, Green Tea Latte M"
  },
  {
    "id": 138,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Chocolate Muffin là bao nhiêu?",
    "ground_truth": "90,000"
  },
  {
    "id": 139,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "id": 140,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "437,800"
  },
  {
    "id": 141,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 15:04"
  },
  {
    "id": 142,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 198 Cách Mạng Tháng Tám, Quận 1, Đồng Nai"
  },
  {
    "id": 143,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Nước ngọt Pepsi L, Khoai Tây Chiên L, Burger Tôm, Bánh Trứng Egg Tart, Bắp Cải Trộn L"
  },
  {
    "id": 144,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Nước ngọt Pepsi L là bao nhiêu?",
    "ground_truth": "38,000"
  },
  {
    "id": 145,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC VIỆT NAM"
  },
  {
    "id": 146,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "364,100"
  },
  {
    "id": 147,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "07/06/2026 15:54"
  },
  {
    "id": 148,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 437 Bùi Thị Xuân, Quận 1, Bình Dương"
  },
  {
    "id": 149,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bắp Cải Trộn L, Nước ngọt Pepsi L, Khoai Tây Chiên L, Gà Giòn Cay 1 miếng, Bánh Trứng Egg Tart"
  },
  {
    "id": 150,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bắp Cải Trộn L là bao nhiêu?",
    "ground_truth": "88,000"
  },
  {
    "id": 151,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "id": 152,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "127,440đ"
  },
  {
    "id": 153,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 21:31"
  },
  {
    "id": 154,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 408 Cách Mạng Tháng Tám, Quận Tân Bình, TP. Hồ Chí Minh"
  },
  {
    "id": 155,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Khoai Tây Lắc Phô Mai, Nước ngọt 7Up L"
  },
  {
    "id": 156,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Khoai Tây Lắc Phô Mai là bao nhiêu?",
    "ground_truth": "50,000"
  },
  {
    "id": 157,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE VINCOM"
  },
  {
    "id": 158,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "422,400đ"
  },
  {
    "id": 159,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "25/05/2026 22:53"
  },
  {
    "id": 160,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 179 Hai Bà Trưng, Quận Đống Đa, Hà Nội"
  },
  {
    "id": 161,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Gà Giòn Vui Vẻ 1 miếng, Khoai Tây Lắc Phô Mai, Mì Ý Sốt Bò Bằm, Bánh Khoai Môn, Nước ngọt 7Up L"
  },
  {
    "id": 162,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Gà Giòn Vui Vẻ 1 miếng là bao nhiêu?",
    "ground_truth": "105,000"
  },
  {
    "id": 163,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "id": 164,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "207.900"
  },
  {
    "id": 165,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 20:13"
  },
  {
    "id": 166,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 151 Nguyễn Huệ, Quận Đống Đa, Hà Nội"
  },
  {
    "id": 167,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Kem đánh răng Colgate, Bánh bao trứng muối, Xúc xích tiệt trùng Ponnie, Mì Ly ăn liền Modern, Nước uống Aquafina 500ml, Trà xanh Không Độ 500ml"
  },
  {
    "id": 168,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Kem đánh răng Colgate là bao nhiêu?",
    "ground_truth": "64.000"
  },
  {
    "id": 169,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "MINIMART ANAN"
  },
  {
    "id": 170,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "661.100"
  },
  {
    "id": 171,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "26/05/2026 14:30"
  },
  {
    "id": 172,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "ADDRESS",
    "question": "Địa chỉ của đơn vị bán hàng là ở đâu?",
    "ground_truth": "Số 70 Nguyễn Huệ, Quận Thanh Khê, Đà Nẵng"
  },
  {
    "id": 173,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "ITEMS_LIST",
    "question": "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?",
    "ground_truth": "Bánh bao trứng muối, Bột giặt Ariel 3.2kg, Kẹo cao su Cool Air hũ"
  },
  {
    "id": 174,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_002.png",
    "field": "ITEM_PRICE",
    "question": "Thành tiền của Bánh bao trứng muối là bao nhiêu?",
    "ground_truth": "32.000"
  }
]

matched_val = []
for s in validation_samples:
    img_name = s["image_name"]
    if img_name in image_map:
        s["full_image_path"] = image_map[img_name]
        matched_val.append(s)

def clean_prediction(text):
    t = str(text).strip()
    prefixes = [
        r'^Hóa đơn được lập vào ngày\s*',
        r'^Theo thông tin trong phiếu thanh toán, ngày lập hóa đơn là\s*',
        r'^Theo hóa đơn bán lẻ, các mặt hàng/dịch vụ được mua bao gồm:\s*',
        r'^Theo hóa đơn, các mặt hàng/dịch vụ được mua bao gồm:\s*',
        r'^The hóa đơn được lập vào ngày\s*',
        r'^The address of the selling company is at\s*'
    ]
    for p in prefixes:
        t = re.sub(p, '', t, flags=re.IGNORECASE).strip()
    return t

def levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2): return levenshtein_distance(s2, s1)
    if len(s2) == 0: return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def calculate_anls(prediction: str, ground_truth: str, threshold: float = 0.5) -> float:
    p = str(prediction).strip().lower()
    gt = str(ground_truth).strip().lower()
    if not p and not gt: return 1.0
    if not p or not gt: return 0.0
    dist = levenshtein_distance(p, gt)
    norm_dist = dist / max(len(p), len(gt))
    return 1.0 - norm_dist if norm_dist < threshold else 0.0

def calculate_token_f1(prediction: str, ground_truth: str) -> float:
    p_tokens = re.findall(r'\w+', str(prediction).lower())
    g_tokens = re.findall(r'\w+', str(ground_truth).lower())
    if not p_tokens and not g_tokens: return 1.0
    if not p_tokens or not g_tokens: return 0.0
    common = set(p_tokens) & set(g_tokens)
    if not common: return 0.0
    prec = sum(p_tokens.count(t) for t in common) / len(p_tokens)
    rec = sum(g_tokens.count(t) for t in common) / len(g_tokens)
    return (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

model.eval()
total_anls, total_em, total_f1 = 0.0, 0.0, 0.0
latencies = []
eval_results = []

for idx, s in enumerate(matched_val):
    t0 = time.time()
    img = Image.open(s["full_image_path"]).convert("RGB")
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": s["question"]}]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, gen_ids)]
        pred_raw = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        pred = clean_prediction(pred_raw)
        
    lat = time.time() - t0
    latencies.append(lat)
    anls = calculate_anls(pred, s["ground_truth"])
    em = 1.0 if pred.strip().lower() == s["ground_truth"].strip().lower() else 0.0
    f1 = calculate_token_f1(pred, s["ground_truth"])
    
    total_anls += anls
    total_em += em
    total_f1 += f1
    
    eval_results.append({
        "id": idx + 1,
        "template": s.get("template", ""),
        "question": s["question"],
        "ground_truth": s["ground_truth"],
        "prediction": pred,
        "anls": round(anls, 4),
        "exact_match": int(em),
        "f1": round(f1, 4)
    })

n = len(matched_val)
final_report = {
    "model_name": "Qwen/Qwen2.5-VL-3B-Instruct (LoRA Advanced Optimized)",
    "hardware": f"Kaggle GPU {torch.cuda.get_device_name(0)}",
    "total_test_records": n,
    "anls_percentage": f"{total_anls / n * 100:.2f}%",
    "exact_match_percentage": f"{total_em / n * 100:.2f}%",
    "f1_percentage": f"{total_f1 / n * 100:.2f}%",
    "avg_latency_seconds": round(sum(latencies)/len(latencies), 3),
    "vram_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024**3), 2),
    "details": eval_results
}

with open("/kaggle/working/evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 85)
print("📊 TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH TỐI ƯU HÓA TOÀN DIỆN:")
print("=" * 85)
print(f"- Điểm ANLS Score        : {final_report['anls_percentage']}")
print(f"- Tỉ lệ Exact Match (EM) : {final_report['exact_match_percentage']}")
print(f"- Điểm Token F1-Score    : {final_report['f1_percentage']}")
print(f"- Độ trễ trung bình      : {final_report['avg_latency_seconds']} s/câu")
print("=" * 85)
